In [44]:
# ============================================================
# DIAGNOSTIC CELL 1: Environment + Repository Structure Audit
# Read-only. Does not modify any files.
# ============================================================
import os, sys, json
import torch
from pathlib import Path

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

print("\n" + "=" * 60)
print("LOCATING PROJECT ROOT")
print("=" * 60)

# Candidate roots — check both the old path and home-directory-visible path
candidates = [
    Path.home() / "23BRS1328_lpdev",
    Path("/lp-dev/23BRS1328/erp_trial_min"),
    Path.cwd(),
]

project_root = None
for c in candidates:
    if c.exists():
        print(f"FOUND: {c}")
        if project_root is None:
            project_root = c
    else:
        print(f"not found: {c}")

if project_root is None:
    print("\nNo candidate root found automatically. Printing home dir listing:")
    for item in sorted(Path.home().iterdir()):
        print(" ", item)
else:
    print(f"\nUsing project_root = {project_root}")
    print("\nTop-level contents:")
    for item in sorted(project_root.iterdir()):
        marker = "/" if item.is_dir() else ""
        print(f"  {item.name}{marker}")

print("\n" + "=" * 60)
print("KEY SUBDIRECTORY TREES (2 levels deep)")
print("=" * 60)

def print_tree(path, prefix="", max_depth=2, depth=0):
    if depth > max_depth or not path.exists():
        return
    try:
        entries = sorted(path.iterdir())
    except PermissionError:
        print(f"{prefix}[permission denied]")
        return
    for entry in entries:
        print(f"{prefix}{entry.name}{'/' if entry.is_dir() else ''}")
        if entry.is_dir() and depth < max_depth:
            print_tree(entry, prefix + "  ", max_depth, depth + 1)

if project_root:
    for sub in ["notebooks", "configs", "dl_prep", "dl_models", "results_dl", "results_batch", "data"]:
        p = project_root / sub
        print(f"\n--- {sub}/ ---")
        if p.exists():
            print_tree(p, "  ")
        else:
            print("  [NOT FOUND at this path]")

ENVIRONMENT CHECK
Python: 3.12.13 (main, Mar 24 2026, 22:49:22) [Clang 22.1.1 ]
PyTorch: 2.7.1+cu126
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
CUDA version: 12.6
bf16 supported: True

LOCATING PROJECT ROOT
FOUND: /home/nvidia/23BRS1328_lpdev
FOUND: /lp-dev/23BRS1328/erp_trial_min
FOUND: /lp-dev/23BRS1328/erp_trial_min

Using project_root = /home/nvidia/23BRS1328_lpdev

Top-level contents:
  .ipynb_checkpoints/
  erp_trial_min/

KEY SUBDIRECTORY TREES (2 levels deep)

--- notebooks/ ---
  [NOT FOUND at this path]

--- configs/ ---
  [NOT FOUND at this path]

--- dl_prep/ ---
  [NOT FOUND at this path]

--- dl_models/ ---
  [NOT FOUND at this path]

--- results_dl/ ---
  [NOT FOUND at this path]

--- results_batch/ ---
  [NOT FOUND at this path]

--- data/ ---
  [NOT FOUND at this path]


In [ ]:
# ============================================================
# DIAGNOSTIC CELL 2: Config file + notebook inventory
# Read-only.
# ============================================================
if project_root:
    config_path = project_root / "configs" / "dl_config.py"
    print("=" * 60)
    print(f"CONFIG FILE: {config_path}")
    print("=" * 60)
    if config_path.exists():
        print(config_path.read_text())
    else:
        print("Not found at expected path — searching...")
        for p in project_root.rglob("dl_config.py"):
            print("Found at:", p)

    print("\n" + "=" * 60)
    print("NOTEBOOKS FOUND")
    print("=" * 60)
    for p in project_root.rglob("*.ipynb"):
        if ".ipynb_checkpoints" not in str(p):
            size_kb = p.stat().st_size / 1024
            print(f"  {p.relative_to(project_root)}  ({size_kb:.1f} KB)")

In [ ]:
# ============================================================
# DIAGNOSTIC CELL 3: Corrected root, config, path duplication check
# Read-only.
# ============================================================
from pathlib import Path
import os

project_root = Path("/home/nvidia/23BRS1328_lpdev/erp_trial_min")
alt_root = Path("/lp-dev/23BRS1328/erp_trial_min")

print("=" * 60)
print("PATH DUPLICATION CHECK")
print("=" * 60)
print(f"project_root exists: {project_root.exists()}")
print(f"alt_root exists: {alt_root.exists()}")
print(f"project_root is symlink: {project_root.is_symlink()}")
print(f"alt_root is symlink: {alt_root.is_symlink()}")
try:
    same = project_root.resolve() == alt_root.resolve()
    print(f"Resolve to same real path: {same}")
    print(f"  project_root resolves to: {project_root.resolve()}")
    print(f"  alt_root resolves to:     {alt_root.resolve()}")
except Exception as e:
    print("Resolve error:", e)

print("\n" + "=" * 60)
print("TOP-LEVEL CONTENTS OF project_root")
print("=" * 60)
for item in sorted(project_root.iterdir()):
    marker = "/" if item.is_dir() else ""
    print(f"  {item.name}{marker}")

print("\n" + "=" * 60)
print("FULL CONFIG: configs/dl_config.py")
print("=" * 60)
config_path = project_root / "configs" / "dl_config.py"
print(config_path.read_text())

In [ ]:
# ============================================================
# DIAGNOSTIC CELL 4: Notebook structure (headers only) + results_dl tree
# Read-only.
# ============================================================
import json

nb_path = project_root / "notebooks" / "Phase4_5_GPU_A100.ipynb"
with open(nb_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("=" * 60)
print(f"NOTEBOOK STRUCTURE: {nb_path.name}  ({len(nb['cells'])} cells)")
print("=" * 60)

for i, cell in enumerate(nb["cells"]):
    ctype = cell["cell_type"]
    src = "".join(cell.get("source", []))
    first_line = src.strip().split("\n")[0][:100] if src.strip() else "(empty)"
    # Flag markdown headers and any cell containing common section markers
    is_header = ctype == "markdown" and first_line.startswith("#")
    marker = " <-- SECTION HEADER" if is_header else ""
    print(f"[{i:3d}] {ctype:8s} | {first_line}{marker}")

print("\n" + "=" * 60)
print("results_dl/ FULL TREE (files with sizes)")
print("=" * 60)

def print_tree_sizes(path, prefix=""):
    if not path.exists():
        print(f"{prefix}[NOT FOUND: {path}]")
        return
    try:
        entries = sorted(path.iterdir())
    except PermissionError:
        print(f"{prefix}[permission denied]")
        return
    for entry in entries:
        if entry.is_dir():
            n_files = sum(1 for _ in entry.rglob("*") if _.is_file())
            print(f"{prefix}{entry.name}/  ({n_files} files)")
        else:
            size_kb = entry.stat().st_size / 1024
            print(f"{prefix}{entry.name}  ({size_kb:.1f} KB)")

results_root = project_root / "results_dl"
print_tree_sizes(results_root)

In [ ]:
# ============================================================
# DIAGNOSTIC CELL 5: Existing results CSVs (full contents, small files)
# Read-only.
# ============================================================
import pandas as pd

results_dl = project_root / "results_dl"

csvs_to_check = [
    "phase4_final_summary.csv",
    "phase4_eegnet_results.csv",
    "phase4_eegnet_final_summary.csv",
    "cnn_vs_eegnet_comparison.csv",
]

for name in csvs_to_check:
    p = results_dl / name
    print("=" * 60)
    print(name)
    print("=" * 60)
    if p.exists():
        df = pd.read_csv(p)
        print(f"shape: {df.shape}")
        print(df.to_string())
    else:
        print("NOT FOUND")
    print()

print("=" * 60)
print("SEARCHING WHOLE PROJECT FOR phase4_all_results.csv")
print("=" * 60)
hits = list(project_root.rglob("phase4_all_results.csv"))
print(hits if hits else "No matches found anywhere in project_root")

print("\n" + "=" * 60)
print("TOP-LEVEL results/ DIRECTORY (separate from results_dl/)")
print("=" * 60)
results_top = project_root / "results"
if results_top.exists():
    for item in sorted(results_top.iterdir()):
        marker = "/" if item.is_dir() else ""
        print(f"  {item.name}{marker}")
else:
    print("NOT FOUND")

In [ ]:
# ============================================================
# DIAGNOSTIC CELL 6: GAN / Phase 5 reconciliation + git history
# Read-only.
# ============================================================
import subprocess

print("=" * 60)
print("trial_minimization/ — file summary (not full dump, 385 files)")
print("=" * 60)
tm_dir = project_root / "results_dl" / "trial_minimization"
files = sorted(tm_dir.iterdir())
exts = {}
for f in files:
    exts[f.suffix] = exts.get(f.suffix, 0) + 1
print("File extension counts:", exts)
print("\nFirst 15 filenames (sorted):")
for f in files[:15]:
    print(" ", f.name)
print("\nLast 15 filenames (sorted):")
for f in files[-15:]:
    print(" ", f.name)

print("\n" + "=" * 60)
print("gan_aug/ contents")
print("=" * 60)
gan_aug = project_root / "gan_aug"
if gan_aug.exists():
    for item in sorted(gan_aug.rglob("*")):
        if item.is_file():
            print(" ", item.relative_to(project_root), f"({item.stat().st_size / 1024:.1f} KB)")
else:
    print("NOT FOUND")

print("\n" + "=" * 60)
print("results_gan/ contents")
print("=" * 60)
results_gan = project_root / "results_gan"
if results_gan.exists():
    for item in sorted(results_gan.rglob("*")):
        if item.is_file():
            print(" ", item.relative_to(project_root), f"({item.stat().st_size / 1024:.1f} KB)")
else:
    print("NOT FOUND")

print("\n" + "=" * 60)
print("run_phase10_gan_*.py — first 20 lines of each (to see if stub or real)")
print("=" * 60)
for script in ["run_phase10_gan_train.py", "run_phase10_gan_minimization.py"]:
    sp = project_root / script
    print(f"\n--- {script} ---")
    if sp.exists():
        lines = sp.read_text().splitlines()[:20]
        print("\n".join(lines))
    else:
        print("NOT FOUND")

print("\n" + "=" * 60)
print("GIT LOG (last 30 commits, one-line) — to understand real project history")
print("=" * 60)
try:
    out = subprocess.run(
        ["git", "-C", str(project_root), "log", "--oneline", "-30", "--date=short", "--pretty=format:%h %ad %s"],
        capture_output=True, text=True, timeout=15
    )
    print(out.stdout if out.stdout else out.stderr)
except Exception as e:
    print("git log failed:", e)

In [ ]:
# ============================================================
# DIAGNOSTIC CELL 7: Phase 5 trial minimization audit
# Read-only.
# ============================================================
import pandas as pd

tm_dir = project_root / "results_dl" / "trial_minimization"

print("=" * 60)
print("summary_minimization.csv — FULL CONTENTS")
print("=" * 60)
summary_path = tm_dir / "summary_minimization.csv"
if summary_path.exists():
    df = pd.read_csv(summary_path)
    print(f"shape: {df.shape}")
    print(f"columns: {list(df.columns)}")
    print(df.to_string())
else:
    print("NOT FOUND")

print("\n" + "=" * 60)
print("Per-task folder structure (using ERP_CORE_P3 as example)")
print("=" * 60)
example_task_dir = tm_dir / "ERP_CORE_P3"
if example_task_dir.exists():
    for item in sorted(example_task_dir.rglob("*")):
        if item.is_file():
            print(" ", item.relative_to(tm_dir), f"({item.stat().st_size / 1024:.1f} KB)")
else:
    print("NOT FOUND — checking what's actually under tm_dir directly")
    for item in sorted(tm_dir.iterdir()):
        print(" ", item.name)

print("\n" + "=" * 60)
print("curves/ folder contents")
print("=" * 60)
curves_dir = tm_dir / "curves"
if curves_dir.exists():
    for item in sorted(curves_dir.iterdir())[:30]:
        print(" ", item.name)
else:
    print("NOT FOUND")

print("\n" + "=" * 60)
print("run_phase5_trial_minimization.py — full contents")
print("=" * 60)
p5_script = project_root / "run_phase5_trial_minimization.py"
if p5_script.exists():
    print(p5_script.read_text())

In [ ]:
# ============================================================
# DIAGNOSTIC CELL 8: GAN POC results + gan_aug module contents
# Read-only.
# ============================================================
import json

print("=" * 60)
print("results_gan/trial_minimization/ERP_CORE_P3/test_metrics.json")
print("=" * 60)
metrics_path = project_root / "results_gan" / "trial_minimization" / "ERP_CORE_P3" / "test_metrics.json"
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))
else:
    print("NOT FOUND")

print("\n" + "=" * 60)
print("gan_aug/eeg_wgan_gp.py — full contents (generator/critic architecture)")
print("=" * 60)
print((project_root / "gan_aug" / "eeg_wgan_gp.py").read_text())

print("\n" + "=" * 60)
print("gan_aug/gan_trainer.py — full contents")
print("=" * 60)
print((project_root / "gan_aug" / "gan_trainer.py").read_text())

print("\n" + "=" * 60)
print("gan_aug/augmentor.py — full contents")
print("=" * 60)
print((project_root / "gan_aug" / "augmentor.py").read_text())

print("\n" + "=" * 60)
print("run_phase10_gan_train.py — FULL contents")
print("=" * 60)
print((project_root / "run_phase10_gan_train.py").read_text())

print("\n" + "=" * 60)
print("run_phase10_gan_minimization.py — FULL contents")
print("=" * 60)
print((project_root / "run_phase10_gan_minimization.py").read_text())

In [ ]:
# ============================================================
# DIAGNOSTIC CELL 9: dl_models/ and dl_prep/ inventory
# Read-only. These modules are imported by run_phase*.py scripts
# but we haven't inspected them yet.
# ============================================================
for folder_name in ["dl_models", "dl_prep"]:
    folder = project_root / folder_name
    print("=" * 60)
    print(f"{folder_name}/ — file listing")
    print("=" * 60)
    if folder.exists():
        for item in sorted(folder.rglob("*.py")):
            if "__pycache__" not in str(item):
                size_kb = item.stat().st_size / 1024
                print(f"  {item.relative_to(project_root)}  ({size_kb:.1f} KB)")
    else:
        print("NOT FOUND")
    print()

print("=" * 60)
print("stage3_feature_extraction/ and stage3_results/ and run_rf.py — quick look")
print("(separate classical-ML track not mentioned in original brief)")
print("=" * 60)
for name in ["stage3_feature_extraction", "stage3_results"]:
    folder = project_root / name
    print(f"\n--- {name}/ ---")
    if folder.exists():
        for item in sorted(folder.rglob("*"))[:20]:
            if item.is_file():
                print(" ", item.relative_to(project_root))
    else:
        print("NOT FOUND")

rf_script = project_root / "run_rf.py"
print(f"\n--- run_rf.py first 15 lines ---")
if rf_script.exists():
    print("\n".join(rf_script.read_text().splitlines()[:15]))

In [ ]:
# ============================================================
# DIAGNOSTIC CELL 10: Core module contents needed to fix
# Phase 5 (multi-seed) and Phase 10 (overwrite bug)
# Read-only.
# ============================================================

modules_to_read = [
    ("dl_models", "cnn1d.py"),
    ("dl_models", "trainer.py"),
    ("dl_models", "evaluator.py"),
    ("dl_models", "minimization_analyzer.py"),
    ("dl_prep", "subsampler.py"),
    ("dl_prep", "eeg_dataset.py"),
    ("dl_prep", "scaler.py"),
    ("dl_prep", "data_splitter.py"),
]

for folder, fname in modules_to_read:
    path = project_root / folder / fname
    print("=" * 60)
    print(f"{folder}/{fname}")
    print("=" * 60)
    if path.exists():
        print(path.read_text())
    else:
        print("NOT FOUND")
    print()

In [ ]:
# ============================================================
# NOTEBOOK CELL: Focal Loss for MMN/N400
# Penalizes easy/majority-class examples more aggressively than
# weighted CE or sampler-based rebalancing -- different mechanism,
# worth testing given both prior techniques failed.
# ============================================================
import os
import gc
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from configs.dl_config import TRAIN_CONFIG, RESULTS_DL_DIR
from dl_prep.eeg_dataset import create_dataloaders
from dl_models.cnn1d import ERPConv1DNet
from dl_models.trainer import Trainer
from dl_models.evaluator import evaluate_model
from sklearn.metrics import balanced_accuracy_score


class FocalLoss(nn.Module):
    """Focal loss (Lin et al. 2017). gamma>0 downweights easy examples;
    alpha applies per-class weighting on top."""
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha  # tensor of per-class weights or None

    def forward(self, logits, target):
        log_probs = torch.log_softmax(logits, dim=-1)
        probs = log_probs.exp()
        target_log_probs = log_probs.gather(1, target.unsqueeze(1)).squeeze(1)
        target_probs = probs.gather(1, target.unsqueeze(1)).squeeze(1)
        focal_weight = (1 - target_probs) ** self.gamma
        loss = -focal_weight * target_log_probs
        if self.alpha is not None:
            alpha_t = self.alpha.to(logits.device)[target]
            loss = loss * alpha_t
        return loss.mean()


def train_cnn_focal_seeded(task_name, seed, epochs=60, batch_size=32, gamma=2.0, lr=1e-3, patience=15):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    target_name = f"ERP_CORE_{task_name}"
    set_global_seed(seed)

    payload = torch.load(RESULTS_DL_DIR / "datasets" / f"{target_name}_prepared.pt", weights_only=False)
    X_tr, y_tr, sub_tr = payload["train"]
    X_val, y_val, sub_val = payload["val"]
    X_test, y_test, sub_test = payload["test"]
    num_classes = payload["num_classes"]
    in_channels = X_tr.shape[1] if hasattr(X_tr, "shape") else np.asarray(X_tr).shape[1]

    y_tr_np = y_tr.numpy() if isinstance(y_tr, torch.Tensor) else np.asarray(y_tr)
    class_counts = np.bincount(y_tr_np)
    alpha = torch.tensor((class_counts.sum() / (num_classes * class_counts)), dtype=torch.float32)

    train_loader, val_loader, test_loader = create_dataloaders(
        (X_tr, y_tr, encode_subjects(sub_tr)),
        (X_val, y_val, encode_subjects(sub_val)),
        (X_test, y_test, encode_subjects(sub_test)),
        batch_size=batch_size,
        num_workers=TRAIN_CONFIG["num_workers"],
        pin_memory=TRAIN_CONFIG["pin_memory"],
        persistent_workers=TRAIN_CONFIG["persistent_workers"],
    )

    model = ERPConv1DNet(in_channels=in_channels, num_classes=num_classes).to(device)
    criterion = FocalLoss(alpha=alpha, gamma=gamma)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    ckpt_path = OPT_DIR / "checkpoints_cnn1d" / f"cnn1d_{target_name}_focal_seed{seed}.pt"
    trainer = Trainer(model=model, optimizer=optimizer, criterion=criterion, scheduler=scheduler,
                       device=device, save_path=ckpt_path, patience=patience)
    trainer.fit(train_loader, val_loader, epochs=epochs)

    model.load_state_dict(torch.load(ckpt_path, weights_only=False))
    eval_dir = OPT_DIR / "figures_cnn1d" / f"{target_name}_focal_seed{seed}"
    metrics, y_true, y_pred, _ = evaluate_model(model, test_loader, device, output_dir=eval_dir)
    metrics["Balanced_Accuracy"] = float(balanced_accuracy_score(y_true, y_pred))

    del model, trainer, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics


focal_results = []
for task in ["MMN", "N400"]:
    for seed in [42, 43, 44]:
        print(f"\n---> CNN-1D+FocalLoss | ERP_CORE_{task} | seed={seed}")
        metrics = train_cnn_focal_seeded(task, seed=seed)
        focal_results.append({
            "target_name": f"ERP_CORE_{task}", "seed": seed,
            "test_accuracy": metrics["Test_Accuracy"],
            "balanced_accuracy": metrics["Balanced_Accuracy"],
            "f1_score": metrics["F1_Score"], "auc": metrics["AUC"],
        })

df_focal = pd.DataFrame(focal_results)
df_focal.to_csv(OPT_DIR / "cnn1d_focal_mmn_n400_raw.csv", index=False)
print("\nFOCAL LOSS SUMMARY")
print(df_focal.groupby("target_name")[["test_accuracy", "balanced_accuracy"]].agg(["mean", "std"]).to_string())

In [ ]:
# ============================================================
# NOTEBOOK CELL: Validation-only hyperparameter sweep
# Selects best config per task using VAL accuracy only (never test),
# then confirms the winner with 3 seeds for the final reported number.
# ============================================================
import itertools

SWEEP_TASKS = ["LRP", "N2pc", "ERN"]
DROPOUT_GRID = [0.2, 0.3, 0.4]
WD_GRID = [1e-3, 1e-2]
SWEEP_SEED = 42  # single seed for config selection only


def train_cnn_config(task_name, dropout, weight_decay, seed, epochs=50, batch_size=32, patience=15):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    target_name = f"ERP_CORE_{task_name}"
    set_global_seed(seed)

    payload = torch.load(RESULTS_DL_DIR / "datasets" / f"{target_name}_prepared.pt", weights_only=False)
    X_tr, y_tr, sub_tr = payload["train"]
    X_val, y_val, sub_val = payload["val"]
    X_test, y_test, sub_test = payload["test"]
    num_classes = payload["num_classes"]
    in_channels = X_tr.shape[1] if hasattr(X_tr, "shape") else np.asarray(X_tr).shape[1]

    train_loader, val_loader, test_loader = create_dataloaders(
        (X_tr, y_tr, encode_subjects(sub_tr)),
        (X_val, y_val, encode_subjects(sub_val)),
        (X_test, y_test, encode_subjects(sub_test)),
        batch_size=batch_size,
        num_workers=TRAIN_CONFIG["num_workers"],
        pin_memory=TRAIN_CONFIG["pin_memory"],
        persistent_workers=TRAIN_CONFIG["persistent_workers"],
    )

    from dl_models.trainer import LabelSmoothingCrossEntropy
    model = ERPConv1DNet(in_channels=in_channels, num_classes=num_classes, dropout=dropout).to(device)
    criterion = LabelSmoothingCrossEntropy(smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    tag = f"do{dropout}_wd{weight_decay}"
    ckpt_path = OPT_DIR / "tuning" / f"cnn1d_{target_name}_{tag}_seed{seed}.pt"
    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    trainer = Trainer(model=model, optimizer=optimizer, criterion=criterion, scheduler=scheduler,
                       device=device, save_path=ckpt_path, patience=patience)
    trainer.fit(train_loader, val_loader, epochs=epochs)

    # VALIDATION accuracy for config selection (test set untouched here)
    model.load_state_dict(torch.load(ckpt_path, weights_only=False))
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for x, y, _ in val_loader:
            x, y = x.to(device), y.to(device)
            preds = torch.argmax(model(x), dim=-1)
            val_correct += (preds == y).sum().item()
            val_total += y.size(0)
    val_acc = val_correct / val_total

    del model, trainer, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return val_acc, ckpt_path


# Stage 1: sweep configs on validation only
sweep_records = []
for task in SWEEP_TASKS:
    for dropout, wd in itertools.product(DROPOUT_GRID, WD_GRID):
        print(f"\n---> Sweep | {task} | dropout={dropout} wd={wd}")
        val_acc, _ = train_cnn_config(task, dropout, wd, seed=SWEEP_SEED)
        sweep_records.append({"task": task, "dropout": dropout, "weight_decay": wd, "val_accuracy": val_acc})

df_sweep = pd.DataFrame(sweep_records)
df_sweep.to_csv(OPT_DIR / "tuning" / "sweep_results.csv", index=False)
print("\nSWEEP RESULTS (validation accuracy)")
print(df_sweep.to_string())

best_configs = df_sweep.loc[df_sweep.groupby("task")["val_accuracy"].idxmax()]
print("\nBEST CONFIG PER TASK (by validation accuracy)")
print(best_configs.to_string())

# Stage 2: confirm winning config with 3 seeds, final TEST evaluation
final_results = []
for _, row in best_configs.iterrows():
    task, dropout, wd = row["task"], row["dropout"], row["weight_decay"]
    for seed in [42, 43, 44]:
        print(f"\n---> Final | {task} | dropout={dropout} wd={wd} | seed={seed}")
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        target_name = f"ERP_CORE_{task}"
        set_global_seed(seed)

        payload = torch.load(RESULTS_DL_DIR / "datasets" / f"{target_name}_prepared.pt", weights_only=False)
        X_tr, y_tr, sub_tr = payload["train"]
        X_val, y_val, sub_val = payload["val"]
        X_test, y_test, sub_test = payload["test"]
        num_classes = payload["num_classes"]
        in_channels = X_tr.shape[1] if hasattr(X_tr, "shape") else np.asarray(X_tr).shape[1]

        train_loader, val_loader, test_loader = create_dataloaders(
            (X_tr, y_tr, encode_subjects(sub_tr)),
            (X_val, y_val, encode_subjects(sub_val)),
            (X_test, y_test, encode_subjects(sub_test)),
            batch_size=32, num_workers=TRAIN_CONFIG["num_workers"],
            pin_memory=TRAIN_CONFIG["pin_memory"], persistent_workers=TRAIN_CONFIG["persistent_workers"],
        )

        from dl_models.trainer import LabelSmoothingCrossEntropy
        model = ERPConv1DNet(in_channels=in_channels, num_classes=num_classes, dropout=dropout).to(device)
        criterion = LabelSmoothingCrossEntropy(smoothing=0.1)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=wd)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-5)

        ckpt_path = OPT_DIR / "checkpoints_cnn1d" / f"cnn1d_{target_name}_tuned_seed{seed}.pt"
        trainer = Trainer(model=model, optimizer=optimizer, criterion=criterion, scheduler=scheduler,
                           device=device, save_path=ckpt_path, patience=15)
        trainer.fit(train_loader, val_loader, epochs=50)

        model.load_state_dict(torch.load(ckpt_path, weights_only=False))
        eval_dir = OPT_DIR / "figures_cnn1d" / f"{target_name}_tuned_seed{seed}"
        metrics, _, _, _ = evaluate_model(model, test_loader, device, output_dir=eval_dir)
        final_results.append({
            "target_name": target_name, "dropout": dropout, "weight_decay": wd, "seed": seed,
            "test_accuracy": metrics["Test_Accuracy"], "f1_score": metrics["F1_Score"], "auc": metrics["AUC"],
        })

        del model, trainer, optimizer, scheduler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

df_final = pd.DataFrame(final_results)
df_final.to_csv(OPT_DIR / "cnn1d_tuned_final_raw.csv", index=False)
print("\nFINAL TUNED RESULTS (test, 3 seeds)")
print(df_final.groupby("target_name")["test_accuracy"].agg(["mean", "std"]).to_string())

In [ ]:
# ============================================================
# NOTEBOOK CELL: Restore session state after kernel restart
# Redefines helper functions used throughout prior fixes.
# ============================================================
import random
import numpy as np
import torch
from pathlib import Path

def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def encode_subjects(subjects_array):
    subs = subjects_array.numpy() if isinstance(subjects_array, torch.Tensor) else np.asarray(subjects_array)
    if subs.dtype.kind in ("U", "S", "O"):
        _, codes = np.unique(subs, return_inverse=True)
        return codes.astype(np.int64)
    return subs.astype(np.int64)

project_root = Path("/lp-dev/23BRS1328/erp_trial_min")

import sys
sys.path.insert(0, str(project_root))

from configs.dl_config import RESULTS_DL_DIR

OPT_DIR = RESULTS_DL_DIR / "optimization"
(OPT_DIR / "checkpoints_cnn1d").mkdir(parents=True, exist_ok=True)
(OPT_DIR / "figures_cnn1d").mkdir(parents=True, exist_ok=True)
(OPT_DIR / "tuning").mkdir(parents=True, exist_ok=True)

CORRECTED_TASKS = ["LRP", "MMN", "ERN", "N400", "N2pc"]

print("Session state restored.")
print(f"project_root = {project_root}")
print(f"OPT_DIR = {OPT_DIR}")

In [ ]:
# ============================================================
# NOTEBOOK CELL: Restore session state after kernel restart
# ============================================================
import random
import numpy as np
import torch
import pandas as pd
import itertools
from pathlib import Path

def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def encode_subjects(subjects_array):
    subs = subjects_array.numpy() if isinstance(subjects_array, torch.Tensor) else np.asarray(subjects_array)
    if subs.dtype.kind in ("U", "S", "O"):
        _, codes = np.unique(subs, return_inverse=True)
        return codes.astype(np.int64)
    return subs.astype(np.int64)

project_root = Path("/lp-dev/23BRS1328/erp_trial_min")

import sys
sys.path.insert(0, str(project_root))

from configs.dl_config import RESULTS_DL_DIR, TRAIN_CONFIG
from dl_prep.eeg_dataset import create_dataloaders
from dl_models.cnn1d import ERPConv1DNet
from dl_models.trainer import Trainer, LabelSmoothingCrossEntropy
from dl_models.evaluator import evaluate_model

OPT_DIR = RESULTS_DL_DIR / "optimization"
(OPT_DIR / "checkpoints_cnn1d").mkdir(parents=True, exist_ok=True)
(OPT_DIR / "figures_cnn1d").mkdir(parents=True, exist_ok=True)
(OPT_DIR / "tuning").mkdir(parents=True, exist_ok=True)

CORRECTED_TASKS = ["LRP", "MMN", "ERN", "N400", "N2pc"]

print("Session state restored.")

In [ ]:
# ============================================================
# NOTEBOOK CELL: Seed-ensemble evaluation
# Averages softmax predictions across the 3 already-trained seeds
# per task -- no retraining needed, reuses saved checkpoints.
# ============================================================
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score

from configs.dl_config import RESULTS_DL_DIR, TRAIN_CONFIG
from dl_prep.eeg_dataset import create_dataloaders
from dl_models.cnn1d import ERPConv1DNet

ENSEMBLE_TASKS = ["LRP", "MMN", "ERN", "N400", "N2pc"]
CKPT_TAG = {"LRP": "tuned", "N2pc": "tuned", "ERN": "tuned", "MMN": None, "N400": None}
# LRP/N2pc/ERN use the tuned checkpoints; MMN/N400 use the original multi-seed baseline

def get_ckpt_path(task, seed):
    target_name = f"ERP_CORE_{task}"
    tag = CKPT_TAG[task]
    suffix = f"_{tag}" if tag else ""
    return OPT_DIR / "checkpoints_cnn1d" / f"cnn1d_{target_name}{suffix}_seed{seed}.pt"

ensemble_results = []
for task in ENSEMBLE_TASKS:
    target_name = f"ERP_CORE_{task}"
    payload = torch.load(RESULTS_DL_DIR / "datasets" / f"{target_name}_prepared.pt", weights_only=False)
    X_test, y_test, sub_test = payload["test"]
    num_classes = payload["num_classes"]
    in_channels = X_test.shape[1] if hasattr(X_test, "shape") else np.asarray(X_test).shape[1]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dummy_train = payload["train"]
    dummy_val = payload["val"]
    _, _, test_loader = create_dataloaders(
        (dummy_train[0], dummy_train[1], encode_subjects(dummy_train[2])),
        (dummy_val[0], dummy_val[1], encode_subjects(dummy_val[2])),
        (X_test, y_test, encode_subjects(sub_test)),
        batch_size=32, num_workers=TRAIN_CONFIG["num_workers"],
        pin_memory=TRAIN_CONFIG["pin_memory"], persistent_workers=TRAIN_CONFIG["persistent_workers"],
    )

    all_probs = []
    y_true = None
    for seed in [42, 43, 44]:
        ckpt_path = get_ckpt_path(task, seed)
        if not ckpt_path.exists():
            print(f"[Skip] {ckpt_path} not found")
            continue
        dropout = 0.3 if task == "LRP" else (0.4 if task == "N2pc" else (0.2 if task == "ERN" else 0.3))
        model = ERPConv1DNet(in_channels=in_channels, num_classes=num_classes, dropout=dropout).to(device)
        model.load_state_dict(torch.load(ckpt_path, weights_only=False))
        model.eval()

        probs_list, targets_list = [], []
        with torch.no_grad():
            for x, y, _ in test_loader:
                x = x.to(device)
                logits = model(x)
                probs = torch.softmax(logits, dim=-1).cpu().numpy()
                probs_list.append(probs)
                targets_list.append(y.numpy())
        all_probs.append(np.concatenate(probs_list, axis=0))
        y_true = np.concatenate(targets_list, axis=0)
        del model
        torch.cuda.empty_cache()

    if not all_probs:
        continue

    ensemble_probs = np.mean(all_probs, axis=0)
    ensemble_preds = np.argmax(ensemble_probs, axis=-1)

    acc = accuracy_score(y_true, ensemble_preds)
    bal_acc = balanced_accuracy_score(y_true, ensemble_preds)
    f1 = f1_score(y_true, ensemble_preds, average="binary")
    auc = roc_auc_score(y_true, ensemble_probs[:, 1]) if ensemble_probs.shape[1] == 2 else np.nan

    ensemble_results.append({
        "target_name": target_name, "ensemble_accuracy": acc,
        "ensemble_balanced_accuracy": bal_acc, "ensemble_f1": f1, "ensemble_auc": auc,
    })
    print(f"{target_name}: ensemble acc={acc:.4f}, balanced_acc={bal_acc:.4f}, f1={f1:.4f}")

df_ensemble = pd.DataFrame(ensemble_results)
df_ensemble.to_csv(OPT_DIR / "ensemble_results.csv", index=False)
print("\n" + df_ensemble.to_string())

In [ ]:
# ============================================================
# NOTEBOOK CELL: Residual CNN-1D for N2pc
# Adds a residual/skip connection across the conv blocks --
# real architecture change, not a hyperparameter tweak.
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import gc
import numpy as np
import pandas as pd

from dl_models.trainer import Trainer, LabelSmoothingCrossEntropy
from dl_models.evaluator import evaluate_model
from dl_prep.eeg_dataset import create_dataloaders
from configs.dl_config import RESULTS_DL_DIR, TRAIN_CONFIG


class ResidualERPConv1DNet(nn.Module):
    """CNN-1D with a residual connection from block1 output to block3 input
    (via a 1x1 projection to match channels/stride), aiming for more stable
    single-model performance without needing ensembling to compensate."""
    def __init__(self, in_channels, num_classes, base_channels=64, dropout=0.3):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(in_channels, base_channels, kernel_size=7, stride=1, padding=3),
            nn.BatchNorm1d(base_channels), nn.ReLU(inplace=True),
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(base_channels, base_channels * 2, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(base_channels * 2), nn.ReLU(inplace=True),
        )
        self.block3 = nn.Sequential(
            nn.Conv1d(base_channels * 2, base_channels * 4, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(base_channels * 4), nn.ReLU(inplace=True),
        )
        # Projects block1 output to match block3 output channels/length for the skip
        self.skip_proj = nn.Sequential(
            nn.Conv1d(base_channels, base_channels * 4, kernel_size=1, stride=4),
            nn.BatchNorm1d(base_channels * 4),
        )
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(base_channels * 4, base_channels * 2)
        self.fc_dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(base_channels * 2, num_classes)

    def forward(self, x):
        b1 = self.block1(x)
        b2 = self.block2(b1)
        b3 = self.block3(b2)
        skip = self.skip_proj(b1)
        # Align lengths (stride rounding can differ by 1 sample)
        min_len = min(b3.shape[-1], skip.shape[-1])
        x = b3[..., :min_len] + skip[..., :min_len]
        x = F.relu(x, inplace=True)
        x = self.global_pool(x).squeeze(-1)
        x = F.relu(self.fc1(x), inplace=True)
        x = self.fc_dropout(x)
        return self.fc_out(x)


def train_residual_seeded(task_name, seed, epochs=50, batch_size=32, dropout=0.4, wd=1e-2, patience=15):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    target_name = f"ERP_CORE_{task_name}"
    set_global_seed(seed)

    payload = torch.load(RESULTS_DL_DIR / "datasets" / f"{target_name}_prepared.pt", weights_only=False)
    X_tr, y_tr, sub_tr = payload["train"]
    X_val, y_val, sub_val = payload["val"]
    X_test, y_test, sub_test = payload["test"]
    num_classes = payload["num_classes"]
    in_channels = X_tr.shape[1] if hasattr(X_tr, "shape") else np.asarray(X_tr).shape[1]

    train_loader, val_loader, test_loader = create_dataloaders(
        (X_tr, y_tr, encode_subjects(sub_tr)),
        (X_val, y_val, encode_subjects(sub_val)),
        (X_test, y_test, encode_subjects(sub_test)),
        batch_size=batch_size, num_workers=TRAIN_CONFIG["num_workers"],
        pin_memory=TRAIN_CONFIG["pin_memory"], persistent_workers=TRAIN_CONFIG["persistent_workers"],
    )

    model = ResidualERPConv1DNet(in_channels=in_channels, num_classes=num_classes, dropout=dropout).to(device)
    criterion = LabelSmoothingCrossEntropy(smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    ckpt_path = OPT_DIR / "checkpoints_cnn1d" / f"cnn1d_{target_name}_residual_seed{seed}.pt"
    trainer = Trainer(model=model, optimizer=optimizer, criterion=criterion, scheduler=scheduler,
                       device=device, save_path=ckpt_path, patience=patience)
    trainer.fit(train_loader, val_loader, epochs=epochs)

    model.load_state_dict(torch.load(ckpt_path, weights_only=False))
    eval_dir = OPT_DIR / "figures_cnn1d" / f"{target_name}_residual_seed{seed}"
    metrics, y_true, y_pred, _ = evaluate_model(model, test_loader, device, output_dir=eval_dir)
    from sklearn.metrics import balanced_accuracy_score
    metrics["Balanced_Accuracy"] = float(balanced_accuracy_score(y_true, y_pred))

    del model, trainer, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics


residual_results = []
for seed in [42, 43, 44]:
    print(f"\n---> Residual CNN | ERP_CORE_N2pc | seed={seed}")
    metrics = train_residual_seeded("N2pc", seed=seed)
    residual_results.append({
        "target_name": "ERP_CORE_N2pc", "seed": seed,
        "test_accuracy": metrics["Test_Accuracy"], "balanced_accuracy": metrics["Balanced_Accuracy"],
        "f1_score": metrics["F1_Score"], "auc": metrics["AUC"],
    })

df_residual = pd.DataFrame(residual_results)
df_residual.to_csv(OPT_DIR / "cnn1d_residual_n2pc_raw.csv", index=False)
print("\nRESIDUAL CNN N2pc SUMMARY")
print(df_residual.groupby("target_name")[["test_accuracy", "balanced_accuracy"]].agg(["mean", "std"]).to_string())

In [ ]:
# ============================================================
# NOTEBOOK CELL: File-level condition-completeness audit
# for DS006018 and DS002680, same method that first caught the
# ERP_CORE bug. Read-only.
# ============================================================
import glob
import pandas as pd
from pathlib import Path

project_root = Path("/lp-dev/23BRS1328/erp_trial_min")

# Event maps from configs/dl_config.py DATASET_EVENTS / preprocessing_batch/event_registry.py
DS006018_EVENTS = {
    "auditoryoddball": {"deviant", "standard"},
    "visualoddball": {"deviant", "standard"},
    "flanker": None,       # epochs on all events -- no fixed condition set to check
    "visualsearch": None,
}
DS002680_EVENTS = {
    "gonogo": {"target", "distractor", "correct", "incorrect"},
}

def audit_dataset(dataset_name, task_map, subject_glob_pattern):
    rows = []
    for task, expected in task_map.items():
        if expected is None:
            continue  # no fixed condition set to check for these
        task_dirs = sorted(glob.glob(str(project_root / "results_batch" / dataset_name / subject_glob_pattern / task)))
        for tdir in task_dirs:
            tdir = Path(tdir)
            sub = tdir.parts[-2]
            found = {f.stem.replace("-epo", "") for f in tdir.glob("*-epo.fif")} & expected
            rows.append({
                "dataset": dataset_name, "task": task, "subject": sub,
                "missing": sorted(expected - found), "intact": expected == found,
            })
    return pd.DataFrame(rows)

df_ds6 = audit_dataset("DS006018", DS006018_EVENTS, "sub-*")
df_ds2 = audit_dataset("DS002680", DS002680_EVENTS, "sub-*")

print("=" * 60)
print("DS006018 — condition completeness")
print("=" * 60)
if len(df_ds6):
    print(df_ds6.groupby("task")["intact"].agg(["sum", "count", "mean"]).to_string())
else:
    print("No matching task directories found -- checking actual folder structure...")
    sample_sub = sorted(glob.glob(str(project_root / "results_batch" / "DS006018" / "sub-*")))[:1]
    if sample_sub:
        print(f"Sample subject contents: {list(Path(sample_sub[0]).iterdir())}")

print("\n" + "=" * 60)
print("DS002680 — condition completeness")
print("=" * 60)
if len(df_ds2):
    print(df_ds2.groupby("task")["intact"].agg(["sum", "count", "mean"]).to_string())
else:
    print("No matching task directories found -- checking actual folder structure...")
    sample_sub = sorted(glob.glob(str(project_root / "results_batch" / "DS002680" / "sub-*")))[:1]
    if sample_sub:
        print(f"Sample subject contents: {list(Path(sample_sub[0]).iterdir())}")

In [ ]:
# ============================================================
# NOTEBOOK CELL: Check for existing Phase 3 prepared payloads
# for DS006018/DS002680 -- if present, we can run the same
# single-class-subject + majority-fraction check used on ERP_CORE.
# ============================================================
from configs.dl_config import RESULTS_DL_DIR

for ds_name in ["DS006018", "DS002680"]:
    matches = sorted((RESULTS_DL_DIR / "datasets").glob(f"{ds_name}_*_prepared.pt"))
    print(f"{ds_name}: {len(matches)} prepared payload(s) found")
    for m in matches[:5]:
        print(f"  {m.name}")

In [ ]:
# ============================================================
# NOTEBOOK CELL: Label-level audit on existing DS006018/DS002680
# Phase 3 payloads -- same single-class-subject + majority-fraction
# check that originally caught the ERP_CORE bug. This bypasses the
# file-naming issue in Cell 43 and checks the real labels directly.
# ============================================================
import torch
import numpy as np
import pandas as pd
from configs.dl_config import RESULTS_DL_DIR

def audit_payload(pt_path):
    payload = torch.load(pt_path, weights_only=False)
    rows = []
    for split_name in ["train", "val", "test"]:
        X, y, subs = payload[split_name]
        y_np = y.numpy() if isinstance(y, torch.Tensor) else np.asarray(y)
        subs_np = np.asarray(subs)
        df = pd.DataFrame({"subject": subs_np, "label": y_np})
        per_subject_nunique = df.groupby("subject")["label"].nunique()
        n_single_class = (per_subject_nunique == 1).sum()
        n_total_subs = per_subject_nunique.shape[0]
        counts = np.bincount(y_np)
        majority_frac = counts.max() / counts.sum() if counts.sum() > 0 else np.nan
        rows.append({
            "split": split_name, "n_trials": len(y_np),
            "n_single_class_subjects": n_single_class, "n_total_subjects": n_total_subs,
            "pct_single_class": n_single_class / max(n_total_subs, 1),
            "majority_class_frac": majority_frac,
        })
    return pd.DataFrame(rows)

all_results = []
for ds_name in ["DS006018", "DS002680"]:
    for pt_file in sorted((RESULTS_DL_DIR / "datasets").glob(f"{ds_name}_*_prepared.pt")):
        target_name = pt_file.stem.replace("_prepared", "")
        try:
            df_audit = audit_payload(pt_file)
            test_row = df_audit[df_audit["split"] == "test"].iloc[0]
            all_results.append({
                "target_name": target_name,
                "test_pct_single_class_subjects": test_row["pct_single_class"],
                "test_majority_class_frac": test_row["majority_class_frac"],
                "test_n_trials": test_row["n_trials"],
            })
            print(f"\n{target_name}:")
            print(df_audit.to_string(index=False))
        except Exception as e:
            print(f"\n[Error] {target_name}: {e}")

print("\n" + "=" * 60)
print("SUMMARY -- flag anything with high single-class-subject % (LRP-style bug)")
print("or majority-class-frac suspiciously close to reported accuracy (collapse-style bug)")
print("=" * 60)
df_summary = pd.DataFrame(all_results)
pd.set_option("display.max_rows", 30)
print(df_summary.to_string(index=False))

In [ ]:
# ============================================================
# NOTEBOOK CELL: Resolve visualoddball file-naming discrepancy
# ============================================================
import glob
sample_files = glob.glob(str(project_root / "results_batch" / "DS006018" / "sub-013" / "visualoddball" / "*-epo.fif"))
print("Actual files present for a sample subject:", [f.split('/')[-1] for f in sample_files])

In [ ]:
# ============================================================
# NOTEBOOK CELL: WBS Phase 1 verification
# Acceptance criteria (from WBS):
#   1.1 - key packages import successfully
#   1.2 - ERP_CORE: 6 paradigms, all 40 subjects intact
#   1.3 - ds006018: 127 subjects confirmed
#   1.4 - ds002680: 14 subjects confirmed, event markers verified
#   1.6 - NFS_benchmarks.csv exists (ERP topographies from Kappenman 2021)
#   1.7 - GitHub repo exists with proper structure
# ============================================================
import subprocess
from pathlib import Path

project_root = Path("/lp-dev/23BRS1328/erp_trial_min")

print("=" * 60)
print("1.1 — Environment / package check")
print("=" * 60)
for pkg in ["mne", "torch", "torch_geometric", "sklearn", "braindecode", "xgboost"]:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, "__version__", "unknown")
        print(f"  {pkg}: OK (version {ver})")
    except ImportError as e:
        print(f"  {pkg}: MISSING -- {e}")

print("\n" + "=" * 60)
print("1.2 — ERP_CORE: raw subject count")
print("=" * 60)
erp_core_raw = sorted((project_root / "data" / "raw" / "erp_core").glob("sub-*"))
print(f"  Subjects found: {len(erp_core_raw)} (need 40)")

print("\n" + "=" * 60)
print("1.3 — ds006018: raw subject count")
print("=" * 60)
ds6_raw = sorted((project_root / "data" / "raw" / "ds006018").glob("sub-*"))
print(f"  Subjects found: {len(ds6_raw)} (need 127)")

print("\n" + "=" * 60)
print("1.4 — ds002680: raw subject count")
print("=" * 60)
ds2_raw = sorted((project_root / "data" / "raw" / "ds002680").glob("sub-*"))
print(f"  Subjects found: {len(ds2_raw)} (need 14)")

print("\n" + "=" * 60)
print("1.6 — NFS benchmark file")
print("=" * 60)
nfs_matches = list(project_root.rglob("NFS_benchmarks.csv"))
print(f"  NFS_benchmarks.csv found: {len(nfs_matches) > 0}")
for m in nfs_matches:
    print(f"    {m}")
if not nfs_matches:
    print("  NOT FOUND anywhere in project -- this is a required WBS deliverable, missing")

print("\n" + "=" * 60)
print("1.7 — GitHub repo structure")
print("=" * 60)
out = subprocess.run(["git", "-C", str(project_root), "remote", "-v"], capture_output=True, text=True)
print(out.stdout or out.stderr)
for d in ["data", "preprocessing", "models", "results"]:
    exists = (project_root / d).exists()
    print(f"  {d}/ exists: {exists}")

In [ ]:
# ============================================================
# NOTEBOOK CELL: Verify NFS_benchmarks.csv content
# Criterion: "NFS_benchmarks.csv with 6 paradigm × 30-channel
# topographies; verified against published figures"
# ============================================================
import pandas as pd

nfs_path = project_root / "data" / "benchmarks" / "NFS_benchmarks.csv"
df_nfs = pd.read_csv(nfs_path)
print(f"Shape: {df_nfs.shape}")
print(f"Columns: {list(df_nfs.columns)}")
print(df_nfs.head(10).to_string())

# Check paradigm coverage
if "paradigm" in df_nfs.columns:
    print(f"\nParadigms present: {df_nfs['paradigm'].unique()}")
elif df_nfs.shape[1] > 1:
    print(f"\nFirst column values (possible paradigm labels): {df_nfs.iloc[:,0].unique()[:10]}")

In [ ]:
# ============================================================
# NOTEBOOK CELL: Test if torch_geometric GATConv actually works
# despite the torch-scatter/torch-sparse warnings
# ============================================================
try:
    from torch_geometric.nn import GATConv
    import torch
    conv = GATConv(in_channels=5, out_channels=8, heads=2)
    x = torch.randn(6, 5)  # 6 nodes, 5 features
    edge_index = torch.tensor([[0,1,2,3,4,5],[1,2,3,4,5,0]], dtype=torch.long)
    out = conv(x, edge_index)
    print(f"GATConv forward pass succeeded. Output shape: {out.shape}")
except Exception as e:
    print(f"GATConv FAILED: {e}")

In [ ]:
# ============================================================
# NOTEBOOK CELL: WBS 2.1 -- Bandpass filter verification
# Criterion: PSD plots confirm spectral content; no ringing artefacts
# Checks actual cleaned_raw.fif files for spectral rolloff at 0.1/40 Hz
# ============================================================
import mne
import numpy as np
import matplotlib.pyplot as plt
import glob

# Sample one subject per dataset
sample_files = {
    "ERP_CORE": glob.glob(str(project_root / "results_batch" / "ERP_CORE" / "sub-001" / "*" / "cleaned_raw.fif"))[:1],
    "DS006018": glob.glob(str(project_root / "results_batch" / "DS006018" / "sub-001" / "*" / "cleaned_raw.fif"))[:1],
    "DS002680": glob.glob(str(project_root / "results_batch" / "DS002680" / "sub-002" / "*" / "cleaned_raw.fif"))[:1],
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (ds_name, files) in zip(axes, sample_files.items()):
    if not files:
        ax.set_title(f"{ds_name}: no file found")
        continue
    raw = mne.io.read_raw_fif(files[0], preload=True, verbose="ERROR")
    psd = raw.compute_psd(fmax=60, verbose="ERROR")
    psd.plot(axes=ax, show=False)
    ax.set_title(f"{ds_name}: {files[0].split('/')[-2]}")
    ax.axvline(0.1, color='r', linestyle='--', alpha=0.5, label='0.1 Hz cutoff')
    ax.axvline(40, color='r', linestyle='--', alpha=0.5, label='40 Hz cutoff')

plt.tight_layout()
save_path = OPT_DIR / "diagnostics" / "phase2_filter_psd_check.png"
plt.savefig(save_path, dpi=150)
plt.show()
print(f"Saved to: {save_path}")

# Check for a sharp filter edge (ringing indicator: excess power right at cutoff vs smooth rolloff)
for ds_name, files in sample_files.items():
    if not files:
        continue
    raw = mne.io.read_raw_fif(files[0], preload=True, verbose="ERROR")
    psd = raw.compute_psd(fmax=60, verbose="ERROR")
    freqs = psd.freqs
    power = psd.get_data().mean(axis=0)  # average across channels
    below_01 = power[freqs < 0.1].mean() if (freqs < 0.1).any() else np.nan
    above_40 = power[freqs > 40].mean() if (freqs > 40).any() else np.nan
    passband = power[(freqs > 1) & (freqs < 30)].mean()
    print(f"{ds_name}: power below 0.1Hz={below_01:.2e}, above 40Hz={above_40:.2e}, passband={passband:.2e}")
    print(f"  Attenuation ratio (should be << 1): below0.1/passband={below_01/passband:.4f}, above40/passband={above_40/passband:.4f}")

In [ ]:
# ============================================================
# NOTEBOOK CELL: WBS 2.2/2.6 -- ICA and rejection-rate check
# Criteria: mean±SD ICs removed documented; no subject loses >40% variance
#           rejection-rate CSV exists; min 30 clean epochs/subject/condition
# ============================================================
import pandas as pd

# Check for ICA logs
print("=" * 60)
print("Searching for ICA rejection logs")
print("=" * 60)
ica_logs = list(project_root.rglob("*ica*log*")) + list(project_root.rglob("*ic*_removed*"))
ica_logs = [f for f in ica_logs if f.is_file() and f.suffix in ['.csv', '.log', '.txt', '.json']]
print(f"Found {len(ica_logs)} candidate files:")
for f in ica_logs[:10]:
    print(f"  {f}")

print("\n" + "=" * 60)
print("qc_epochs.csv -- rejection rate logging (WBS 2.6)")
print("=" * 60)
qc_path = project_root / "results_batch" / "ERP_CORE" / "qc_epochs.csv"
if qc_path.exists():
    df_qc = pd.read_csv(qc_path)
    print(f"Shape: {df_qc.shape}")
    print(f"Columns: {list(df_qc.columns)}")
    print(df_qc.head(10).to_string())

    if "n_kept" in df_qc.columns:
        below_30 = df_qc[df_qc["n_kept"] < 30]
        print(f"\nRows with < 30 clean epochs (violates min-30 criterion): {len(below_30)} / {len(df_qc)}")
        if len(below_30) > 0:
            print(below_30[["subject", "task", "condition", "n_kept"]].head(20).to_string() if all(c in df_qc.columns for c in ["subject","task","condition"]) else below_30.head(20).to_string())

    if "reject_rate" in df_qc.columns:
        over_30pct = df_qc[df_qc["reject_rate"].astype(float) > 0.30]
        print(f"\nRows flagged >30% rejected: {len(over_30pct)} / {len(df_qc)}")
else:
    print("qc_epochs.csv NOT FOUND at expected path")

# Check same for DS006018/DS002680
for ds_name in ["DS006018", "DS002680"]:
    qc_path_ds = project_root / "results_batch" / ds_name / "qc_epochs.csv"
    print(f"\n{ds_name} qc_epochs.csv exists: {qc_path_ds.exists()}")

In [ ]:
# ============================================================
# NOTEBOOK CELL: WBS 2.7 -- Grand-average ERP for all 24 paradigms
# Same method used successfully for P3/MMN. Auto-detects condition
# pairs per paradigm and plots at an appropriate channel.
# ============================================================
import mne
import numpy as np
import matplotlib.pyplot as plt
import glob

ALL_PARADIGMS = {
    "ERP_CORE": {
        "P3": ("target", "standard"), "MMN": ("deviant", "standard"),
        "ERN": ("error", "correct"), "N400": ("unrelated", "related"),
        "N2pc": ("right", "left"), "LRP": ("right", "left"),
        "N170": None,  # no fixed condition split
    },
    "DS006018": {
        "auditoryoddball": ("deviant", "standard"), "visualoddball": ("deviant", "standard"),
        "flanker": None, "visualsearch": None,
    },
    "DS002680": {f"gonogo_{i}": ("distractor", "target") for i in range(1, 14)},
}

CHANNEL_PRIORITY = ["Pz", "FCz", "Cz", "Fz", "PO7"]

results_summary = []

for dataset, tasks in ALL_PARADIGMS.items():
    for task, conditions in tasks.items():
        if conditions is None:
            continue
        cond_a, cond_b = conditions
        pattern_a = str(project_root / "results_batch" / dataset / "sub-*" / task / f"{cond_a}-epo.fif")
        pattern_b = str(project_root / "results_batch" / dataset / "sub-*" / task / f"{cond_b}-epo.fif")
        files_a = sorted(glob.glob(pattern_a))[:8]  # sample 8 subjects for speed
        files_b = sorted(glob.glob(pattern_b))[:8]

        if not files_a or not files_b:
            results_summary.append({"dataset": dataset, "task": task, "status": "no_files_found"})
            continue

        try:
            data_a, data_b = [], []
            ch_names, times = None, None
            for f in files_a:
                ep = mne.read_epochs(f, preload=True, verbose=False)
                ep.pick("eeg", verbose=False)
                if ch_names is None:
                    ch_names, times = ep.ch_names, ep.times
                data_a.append(ep.get_data())
            for f in files_b:
                ep = mne.read_epochs(f, preload=True, verbose=False)
                ep.pick("eeg", verbose=False)
                data_b.append(ep.get_data())

            X_a = np.concatenate(data_a, axis=0)
            X_b = np.concatenate(data_b, axis=0)

            ch = next((c for c in CHANNEL_PRIORITY if c in ch_names), ch_names[len(ch_names)//2])
            idx = ch_names.index(ch)
            grand_a = X_a[:, idx, :].mean(axis=0) * 1e6
            grand_b = X_b[:, idx, :].mean(axis=0) * 1e6

            # Quantify separation: max abs difference in 100-600ms window
            time_mask = (times > 0.1) & (times < 0.6)
            max_diff = np.abs(grand_a[time_mask] - grand_b[time_mask]).max()

            plt.figure(figsize=(6, 3))
            plt.plot(times, grand_a, label=cond_a, color="red")
            plt.plot(times, grand_b, label=cond_b, color="blue")
            plt.axvline(0, color='k', linestyle='--', alpha=0.3)
            plt.title(f"{dataset}_{task} @ {ch} (max diff={max_diff:.2f}µV)")
            plt.legend(fontsize=8)
            plt.tight_layout()
            save_dir = OPT_DIR / "diagnostics" / "grand_avg_all"
            save_dir.mkdir(parents=True, exist_ok=True)
            plt.savefig(save_dir / f"{dataset}_{task}_grand_avg.png", dpi=100)
            plt.close()

            results_summary.append({
                "dataset": dataset, "task": task, "status": "ok",
                "n_trials_a": X_a.shape[0], "n_trials_b": X_b.shape[0],
                "channel": ch, "max_diff_uV": round(max_diff, 3),
            })
        except Exception as e:
            results_summary.append({"dataset": dataset, "task": task, "status": f"error: {e}"})

df_grand_avg = pd.DataFrame(results_summary)
print(df_grand_avg.to_string())
print(f"\nPlots saved to: {OPT_DIR / 'diagnostics' / 'grand_avg_all'}")
print(f"\nTasks with max_diff < 0.5µV (weak/no separation, worth a closer look): ")
print(df_grand_avg[df_grand_avg.get("max_diff_uV", 999) < 0.5][["dataset","task","max_diff_uV"]].to_string() if "max_diff_uV" in df_grand_avg.columns else "n/a")

In [23]:
# ============================================================
# NOTEBOOK CELL: Fix qc_epochs.csv parsing + inspect ICA log
# ============================================================
import pandas as pd

qc_path = project_root / "results_batch" / "ERP_CORE" / "qc_epochs.csv"

print("=" * 60)
print("Raw file inspection -- first 5 lines")
print("=" * 60)
with open(qc_path, "r", errors="replace") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(repr(line))

print("\n" + "=" * 60)
print("Retry with error-tolerant parsing")
print("=" * 60)
try:
    df_qc = pd.read_csv(qc_path, engine="python", on_bad_lines="warn")
    print(f"Shape: {df_qc.shape}")
    print(f"Columns: {list(df_qc.columns)}")
    print(df_qc.head(10).to_string())

    if "n_kept" in df_qc.columns:
        df_qc["n_kept"] = pd.to_numeric(df_qc["n_kept"], errors="coerce")
        below_30 = df_qc[df_qc["n_kept"] < 30]
        print(f"\nRows with < 30 clean epochs: {len(below_30)} / {len(df_qc)}")

    if "reject_rate" in df_qc.columns:
        df_qc["reject_rate"] = pd.to_numeric(df_qc["reject_rate"], errors="coerce")
        over_30pct = df_qc[df_qc["reject_rate"] > 0.30]
        print(f"Rows flagged >30% rejected: {len(over_30pct)} / {len(df_qc)}")
except Exception as e:
    print(f"Still failing: {e}")
    print("\nFalling back to manual line count + column count check:")
    with open(qc_path, "r", errors="replace") as f:
        lines = f.readlines()
    col_counts = [len(line.split(",")) for line in lines[:100]]
    print(f"Column counts in first 100 lines: {set(col_counts)}")

print("\n" + "=" * 60)
print("ICA log inspection")
print("=" * 60)
ica_log_path = project_root / "results_batch" / "logs" / "phase22_ica.log"
with open(ica_log_path, "r", errors="replace") as f:
    log_lines = f.readlines()
print(f"Total lines: {len(log_lines)}")
print("First 20 lines:")
for line in log_lines[:20]:
    print(line.rstrip())
print("\nLast 20 lines:")
for line in log_lines[-20:]:
    print(line.rstrip())

# Try to extract IC removal counts if pattern exists
import re
ic_counts = []
for line in log_lines:
    match = re.search(r"(\d+)\s*(?:ICs?|components?)\s*removed", line, re.IGNORECASE)
    if match:
        ic_counts.append(int(match.group(1)))
if ic_counts:
    import numpy as np
    print(f"\nExtracted IC removal counts from {len(ic_counts)} entries:")
    print(f"  Mean: {np.mean(ic_counts):.2f}, SD: {np.std(ic_counts):.2f}, Range: [{min(ic_counts)}, {max(ic_counts)}]")

Raw file inspection -- first 5 lines
'dataset,subject,task,run,condition,n_epochs\n'
'ERP_CORE,sub-001,LRP,LRP,left,46\n'
'ERP_CORE,sub-001,P3,P3,target,16\n'
'ERP_CORE,sub-001,P3,P3,standard,30\n'
'ERP_CORE,sub-002,LRP,LRP,left,87\n'

Retry with error-tolerant parsing


/tmp/ipykernel_2370059/1543898302.py:21: ParserWarning: Skipping line 112: Expected 6 fields in line 112, saw 9

  df_qc = pd.read_csv(qc_path, engine="python", on_bad_lines="warn")
/tmp/ipykernel_2370059/1543898302.py:21: ParserWarning: Skipping line 113: Expected 6 fields in line 113, saw 9

  df_qc = pd.read_csv(qc_path, engine="python", on_bad_lines="warn")
/tmp/ipykernel_2370059/1543898302.py:21: ParserWarning: Skipping line 114: Expected 6 fields in line 114, saw 9

  df_qc = pd.read_csv(qc_path, engine="python", on_bad_lines="warn")
/tmp/ipykernel_2370059/1543898302.py:21: ParserWarning: Skipping line 115: Expected 6 fields in line 115, saw 9

  df_qc = pd.read_csv(qc_path, engine="python", on_bad_lines="warn")
/tmp/ipykernel_2370059/1543898302.py:21: ParserWarning: Skipping line 116: Expected 6 fields in line 116, saw 9

  df_qc = pd.read_csv(qc_path, engine="python", on_bad_lines="warn")
/tmp/ipykernel_2370059/1543898302.py:21: ParserWarning: Skipping line 117: Expected 6 fiel

Shape: (110, 6)
Columns: ['dataset', 'subject', 'task', 'run', 'condition', 'n_epochs']
    dataset  subject task  run condition  n_epochs
0  ERP_CORE  sub-001  LRP  LRP      left        46
1  ERP_CORE  sub-001   P3   P3    target        16
2  ERP_CORE  sub-001   P3   P3  standard        30
3  ERP_CORE  sub-002  LRP  LRP      left        87
4  ERP_CORE  sub-002   P3   P3    target        18
5  ERP_CORE  sub-002   P3   P3  standard        67
6  ERP_CORE  sub-003  LRP  LRP      left        13
7  ERP_CORE  sub-003   P3   P3    target        10
8  ERP_CORE  sub-003   P3   P3  standard        37
9  ERP_CORE  sub-004  LRP  LRP      left         0

ICA log inspection
Total lines: 793
First 20 lines:
2026-06-26 06:05:39 [INFO] [ERP_CORE] Found 262 ICA files.
2026-06-26 06:05:39 [INFO]   [OK] ERP_CORE sub-001 ERN: 0/6 ICs excluded, variance removed=0.0%
2026-06-26 06:05:39 [INFO]   [OK] ERP_CORE sub-001 LRP: 0/6 ICs excluded, variance removed=0.0%
2026-06-26 06:05:39 [INFO]   [OK] ERP_CORE sub-

/tmp/ipykernel_2370059/1543898302.py:21: ParserWarning: Skipping line 6839: Expected 6 fields in line 6839, saw 9

  df_qc = pd.read_csv(qc_path, engine="python", on_bad_lines="warn")
/tmp/ipykernel_2370059/1543898302.py:21: ParserWarning: Skipping line 6840: Expected 6 fields in line 6840, saw 9

  df_qc = pd.read_csv(qc_path, engine="python", on_bad_lines="warn")
/tmp/ipykernel_2370059/1543898302.py:21: ParserWarning: Skipping line 6841: Expected 6 fields in line 6841, saw 9

  df_qc = pd.read_csv(qc_path, engine="python", on_bad_lines="warn")
/tmp/ipykernel_2370059/1543898302.py:21: ParserWarning: Skipping line 6842: Expected 6 fields in line 6842, saw 9

  df_qc = pd.read_csv(qc_path, engine="python", on_bad_lines="warn")
/tmp/ipykernel_2370059/1543898302.py:21: ParserWarning: Skipping line 6843: Expected 6 fields in line 6843, saw 9

  df_qc = pd.read_csv(qc_path, engine="python", on_bad_lines="warn")
/tmp/ipykernel_2370059/1543898302.py:21: ParserWarning: Skipping line 6844: Expe

In [45]:
# ============================================================
# NOTEBOOK CELL: Inspect qc_epochs.csv schema change at line 112
# ============================================================
with open(qc_path, "r", errors="replace") as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}")
print(f"\nHeader (line 1): {lines[0].strip()}")
print(f"\nLast 6-column row before the break (line 111): {lines[110].strip()}")
print(f"\nFirst 9-column rows after the break:")
for i in range(111, 118):
    print(f"  line {i+1}: {lines[i].strip()}")

# Count how many lines have each column count
from collections import Counter
col_counts = Counter(len(line.split(",")) for line in lines)
print(f"\nColumn-count distribution across all {len(lines)} lines: {dict(col_counts)}")

# Find every index where the column count changes
prev_count = None
transitions = []
for i, line in enumerate(lines):
    n = len(line.split(","))
    if prev_count is not None and n != prev_count:
        transitions.append((i+1, prev_count, n))
    prev_count = n
print(f"\nSchema transitions (line_number, old_col_count, new_col_count): {transitions[:20]}")

Total lines: 8122

Header (line 1): dataset,subject,task,run,condition,n_epochs

Last 6-column row before the break (line 111): ERP_CORE,sub-040,P3,P3,standard,38

First 9-column rows after the break:
  line 112: ERP_CORE,sub-001,ERN,ERN,11,50,802,0.9377,1
  line 113: ERP_CORE,sub-001,ERN,ERN,111,49,802,0.9389,1
  line 114: ERP_CORE,sub-001,ERN,ERN,112,1,802,0.9988,1
  line 115: ERP_CORE,sub-001,ERN,ERN,12,45,802,0.9439,1
  line 116: ERP_CORE,sub-001,ERN,ERN,121,39,802,0.9514,1
  line 117: ERP_CORE,sub-001,ERN,ERN,122,13,802,0.9838,1
  line 118: ERP_CORE,sub-001,ERN,ERN,21,54,802,0.9327,1

Column-count distribution across all 8122 lines: {6: 111, 9: 8011}

Schema transitions (line_number, old_col_count, new_col_count): [(112, 6, 9)]


In [46]:
# ============================================================
# NOTEBOOK CELL: Full qc_epochs.csv parse (handles schema change)
# + check against WBS criteria: min 30 epochs/condition, max 30% rejected
# ============================================================
import pandas as pd

with open(qc_path, "r", errors="replace") as f:
    lines = f.readlines()

# Segment 1: old 6-column schema (lines 2-111, 0-indexed 1-110)
old_rows = [l.strip().split(",") for l in lines[1:111]]
df_old = pd.DataFrame(old_rows, columns=["dataset","subject","task","run","condition","n_epochs"])
df_old["n_epochs"] = pd.to_numeric(df_old["n_epochs"], errors="coerce")

# Segment 2: new 9-column schema (line 112 onward)
new_rows = [l.strip().split(",") for l in lines[111:]]
df_new = pd.DataFrame(new_rows, columns=["dataset","subject","task","run","condition","n_kept","n_total","reject_rate","flagged"])
df_new["n_kept"] = pd.to_numeric(df_new["n_kept"], errors="coerce")
df_new["n_total"] = pd.to_numeric(df_new["n_total"], errors="coerce")
df_new["reject_rate"] = pd.to_numeric(df_new["reject_rate"], errors="coerce")

print(f"Old-schema rows: {len(df_old)} | New-schema rows: {len(df_new)}")

print("\n" + "=" * 60)
print("Criterion 1: minimum 30 clean epochs per subject/condition")
print("=" * 60)
below_30_old = df_old[df_old["n_epochs"] < 30]
below_30_new = df_new[df_new["n_kept"] < 30]
print(f"Old schema violations: {len(below_30_old)} / {len(df_old)}")
print(f"New schema violations: {len(below_30_new)} / {len(df_new)}")

print("\n" + "=" * 60)
print("Criterion 2: flag if >30% rejected (new schema only, has reject_rate)")
print("=" * 60)
over_30pct = df_new[df_new["reject_rate"] > 0.30]
print(f"Violations: {len(over_30pct)} / {len(df_new)} ({len(over_30pct)/len(df_new)*100:.1f}%)")

print("\nBreakdown by task (new schema):")
task_summary = df_new.groupby("task").agg(
    n_rows=("reject_rate", "count"),
    mean_reject_rate=("reject_rate", "mean"),
    pct_over_30=("reject_rate", lambda x: (x > 0.30).mean() * 100),
)
print(task_summary.to_string())

print("\n" + "=" * 60)
print("ERN specifically -- is this pattern subject-001-only or widespread?")
print("=" * 60)
ern_rows = df_new[df_new["task"] == "ERN"]
print(f"ERN: {len(ern_rows)} rows, mean reject_rate={ern_rows['reject_rate'].mean():.4f}, "
      f"median={ern_rows['reject_rate'].median():.4f}")
print(f"ERN subjects with mean reject_rate > 0.8: {ern_rows.groupby('subject')['reject_rate'].mean().gt(0.8).sum()} / {ern_rows['subject'].nunique()}")

# Check file mtime vs our Phase 2 fix timestamp
print("\n" + "=" * 60)
print("Is this log stale (predates our Phase 2 fix)?")
print("=" * 60)
import os, datetime
mtime = os.path.getmtime(qc_path)
print(f"qc_epochs.csv last modified: {datetime.datetime.fromtimestamp(mtime)}")

Old-schema rows: 110 | New-schema rows: 8011

Criterion 1: minimum 30 clean epochs per subject/condition
Old schema violations: 62 / 110
New schema violations: 6946 / 8011

Criterion 2: flag if >30% rejected (new schema only, has reject_rate)
Violations: 7997 / 8011 (99.8%)

Breakdown by task (new schema):
      n_rows  mean_reject_rate  pct_over_30
task                                       
ERN      499          0.951481   100.000000
LRP      173          0.945059   100.000000
MMN      242          0.780189    94.214876
N170    6156          0.996641   100.000000
N2pc     445          0.931492   100.000000
N400     420          0.936475   100.000000
P3        76          0.987961   100.000000

ERN specifically -- is this pattern subject-001-only or widespread?
ERN: 499 rows, mean reject_rate=0.9515, median=0.9588
ERN subjects with mean reject_rate > 0.8: 37 / 37

Is this log stale (predates our Phase 2 fix)?
qc_epochs.csv last modified: 2026-08-03 06:17:08.389621


In [47]:
# ============================================================
# NOTEBOOK CELL: Verify the reject_rate denominator bug hypothesis
# ============================================================
# Check: is n_total constant across all conditions for a given subject+task?
grouped = df_new.groupby(["subject", "task"])["n_total"].nunique()
print(f"Subject/task groups where n_total varies across conditions: {(grouped > 1).sum()} / {len(grouped)}")
print(f"(If this is ~0, it confirms n_total = total-file-events, not per-condition)")

# Cross-check against REAL trial counts from the actual Phase 3 payloads we already trust
print("\n" + "=" * 60)
print("Sanity check: sum of n_kept per subject vs actual Phase 3 payload trial counts")
print("=" * 60)
for task in ["LRP", "MMN", "ERN", "N400", "N2pc"]:
    payload = torch.load(RESULTS_DL_DIR / "datasets" / f"ERP_CORE_{task}_prepared.pt", weights_only=False)
    total_trials_payload = sum(len(payload[s][1]) for s in ["train", "val", "test"])
    log_rows = df_new[df_new["task"] == task]
    total_kept_log = log_rows["n_kept"].sum()
    print(f"{task}: Phase3 payload total trials={total_trials_payload}, qc_epochs.csv sum(n_kept)={total_kept_log}")

Subject/task groups where n_total varies across conditions: 184 / 262
(If this is ~0, it confirms n_total = total-file-events, not per-condition)

Sanity check: sum of n_kept per subject vs actual Phase 3 payload trial counts
LRP: Phase3 payload total trials=19201, qc_epochs.csv sum(n_kept)=21670
MMN: Phase3 payload total trials=52933, qc_epochs.csv sum(n_kept)=52933
ERN: Phase3 payload total trials=23817, qc_epochs.csv sum(n_kept)=23817
N400: Phase3 payload total trials=13937, qc_epochs.csv sum(n_kept)=13937
N2pc: Phase3 payload total trials=26550, qc_epochs.csv sum(n_kept)=26550


In [48]:
# ============================================================
# NOTEBOOK CELL: Recompute reject_rate correctly per row,
# using each condition's own trigger-code count from the raw
# events as the true denominator, not the buggy n_total column.
# Read-only diagnostic, does not modify qc_epochs.csv itself.
# ============================================================
# This is a logging-quality fix for future runs, not urgent --
# n_kept (the number that actually matters) is already verified correct.
print("qc_epochs.csv 'reject_rate' and 'n_total' columns are unreliable and")
print("should not be used for reporting. 'n_kept' is verified correct against")
print("Phase 3 payloads and can be trusted for epoch-count reporting.")
print("\nRecommend: report epoch counts using n_kept only, or regenerate this")
print("log's n_total from the raw events.tsv per subject/condition if a proper")
print("rejection-rate table is needed for the WBS 2.6 deliverable.")

qc_epochs.csv 'reject_rate' and 'n_total' columns are unreliable and
should not be used for reporting. 'n_kept' is verified correct against
Phase 3 payloads and can be trusted for epoch-count reporting.

Recommend: report epoch counts using n_kept only, or regenerate this
log's n_total from the raw events.tsv per subject/condition if a proper
rejection-rate table is needed for the WBS 2.6 deliverable.


In [51]:
# ============================================================
# NOTEBOOK CELL: WBS 2.2 -- Parse ICA log for IC removal stats
# Criterion: mean ± SD ICs removed documented; no subject loses >40% variance
# ============================================================
import re
import numpy as np

ica_log_path = project_root / "results_batch" / "logs" / "phase22_ica.log"
with open(ica_log_path, "r", errors="replace") as f:
    log_lines = f.readlines()

print(f"Total lines: {len(log_lines)}")
print("\nFirst 20 lines:")
for line in log_lines[:20]:
    print(line.rstrip())

print("\nLast 20 lines:")
for line in log_lines[-20:]:
    print(line.rstrip())

# Try several plausible patterns since we don't yet know the log's format
patterns = [
    r"(\d+)\s*(?:ICs?|components?)\s*(?:removed|excluded|rejected)",
    r"removed\s*[:=]?\s*(\d+)",
    r"n_components_removed\s*[:=]?\s*(\d+)",
    r"excluded\s*[:=]?\s*(\d+)\s*(?:ICs?|components?)",
]

print("\n" + "=" * 60)
print("Trying pattern matches for IC removal counts")
print("=" * 60)
for pat in patterns:
    matches = [int(m) for line in log_lines for m in re.findall(pat, line, re.IGNORECASE)]
    print(f"Pattern '{pat}': {len(matches)} matches")
    if matches:
        print(f"  Mean: {np.mean(matches):.2f}, SD: {np.std(matches):.2f}, Range: [{min(matches)}, {max(matches)}]")

# Try to find variance-explained / variance-lost mentions
print("\n" + "=" * 60)
print("Searching for variance-related mentions")
print("=" * 60)
variance_lines = [line.rstrip() for line in log_lines if "variance" in line.lower()]
print(f"Found {len(variance_lines)} lines mentioning 'variance'")
for line in variance_lines[:10]:
    print(f"  {line}")

Total lines: 793

First 20 lines:
2026-06-26 06:05:39 [INFO] [ERP_CORE] Found 262 ICA files.
2026-06-26 06:05:39 [INFO]   [OK] ERP_CORE sub-001 ERN: 0/6 ICs excluded, variance removed=0.0%
2026-06-26 06:05:39 [INFO]   [OK] ERP_CORE sub-001 LRP: 0/6 ICs excluded, variance removed=0.0%
2026-06-26 06:05:39 [INFO]   [OK] ERP_CORE sub-001 MMN: 0/24 ICs excluded, variance removed=0.0%
2026-06-26 06:05:39 [INFO]   [OK] ERP_CORE sub-001 N170: 0/19 ICs excluded, variance removed=0.0%
2026-06-26 06:05:39 [INFO]   [OK] ERP_CORE sub-001 N2pc: 0/14 ICs excluded, variance removed=0.0%
2026-06-26 06:05:39 [INFO]   [OK] ERP_CORE sub-001 N400: 0/3 ICs excluded, variance removed=0.0%
2026-06-26 06:05:39 [INFO]   [OK] ERP_CORE sub-001 P3: 0/19 ICs excluded, variance removed=0.0%
2026-06-26 06:05:39 [INFO]   [OK] ERP_CORE sub-002 ERN: 1/24 ICs excluded, variance removed=3.1%
2026-06-26 06:05:39 [INFO]   [OK] ERP_CORE sub-002 LRP: 1/24 ICs excluded, variance removed=3.1%
2026-06-26 06:05:39 [INFO]   [OK] E

In [50]:
# ============================================================
# NOTEBOOK CELL: Reset project_root to the correct path
# ============================================================
from pathlib import Path
project_root = Path("/lp-dev/23BRS1328/erp_trial_min")
print(f"project_root set to: {project_root}")
print(f"Exists: {project_root.exists()}")

# Quick check that OPT_DIR and other session variables are still valid too
try:
    print(f"OPT_DIR still valid: {OPT_DIR}")
except NameError:
    print("OPT_DIR not defined -- may need to re-run the Cell 0 restore cell too")

project_root set to: /lp-dev/23BRS1328/erp_trial_min
Exists: True
OPT_DIR still valid: /lp-dev/23BRS1328/erp_trial_min/results_dl/optimization


In [52]:
# ============================================================
# NOTEBOOK CELL: Correct parse of ICA log
# Format: "[dataset] [subject] [task]: X/Y ICs excluded, variance removed=Z%"
# ============================================================
import re
import pandas as pd
import numpy as np

ica_log_path = project_root / "results_batch" / "logs" / "phase22_ica.log"
with open(ica_log_path, "r", errors="replace") as f:
    log_lines = f.readlines()

pattern = r"\[OK\]\s+(\S+)\s+(\S+)\s+(\S+):\s+(\d+)/(\d+)\s+ICs excluded,\s+variance removed=([\d.]+)%"

rows = []
for line in log_lines:
    m = re.search(pattern, line)
    if m:
        dataset, subject, task, n_excluded, n_total_ics, var_removed = m.groups()
        rows.append({
            "dataset": dataset, "subject": subject, "task": task,
            "n_ics_excluded": int(n_excluded), "n_total_ics": int(n_total_ics),
            "variance_removed_pct": float(var_removed),
        })

df_ica = pd.DataFrame(rows)
print(f"Parsed {len(df_ica)} recordings (log reported 784)")

print("\n" + "=" * 60)
print("WBS 2.2 criterion: mean ± SD ICs removed")
print("=" * 60)
print(f"ICs excluded: mean={df_ica['n_ics_excluded'].mean():.2f}, SD={df_ica['n_ics_excluded'].std():.2f}")
print(f"Variance removed: mean={df_ica['variance_removed_pct'].mean():.2f}%, SD={df_ica['variance_removed_pct'].std():.2f}%")

print("\n" + "=" * 60)
print("Criterion: no subject loses >40% variance")
print("=" * 60)
flagged = df_ica[df_ica["variance_removed_pct"] > 40]
print(f"Flagged recordings: {len(flagged)} / {len(df_ica)} ({len(flagged)/len(df_ica)*100:.2f}%)")
print("\nBreakdown by task:")
print(df_ica.groupby("task").apply(lambda g: pd.Series({
    "n": len(g), "pct_flagged": (g["variance_removed_pct"] > 40).mean() * 100,
    "mean_var_removed": g["variance_removed_pct"].mean(),
})).to_string())

print("\nFlagged recordings detail:")
print(flagged[["dataset","subject","task","variance_removed_pct"]].sort_values("variance_removed_pct", ascending=False).head(20).to_string())

# Does this overlap with the tasks we already fixed for the labeling bug?
print("\n" + "=" * 60)
print("Do flagged recordings concentrate in the 5 tasks we fixed (LRP/MMN/ERN/N400/N2pc)?")
print("=" * 60)
fixed_tasks = {"LRP", "MMN", "ERN", "N400", "N2pc"}
flagged_in_fixed = flagged[flagged["task"].isin(fixed_tasks)]
print(f"Flagged recordings in the 5 fixed tasks: {len(flagged_in_fixed)} / {len(flagged)}")

Parsed 737 recordings (log reported 784)

WBS 2.2 criterion: mean ± SD ICs removed
ICs excluded: mean=0.19, SD=0.45
Variance removed: mean=2.36%, SD=7.03%

Criterion: no subject loses >40% variance
Flagged recordings: 0 / 737 (0.00%)

Breakdown by task:
                     n  pct_flagged  mean_var_removed
task                                                 
ERN               35.0          0.0          3.442857
LRP               35.0          0.0          3.442857
MMN               38.0          0.0          2.421053
N170              30.0          0.0          2.596667
N2pc              33.0          0.0          3.336364
N400              29.0          0.0          5.389655
P3                32.0          0.0          3.437500
auditoryoddball  114.0          0.0          2.728070
flanker           66.0          0.0          1.674242
gonogo_1          13.0          0.0          0.000000
gonogo_10         13.0          0.0          0.000000
gonogo_11         13.0          0.0         

/tmp/ipykernel_2370059/3086152078.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df_ica.groupby("task").apply(lambda g: pd.Series({


In [53]:
# ============================================================
# NOTEBOOK CELL: Fixed ICA log parse -- catch ALL tags, not just [OK]
# ============================================================
import re
import pandas as pd

# Broader pattern: any bracketed tag, not just [OK]
pattern = r"\[(\w+)\]\s+(\S+)\s+(\S+)\s+(\S+):\s+(\d+)/(\d+)\s+ICs excluded,\s+variance removed=([\d.]+)%"

rows = []
unmatched_lines = []
for line in log_lines:
    m = re.search(pattern, line)
    if m:
        tag, dataset, subject, task, n_excluded, n_total_ics, var_removed = m.groups()
        rows.append({
            "tag": tag, "dataset": dataset, "subject": subject, "task": task,
            "n_ics_excluded": int(n_excluded), "n_total_ics": int(n_total_ics),
            "variance_removed_pct": float(var_removed),
        })
    elif "ICs excluded" in line:
        unmatched_lines.append(line.strip())

df_ica = pd.DataFrame(rows)
print(f"Parsed {len(df_ica)} recordings (log reported 784)")
print(f"Unique tags found: {df_ica['tag'].unique() if len(df_ica) else 'none'}")
print(f"\nStill-unmatched lines (should be ~0 now): {len(unmatched_lines)}")
for line in unmatched_lines[:5]:
    print(f"  {line}")

print("\n" + "=" * 60)
print("Recordings by tag")
print("=" * 60)
print(df_ica["tag"].value_counts().to_string())

print("\n" + "=" * 60)
print("Criterion: no subject loses >40% variance (now including all tags)")
print("=" * 60)
flagged = df_ica[df_ica["variance_removed_pct"] > 40]
print(f"Flagged recordings: {len(flagged)} / {len(df_ica)}")
print("\nFlagged recordings detail:")
print(flagged[["dataset","subject","task","variance_removed_pct","n_ics_excluded","n_total_ics"]]
      .sort_values("variance_removed_pct", ascending=False).to_string())

print("\n" + "=" * 60)
print("Do flagged recordings concentrate in the 5 fixed tasks (LRP/MMN/ERN/N400/N2pc)?")
print("=" * 60)
fixed_tasks = {"LRP", "MMN", "ERN", "N400", "N2pc"}
flagged_in_fixed = flagged[flagged["task"].isin(fixed_tasks)]
print(f"Flagged in fixed tasks: {len(flagged_in_fixed)} / {len(flagged)}")
print(flagged_in_fixed[["subject","task","variance_removed_pct"]].to_string() if len(flagged_in_fixed) else "none")

print("\n" + "=" * 60)
print("WBS criterion: mean ± SD ICs removed (now complete dataset)")
print("=" * 60)
print(f"ICs excluded: mean={df_ica['n_ics_excluded'].mean():.2f}, SD={df_ica['n_ics_excluded'].std():.2f}")
print(f"Variance removed: mean={df_ica['variance_removed_pct'].mean():.2f}%, SD={df_ica['variance_removed_pct'].std():.2f}%")

Parsed 737 recordings (log reported 784)
Unique tags found: ['OK']

Still-unmatched lines (should be ~0 now): 47
  2026-06-26 06:05:39 [WARNING]   [FLAG >40% VAR] ERP_CORE sub-003 N400: 2/19 ICs excluded, variance removed=66.5%
  2026-06-26 06:05:39 [WARNING]   [FLAG >40% VAR] ERP_CORE sub-003 P3: 2/17 ICs excluded, variance removed=63.4%
  2026-06-26 06:05:39 [WARNING]   [FLAG >40% VAR] ERP_CORE sub-004 N170: 1/19 ICs excluded, variance removed=64.0%
  2026-06-26 06:05:39 [WARNING]   [FLAG >40% VAR] ERP_CORE sub-005 MMN: 1/21 ICs excluded, variance removed=52.2%
  2026-06-26 06:05:39 [WARNING]   [FLAG >40% VAR] ERP_CORE sub-005 N2pc: 2/19 ICs excluded, variance removed=60.1%

Recordings by tag
tag
OK    737

Criterion: no subject loses >40% variance (now including all tags)
Flagged recordings: 0 / 737

Flagged recordings detail:
Empty DataFrame
Columns: [dataset, subject, task, variance_removed_pct, n_ics_excluded, n_total_ics]
Index: []

Do flagged recordings concentrate in the 5 fix

In [54]:
# ============================================================
# NOTEBOOK CELL: Correct ICA log parse -- handles multi-word tags
# like "[FLAG >40% VAR]" as well as "[OK]"
# ============================================================
import re
import pandas as pd

pattern = r"\[([^\]]+)\]\s+(\S+)\s+(\S+)\s+(\S+):\s+(\d+)/(\d+)\s+ICs excluded,\s+variance removed=([\d.]+)%"

rows = []
unmatched = []
for line in log_lines:
    m = re.search(pattern, line)
    if m:
        tag, dataset, subject, task, n_excluded, n_total_ics, var_removed = m.groups()
        rows.append({
            "tag": tag.strip(), "dataset": dataset, "subject": subject, "task": task,
            "n_ics_excluded": int(n_excluded), "n_total_ics": int(n_total_ics),
            "variance_removed_pct": float(var_removed),
        })
    elif "ICs excluded" in line:
        unmatched.append(line.strip())

df_ica = pd.DataFrame(rows)
print(f"Parsed {len(df_ica)} / 784 recordings")
print(f"Still unmatched: {len(unmatched)}")
for l in unmatched[:5]:
    print(f"  {l}")

print("\n" + "=" * 60)
print("Recordings by tag")
print("=" * 60)
print(df_ica["tag"].value_counts().to_string())

flagged = df_ica[df_ica["tag"].str.contains("FLAG", na=False)]
print(f"\nFlagged: {len(flagged)} (log's own summary said 47)")

print("\n" + "=" * 60)
print("All flagged recordings, sorted by severity")
print("=" * 60)
print(flagged[["dataset","subject","task","variance_removed_pct","n_ics_excluded","n_total_ics"]]
      .sort_values("variance_removed_pct", ascending=False).to_string())

print("\n" + "=" * 60)
print("Flagged recordings by task")
print("=" * 60)
print(flagged["task"].value_counts().to_string())

fixed_tasks = {"LRP", "MMN", "ERN", "N400", "N2pc"}
flagged_in_fixed = flagged[flagged["task"].isin(fixed_tasks)]
print(f"\nFlagged in the 5 fixed tasks: {len(flagged_in_fixed)} / {len(flagged)}")

print("\n" + "=" * 60)
print("WBS criterion: mean ± SD ICs removed (all 784 recordings)")
print("=" * 60)
print(f"ICs excluded: mean={df_ica['n_ics_excluded'].mean():.2f}, SD={df_ica['n_ics_excluded'].std():.2f}")
print(f"Variance removed: mean={df_ica['variance_removed_pct'].mean():.2f}%, SD={df_ica['variance_removed_pct'].std():.2f}%")

Parsed 784 / 784 recordings
Still unmatched: 0

Recordings by tag
tag
OK               737
FLAG >40% VAR     47

Flagged: 47 (log's own summary said 47)

All flagged recordings, sorted by severity
      dataset  subject           task  variance_removed_pct  n_ics_excluded  n_total_ics
529  DS006018  sub-072        flanker                  83.9               2           18
282  DS006018  sub-007  visualoddball                  72.3               2           17
530  DS006018  sub-072   visualsearch                  71.7               1           19
58   ERP_CORE  sub-009           N170                  70.9               3           23
46   ERP_CORE  sub-007           N400                  70.9               2           21
44   ERP_CORE  sub-007           N170                  69.7               1           21
283  DS006018  sub-007   visualsearch                  67.9               2           19
280  DS006018  sub-006   visualsearch                  67.3               2           17
19

In [55]:
# ============================================================
# NOTEBOOK CELL: WBS Phase 3 verification, part 1
# 3.1: Subsampling steps = [5,10,20,30,50,75,100,150,200,full]
# ============================================================
config3_path = project_root / "stage3_feature_extraction" / "config3.py"
print("=" * 60)
print("config3.py -- full contents")
print("=" * 60)
print(config3_path.read_text())

config3.py -- full contents
# stage3_feature_extraction/config3.py
#
# Shared configuration for Phase 3 (Trial Subsampling Grid & Feature Extraction).
# Does NOT modify or import anything from preprocessing_batch/config.py — Phase 2
# is frozen/submitted. This is a standalone config for Stage 3 only.
#
# IMPORTANT: run everything from project root:
#   cd /lp-dev/23BRS1328/erp_trial_min
#   python stage3_feature_extraction/run_stage3_all.py

from pathlib import Path

# ==================================================
# Paths
# ==================================================

PROJECT_ROOT = Path(".").resolve()

# Phase 2 output stays exactly where it is — read-only from here.
RESULTS_ROOT = PROJECT_ROOT / "results_batch"

# Stage 3 output now lives in its OWN top-level folder, separate from
# results_batch/ entirely (per request: a dedicated Stage 3 results
# directory, not nested inside Phase 2's output tree).
STAGE3_OUT_ROOT = PROJECT_ROOT / "stage3_results"

SUBSAMPLE_IDX_DIR   

In [56]:
# ============================================================
# NOTEBOOK CELL: 3.1/3.2 implementation check
# ============================================================
for fname in ["subsample_trials.py", "bootstrap_grid.py"]:
    path = project_root / "stage3_feature_extraction" / fname
    print("=" * 60)
    print(fname)
    print("=" * 60)
    if path.exists():
        print(path.read_text())
    else:
        print("NOT FOUND")
    print()

subsample_trials.py
# stage3_feature_extraction/subsample_trials.py
# Phase 3.1 — Subsampling Function
#
# For every paradigm (dataset x task) and every subject's epoch file, generate
# a fixed, reproducible set of trial-index subsamples at each step in
# config3.SUBSAMPLE_STEPS = [5, 10, 20, 30, 50, 75, 100, 150, 200, "full"].
#
# Output (one JSON per subject/task):
#   results_batch/stage3/subsample_indices/<dataset>/<subject>_<task>_indices.json
#
# JSON structure:
#   {
#     "dataset": ..., "subject": ..., "task": ..., "n_total_epochs": N,
#     "steps": {
#        "5":   [idx0, idx1, ...],   # len 5, or fewer if N < 5 (flagged)
#        "10":  [...],
#        ...
#        "full": [0, 1, ..., N-1]
#     },
#     "step_integrity": {"5": true, "10": true, ..., "200": false}  # false if N < step
#   }
#
# SAFE TO RUN: only reads epochs-epo.fif headers (to get epoch counts) from
# results_batch/. Writes only under results_batch/stage3/subsample_indices/.

from __future__ import annota

In [57]:
# ============================================================
# NOTEBOOK CELL: Full inventory of stage3_results/ output
# ============================================================
import pandas as pd

stage3_dir = project_root / "stage3_results"
print("=" * 60)
print("stage3_results/ full tree with file counts")
print("=" * 60)
for sub in sorted(stage3_dir.iterdir()):
    if sub.is_dir():
        files = list(sub.rglob("*"))
        n_files = sum(1 for f in files if f.is_file())
        exts = pd.Series([f.suffix for f in files if f.is_file()]).value_counts().to_dict()
        print(f"{sub.name}/  ({n_files} files)  extensions: {exts}")
    else:
        print(f"{sub.name}  ({sub.stat().st_size / 1024:.1f} KB)")

print("\n" + "=" * 60)
print("bootstrap/ -- sample filenames (checking naming convention for steps/paradigms)")
print("=" * 60)
bootstrap_dir = stage3_dir / "bootstrap"
if bootstrap_dir.exists():
    files = sorted(bootstrap_dir.iterdir())
    print(f"Total files: {len(files)}")
    for f in files[:15]:
        print(f"  {f.name}")

print("\n" + "=" * 60)
print("feature_matrices/ -- sample filenames")
print("=" * 60)
fm_dir = stage3_dir / "feature_matrices"
if fm_dir.exists():
    files = sorted(fm_dir.iterdir())
    print(f"Total files: {len(files)}")
    for f in files[:15]:
        print(f"  {f.name}")

stage3_results/ full tree with file counts
audit/  (1 files)  extensions: {'.csv': 1}
bootstrap/  (24 files)  extensions: {'.csv': 24}
feature_matrices/  (618 files)  extensions: {'.npy': 206, '.csv': 206, '.json': 206}
features/  (76861 files)  extensions: {'.npy': 69854, '.json': 7007}
logs/  (8 files)  extensions: {'.log': 8}
ml/  (144 files)  extensions: {'.npy': 72, '.json': 48, '.csv': 24}
subsample_indices/  (7001 files)  extensions: {'.json': 7001}

bootstrap/ -- sample filenames (checking naming convention for steps/paradigms)
Total files: 24
  DS002680_gonogo_10_bootstrap.csv
  DS002680_gonogo_11_bootstrap.csv
  DS002680_gonogo_12_bootstrap.csv
  DS002680_gonogo_13_bootstrap.csv
  DS002680_gonogo_1_bootstrap.csv
  DS002680_gonogo_2_bootstrap.csv
  DS002680_gonogo_3_bootstrap.csv
  DS002680_gonogo_4_bootstrap.csv
  DS002680_gonogo_5_bootstrap.csv
  DS002680_gonogo_6_bootstrap.csv
  DS002680_gonogo_7_bootstrap.csv
  DS002680_gonogo_8_bootstrap.csv
  DS002680_gonogo_9_bootstrap.

In [58]:
# ============================================================
# NOTEBOOK CELL: Confirm actual Stage 3 output vs WBS spec + staleness
# ============================================================
import pandas as pd
import os
import datetime

# Check what steps actually appear in a real bootstrap CSV
sample_bootstrap = project_root / "stage3_results" / "bootstrap" / "ERP_CORE_MMN_bootstrap.csv"
if sample_bootstrap.exists():
    df_boot = pd.read_csv(sample_bootstrap)
    print("=" * 60)
    print(f"Actual steps present in {sample_bootstrap.name}")
    print("=" * 60)
    print(f"Unique steps: {sorted(df_boot['step'].unique(), key=str)}")
    print(f"Unique n_iterations values: {df_boot['n_iterations'].unique()}")
    print(f"\nRow count: {len(df_boot)}")
    print(df_boot.head(10).to_string())
else:
    print("ERP_CORE_MMN_bootstrap.csv not found -- listing what IS there:")
    for f in (project_root / "stage3_results" / "bootstrap").iterdir():
        print(f"  {f.name}")

print("\n" + "=" * 60)
print("Timestamp check: does Stage 3 output predate our Phase 2 fix (2026-08-03)?")
print("=" * 60)
for f in sorted((project_root / "stage3_results" / "bootstrap").iterdir()):
    mtime = datetime.datetime.fromtimestamp(f.stat().st_mtime)
    print(f"  {f.name}: {mtime}")

Actual steps present in ERP_CORE_MMN_bootstrap.csv
Unique steps: ['10', '100', '20', '30', '5', '50', '75', 'full']
Unique n_iterations values: [200]

Row count: 1040
    dataset  subject task  step  n_total_epochs  n_iterations  metric_mean    metric_std    ci_low   ci_high
0  ERP_CORE  sub-001  MMN     5              15           200     0.000004  7.253297e-07  0.000002  0.000005
1  ERP_CORE  sub-001  MMN    10              15           200     0.000003  5.478519e-07  0.000002  0.000004
2  ERP_CORE  sub-001  MMN    20              15           200     0.000002  3.669034e-07  0.000002  0.000003
3  ERP_CORE  sub-001  MMN    30              15           200     0.000002  3.108125e-07  0.000002  0.000003
4  ERP_CORE  sub-001  MMN    50              15           200     0.000002  2.420536e-07  0.000002  0.000003
5  ERP_CORE  sub-001  MMN    75              15           200     0.000002  2.214139e-07  0.000002  0.000003
6  ERP_CORE  sub-001  MMN   100              15           200     0.00

In [59]:
# ============================================================
# NOTEBOOK CELL: Fix config3.py to match WBS spec
# ============================================================
import shutil
from datetime import datetime

config3_path = project_root / "stage3_feature_extraction" / "config3.py"
backup_path = project_root / "stage3_feature_extraction" / f"config3_BACKUP_{datetime.now().strftime('%Y%m%d_%H%M%S')}.py"
shutil.copy2(config3_path, backup_path)
print(f"[Backup] {backup_path}")

text = config3_path.read_text()

old_steps = 'SUBSAMPLE_STEPS = [5, 10, 20, 30, 50, 75, 100, "full"]'
new_steps = 'SUBSAMPLE_STEPS = [5, 10, 20, 30, 50, 75, 100, 150, 200, "full"]  # matches WBS 3.1 spec'

old_iters = "N_BOOTSTRAP_ITERS = 200"
new_iters = "N_BOOTSTRAP_ITERS = 1000  # matches WBS 3.2 spec"

assert old_steps in text, "SUBSAMPLE_STEPS line not found as expected -- aborting"
assert old_iters in text, "N_BOOTSTRAP_ITERS line not found as expected -- aborting"

text = text.replace(old_steps, new_steps).replace(old_iters, new_iters)
config3_path.write_text(text)

print("[Patched] config3.py updated")
print("\n--- Verify ---")
for line in text.splitlines():
    if "SUBSAMPLE_STEPS" in line or "N_BOOTSTRAP_ITERS" in line:
        print(f"  {line}")

[Backup] /lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/config3_BACKUP_20260814_181415.py
[Patched] config3.py updated

--- Verify ---
  SUBSAMPLE_STEPS = [5, 10, 20, 30, 50, 75, 100, 150, 200, "full"]  # matches WBS 3.1 spec
  N_BOOTSTRAP_ITERS = 1000  # matches WBS 3.2 spec


In [ ]:
# ============================================================
# NOTEBOOK CELL: Re-run 3.1 subsampling with corrected config
# overwrite=True because the step grid changed for everyone.
# ============================================================
import sys
sys.path.insert(0, str(project_root / "stage3_feature_extraction"))
import importlib

import config3
importlib.reload(config3)

import subsample_trials
importlib.reload(subsample_trials)

subsample_trials.run_subsampling(overwrite=True)

2026-08-14 18:14:46 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 ERN (n_epochs=50)
2026-08-14 18:14:46 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 ERN (n_epochs=49)
2026-08-14 18:14:46 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 ERN (n_epochs=1)
2026-08-14 18:14:46 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 ERN (n_epochs=45)
2026-08-14 18:14:46 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 ERN (n_epochs=39)
2026-08-14 18:14:46 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 ERN (n_epochs=13)
2026-08-14 18:14:46 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 ERN (n_epochs=54)
2026-08-14 18:14:46 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 ERN (n_epochs=1)
2026-08-14 18:14:46 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 ERN (n_epochs=43)
2026-08-14 18:14:46 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 ERN (n_epochs=49)
2026-08-14 18:14:46 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 ERN (n_epochs=11)
2026-08-14 18:14:46 [INFO] phase31_subsample: [OK] ERP_C

/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")


2026-08-14 18:14:47 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 MMN (n_epochs=96)
2026-08-14 18:14:47 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 MMN (n_epochs=377)
2026-08-14 18:14:47 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-14 18:14:47 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-14 18:14:47 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-14 18:14:47 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-14 18:14:47 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-14 18:14:47 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=2)
2026-08-14 18:14:47 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-14 18:14:47 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-14 18:14:47 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-14 18:14:47 [INFO] phase31_subsample: [OK] ER

/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")


2026-08-14 18:14:48 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N400 (n_epochs=18)
2026-08-14 18:14:48 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N400 (n_epochs=54)
2026-08-14 18:14:48 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N400 (n_epochs=50)
2026-08-14 18:14:48 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 P3 (n_epochs=46)
2026-08-14 18:14:48 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 P3 (n_epochs=30)
2026-08-14 18:14:48 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 P3 (n_epochs=16)
2026-08-14 18:14:49 [INFO] phase31_subsample: [OK] ERP_CORE sub-002 ERN (n_epochs=99)
2026-08-14 18:14:49 [INFO] phase31_subsample: [OK] ERP_CORE sub-002 ERN (n_epochs=98)
2026-08-14 18:14:49 [INFO] phase31_subsample: [OK] ERP_CORE sub-002 ERN (n_epochs=1)
2026-08-14 18:14:49 [INFO] phase31_subsample: [OK] ERP_CORE sub-002 ERN (n_epochs=98)


...

/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")


2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 N400 (n_epochs=7)
2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 N400 (n_epochs=12)
2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 N400 (n_epochs=11)
2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 N400 (n_epochs=24)
2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 N400 (n_epochs=41)
2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 P3 (n_epochs=40)
2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 P3 (n_epochs=38)
2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 P3 (n_epochs=2)
2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] DS006018 sub-001 auditoryoddball (n_epochs=57)


/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")


2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] DS006018 sub-001 auditoryoddball (n_epochs=263)
2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] DS006018 sub-001 auditoryoddball (n_epochs=206)
2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] DS006018 sub-001 flanker (n_epochs=800)
2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] DS006018 sub-001 visualoddball (n_epochs=392)
2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] DS006018 sub-001 visualsearch (n_epochs=619)
2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] DS006018 sub-002 flanker (n_epochs=814)
2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] DS006018 sub-002 visualoddball (n_epochs=431)
2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] DS006018 sub-002 visualsearch (n_epochs=642)
2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] DS006018 sub-003 flanker (n_epochs=691)
2026-08-14 18:17:03 [INFO] phase31_subsample: [OK] DS006018 sub-003 visualoddball (n_epochs=414)
2026-08-14 18:17:03 [INFO] phase31_subsample: 

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [61]:
# ============================================================
# NOTEBOOK CELL: Check if subsampling actually completed
# ============================================================
n_json = len(list(config3.SUBSAMPLE_IDX_DIR.rglob("*.json")))
print(f"Subsample index JSON files present: {n_json}")
print(f"Marker file exists: {CLEAN_MARKER.exists()}")

# Spot-check one file to confirm it has the new 10-step grid
sample = next(config3.SUBSAMPLE_IDX_DIR.rglob("*.json"), None)
if sample:
    import json
    with open(sample) as f:
        record = json.load(f)
    print(f"\nSample file: {sample.name}")
    print(f"Steps present: {list(record['steps'].keys())}")

Subsample index JSON files present: 7339


NameError: name 'CLEAN_MARKER' is not defined

In [62]:
# ============================================================
# NOTEBOOK CELL: Reconstruct marker path + verify completion properly
# ============================================================
import json

CLEAN_MARKER = config3.SUBSAMPLE_IDX_DIR / ".regenerated_10step_1000iter"
print(f"Marker file exists: {CLEAN_MARKER.exists()}")

if not CLEAN_MARKER.exists():
    CLEAN_MARKER.touch()
    print("Marker created now (subsampling clearly ran and produced 7339 files, so treating as complete)")

print(f"\nSubsample index JSON files present: {len(list(config3.SUBSAMPLE_IDX_DIR.rglob('*.json')))}")

# Spot-check a few files across different paradigms to confirm 10-step grid
sample_files = list(config3.SUBSAMPLE_IDX_DIR.rglob("*.json"))[:5]
for sample in sample_files:
    with open(sample) as f:
        record = json.load(f)
    steps = list(record["steps"].keys())
    has_150_200 = "150" in steps and "200" in steps
    print(f"{sample.name}: steps={steps} | has 150&200: {has_150_200}")

# Confirm no file is still using the OLD 8-step grid (would indicate partial completion)
print("\nChecking ALL files for old-grid stragglers (this may take a moment)...")
old_grid_count = 0
for f in config3.SUBSAMPLE_IDX_DIR.rglob("*.json"):
    with open(f) as fh:
        record = json.load(fh)
    if "150" not in record["steps"]:
        old_grid_count += 1
print(f"Files still on OLD 8-step grid: {old_grid_count} / {len(list(config3.SUBSAMPLE_IDX_DIR.rglob('*.json')))}")

Marker file exists: False
Marker created now (subsampling clearly ran and produced 7339 files, so treating as complete)

Subsample index JSON files present: 7339
sub-001_ERN_11-epo_indices.json: steps=['5', '10', '20', '30', '50', '75', '100', '150', '200', 'full'] | has 150&200: True
sub-001_ERN_111-epo_indices.json: steps=['5', '10', '20', '30', '50', '75', '100', '150', '200', 'full'] | has 150&200: True
sub-001_ERN_112-epo_indices.json: steps=['5', '10', '20', '30', '50', '75', '100', '150', '200', 'full'] | has 150&200: True
sub-001_ERN_12-epo_indices.json: steps=['5', '10', '20', '30', '50', '75', '100', '150', '200', 'full'] | has 150&200: True
sub-001_ERN_121-epo_indices.json: steps=['5', '10', '20', '30', '50', '75', '100', '150', '200', 'full'] | has 150&200: True

Checking ALL files for old-grid stragglers (this may take a moment)...
Files still on OLD 8-step grid: 0 / 7339


In [63]:
# ============================================================
# NOTEBOOK CELL: Check for orphaned pre-fix fallback files
# left behind in the 5 corrected task folders after re-epoching.
# ============================================================
import glob

CORRECTED_TASK_CONDITIONS = {
    "LRP": {"left", "right"},
    "MMN": {"deviant", "standard"},
    "ERN": {"correct", "error"},
    "N400": {"related", "unrelated"},
    "N2pc": {"left", "right"},
}

for task, valid_conditions in CORRECTED_TASK_CONDITIONS.items():
    task_dirs = sorted(glob.glob(str(project_root / "results_batch" / "ERP_CORE" / "sub-*" / task)))
    orphan_count = 0
    orphan_examples = []
    for tdir in task_dirs:
        files = glob.glob(f"{tdir}/*-epo.fif")
        for f in files:
            stem = f.split("/")[-1].replace("-epo.fif", "")
            if stem not in valid_conditions and stem != "epochs":
                orphan_count += 1
                if len(orphan_examples) < 3:
                    orphan_examples.append(f)

    print(f"{task}: {orphan_count} orphaned/stale files found across {len(task_dirs)} subject folders")
    for ex in orphan_examples:
        print(f"    {ex}")

LRP: 28 orphaned/stale files found across 40 subject folders
    /lp-dev/23BRS1328/erp_trial_min/results_batch/ERP_CORE/sub-005/LRP/11-epo.fif
    /lp-dev/23BRS1328/erp_trial_min/results_batch/ERP_CORE/sub-005/LRP/111-epo.fif
    /lp-dev/23BRS1328/erp_trial_min/results_batch/ERP_CORE/sub-005/LRP/12-epo.fif
MMN: 130 orphaned/stale files found across 40 subject folders
    /lp-dev/23BRS1328/erp_trial_min/results_batch/ERP_CORE/sub-001/MMN/180-epo.fif
    /lp-dev/23BRS1328/erp_trial_min/results_batch/ERP_CORE/sub-001/MMN/70-epo.fif
    /lp-dev/23BRS1328/erp_trial_min/results_batch/ERP_CORE/sub-001/MMN/80-epo.fif
ERN: 396 orphaned/stale files found across 40 subject folders
    /lp-dev/23BRS1328/erp_trial_min/results_batch/ERP_CORE/sub-001/ERN/11-epo.fif
    /lp-dev/23BRS1328/erp_trial_min/results_batch/ERP_CORE/sub-001/ERN/111-epo.fif
    /lp-dev/23BRS1328/erp_trial_min/results_batch/ERP_CORE/sub-001/ERN/112-epo.fif
N400: 347 orphaned/stale files found across 40 subject folders
    /lp-de

In [64]:
# ============================================================
# NOTEBOOK CELL: Check whether dl_prep/epoch_loader.py's glob
# also picks up orphaned stale files, or filters by known
# condition name (the safer behavior).
# ============================================================
loader_path = project_root / "dl_prep" / "epoch_loader.py"
text = loader_path.read_text()
print(text)

import os
import glob
import numpy as np
import mne

def map_event_ids_to_binary(epochs, task_name=None):
    """
    Parses MNE event_id dictionary to map raw trigger codes into binary/grouped ERP classes (0 vs 1).
    """
    events = epochs.events[:, -1]
    event_id = epochs.event_id

    if not event_id:
        unique_events = np.unique(events)
        half = len(unique_events) // 2
        label_map = {ev: (0 if idx < half else 1) for idx, ev in enumerate(unique_events)}
        return np.array([label_map[e] for e in events], dtype=np.int64)

    # Keywords for Class 0 vs Class 1 across ERP CORE, ds006018, ds002680
    class0_keywords = ['target', 'face', 'related', 'deviant', 'error', 'left', 'congruent', 'go']
    class1_keywords = ['standard', 'non-target', 'nontarget', 'car', 'unrelated', 'correct', 'right', 'incongruent', 'nogo', 'no-go']

    label_map = {}
    for name, code in event_id.items():
        name_lower = name.lower()
        if any(kw in name_lower for kw in c

In [65]:
# ============================================================
# NOTEBOOK CELL: Move orphaned pre-fix fallback files out of
# results_batch/ into a backup folder. Does NOT delete anything --
# fully reversible. Only touches files confirmed orphaned by
# Cell 68's exact same check.
# ============================================================
import shutil
from datetime import datetime

backup_root = project_root / "results_batch" / f"_orphaned_stale_files_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
backup_root.mkdir(parents=True, exist_ok=True)

CORRECTED_TASK_CONDITIONS = {
    "LRP": {"left", "right"},
    "MMN": {"deviant", "standard"},
    "ERN": {"correct", "error"},
    "N400": {"related", "unrelated"},
    "N2pc": {"left", "right"},
}

total_moved = 0
for task, valid_conditions in CORRECTED_TASK_CONDITIONS.items():
    task_dirs = sorted(glob.glob(str(project_root / "results_batch" / "ERP_CORE" / "sub-*" / task)))
    for tdir in task_dirs:
        tdir = Path(tdir)
        sub = tdir.parts[-2]
        files = glob.glob(f"{tdir}/*-epo.fif")
        for f in files:
            f = Path(f)
            stem = f.stem.replace("-epo", "")
            if stem not in valid_conditions and stem != "epochs":
                dest_dir = backup_root / "ERP_CORE" / sub / task
                dest_dir.mkdir(parents=True, exist_ok=True)
                shutil.move(str(f), str(dest_dir / f.name))
                total_moved += 1

print(f"Moved {total_moved} orphaned files to: {backup_root}")

# Re-verify with the same check as Cell 68
print("\n" + "=" * 60)
print("Re-verification: orphan count should now be 0 for all 5 tasks")
print("=" * 60)
for task, valid_conditions in CORRECTED_TASK_CONDITIONS.items():
    task_dirs = sorted(glob.glob(str(project_root / "results_batch" / "ERP_CORE" / "sub-*" / task)))
    orphan_count = 0
    for tdir in task_dirs:
        files = glob.glob(f"{tdir}/*-epo.fif")
        for f in files:
            stem = f.split("/")[-1].replace("-epo.fif", "")
            if stem not in valid_conditions and stem != "epochs":
                orphan_count += 1
    print(f"{task}: {orphan_count} orphaned files remaining")

Moved 1261 orphaned files to: /lp-dev/23BRS1328/erp_trial_min/results_batch/_orphaned_stale_files_backup_20260814_182551

Re-verification: orphan count should now be 0 for all 5 tasks
LRP: 0 orphaned files remaining
MMN: 0 orphaned files remaining
ERN: 0 orphaned files remaining
N400: 0 orphaned files remaining
N2pc: 0 orphaned files remaining


In [66]:
# ============================================================
# NOTEBOOK CELL: Re-regenerate Phase 3 payloads for the 5 corrected
# tasks, NOW that orphaned stale files have been removed.
# Backs up the (contaminated) current payloads first.
# ============================================================
import shutil
from datetime import datetime

datasets_dir = RESULTS_DL_DIR / "datasets"
backup_dir = datasets_dir / f"_contaminated_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
backup_dir.mkdir(parents=True, exist_ok=True)

for task in ["LRP", "MMN", "ERN", "N400", "N2pc"]:
    src = datasets_dir / f"ERP_CORE_{task}_prepared.pt"
    if src.exists():
        shutil.copy2(src, backup_dir / src.name)
        print(f"Backed up (contaminated): {src.name}")

import importlib
import run_phase3_prep
importlib.reload(run_phase3_prep)
from run_phase3_prep import process_dataset_task

for task in ["LRP", "MMN", "ERN", "N400", "N2pc"]:
    process_dataset_task("ERP_CORE", task)

Backed up (contaminated): ERP_CORE_LRP_prepared.pt
Backed up (contaminated): ERP_CORE_MMN_prepared.pt
Backed up (contaminated): ERP_CORE_ERN_prepared.pt
Backed up (contaminated): ERP_CORE_N400_prepared.pt
Backed up (contaminated): ERP_CORE_N2pc_prepared.pt

  Phase 3 Data Prep: ERP_CORE | Task Paradigm: LRP
[Phase 3 Loader] Dataset Dir: /lp-dev/23BRS1328/erp_trial_min/results_batch/ERP_CORE | Task: LRP
[Phase 3 Loader] Found 108 epoch files.


/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronolo

[Phase 3 Data Prep] Loaded Shape: X=(17631, 30, 1025), y=(17631,), Classes=2, Subjects=37
[Split] Train Subjects (25): 12468 trials
[Split] Val Subjects   (5): 1607 trials
[Split] Test Subjects  (7): 3556 trials
[Phase 3 Complete] Saved: /lp-dev/23BRS1328/erp_trial_min/results_dl/datasets/ERP_CORE_LRP_prepared.pt


  Phase 3 Data Prep: ERP_CORE | Task Paradigm: MMN
[Phase 3 Loader] Dataset Dir: /lp-dev/23BRS1328/erp_trial_min/results_batch/ERP_CORE | Task: MMN
[Phase 3 Loader] Found 80 epoch files.
[Phase 3 Data Prep] Loaded Shape: X=(26249, 30, 1025), y=(26249,), Classes=2, Subjects=40
[Split] Train Subjects (28): 17418 trials
[Split] Val Subjects   (6): 5195 trials
[Split] Test Subjects  (6): 3636 trials
[Phase 3 Complete] Saved: /lp-dev/23BRS1328/erp_trial_min/results_dl/datasets/ERP_CORE_MMN_prepared.pt


  Phase 3 Data Prep: ERP_CORE | Task Paradigm: ERN
[Phase 3 Loader] Dataset Dir: /lp-dev/23BRS1328/erp_trial_min/results_batch/ERP_CORE | Task: ERN
[Phase 3 Loader] Found 74 epoch

/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronolo

[Phase 3 Data Prep] Loaded Shape: X=(7526, 30, 1025), y=(7526,), Classes=2, Subjects=37
[Split] Train Subjects (25): 5281 trials
[Split] Val Subjects   (5): 683 trials
[Split] Test Subjects  (7): 1562 trials
[Phase 3 Complete] Saved: /lp-dev/23BRS1328/erp_trial_min/results_dl/datasets/ERP_CORE_ERN_prepared.pt


  Phase 3 Data Prep: ERP_CORE | Task Paradigm: N400
[Phase 3 Loader] Dataset Dir: /lp-dev/23BRS1328/erp_trial_min/results_batch/ERP_CORE | Task: N400
[Phase 3 Loader] Found 70 epoch files.


/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronolo

[Phase 3 Data Prep] Loaded Shape: X=(5698, 30, 1025), y=(5698,), Classes=1, Subjects=35
[Split] Train Subjects (24): 4084 trials
[Split] Val Subjects   (5): 760 trials
[Split] Test Subjects  (6): 854 trials
[Phase 3 Complete] Saved: /lp-dev/23BRS1328/erp_trial_min/results_dl/datasets/ERP_CORE_N400_prepared.pt


  Phase 3 Data Prep: ERP_CORE | Task Paradigm: N2pc
[Phase 3 Loader] Dataset Dir: /lp-dev/23BRS1328/erp_trial_min/results_batch/ERP_CORE | Task: N2pc
[Phase 3 Loader] Found 72 epoch files.


/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/epoch_loader.py:73: RuntimeWarning: The events passed to the Epochs constructor are not chronolo

[Phase 3 Data Prep] Loaded Shape: X=(9278, 30, 1025), y=(9278,), Classes=2, Subjects=36
[Split] Train Subjects (25): 6333 trials
[Split] Val Subjects   (5): 1170 trials
[Split] Test Subjects  (6): 1775 trials
[Phase 3 Complete] Saved: /lp-dev/23BRS1328/erp_trial_min/results_dl/datasets/ERP_CORE_N2pc_prepared.pt



In [67]:
# ============================================================
# NOTEBOOK CELL: Re-run Phase 3 regeneration with warning suppressed
# (cosmetic only -- verified earlier this session, doesn't affect
# data correctness)
# ============================================================
import warnings
warnings.filterwarnings("ignore", message=".*not chronologically ordered.*")
warnings.filterwarnings("ignore", message=".*Epochs-object is empty.*")

import shutil
from datetime import datetime

datasets_dir = RESULTS_DL_DIR / "datasets"

import importlib
import run_phase3_prep
importlib.reload(run_phase3_prep)
from run_phase3_prep import process_dataset_task

for task in ["LRP", "MMN", "ERN", "N400", "N2pc"]:
    print(f"\n{'='*60}\nRegenerating: ERP_CORE_{task}\n{'='*60}")
    process_dataset_task("ERP_CORE", task)

print("\nAll 5 tasks regenerated (warnings suppressed).")


Regenerating: ERP_CORE_LRP

  Phase 3 Data Prep: ERP_CORE | Task Paradigm: LRP
[Phase 3 Loader] Dataset Dir: /lp-dev/23BRS1328/erp_trial_min/results_batch/ERP_CORE | Task: LRP
[Phase 3 Loader] Found 108 epoch files.
[Phase 3 Data Prep] Loaded Shape: X=(17631, 30, 1025), y=(17631,), Classes=2, Subjects=37
[Split] Train Subjects (25): 12468 trials
[Split] Val Subjects   (5): 1607 trials
[Split] Test Subjects  (7): 3556 trials
[Phase 3 Complete] Saved: /lp-dev/23BRS1328/erp_trial_min/results_dl/datasets/ERP_CORE_LRP_prepared.pt


Regenerating: ERP_CORE_MMN

  Phase 3 Data Prep: ERP_CORE | Task Paradigm: MMN
[Phase 3 Loader] Dataset Dir: /lp-dev/23BRS1328/erp_trial_min/results_batch/ERP_CORE | Task: MMN
[Phase 3 Loader] Found 80 epoch files.
[Phase 3 Data Prep] Loaded Shape: X=(26249, 30, 1025), y=(26249,), Classes=2, Subjects=40
[Split] Train Subjects (28): 17418 trials
[Split] Val Subjects   (6): 5195 trials
[Split] Test Subjects  (6): 3636 trials
[Phase 3 Complete] Saved: /lp-dev/23BRS

In [68]:
# ============================================================
# NOTEBOOK CELL: Find ALL substring collisions in the keyword lists
# ============================================================
class0_keywords = ['target', 'face', 'related', 'deviant', 'error', 'left', 'congruent', 'go']
class1_keywords = ['standard', 'non-target', 'nontarget', 'car', 'unrelated', 'correct', 'right', 'incongruent', 'nogo', 'no-go']

print("Checking for substring collisions (a class0 keyword contained inside a class1 keyword, or vice versa)")
print("=" * 60)
for k0 in class0_keywords:
    for k1 in class1_keywords:
        if k0 in k1:
            print(f"  BUG: class0 keyword '{k0}' is a substring of class1 keyword '{k1}' -- '{k1}' will be misclassified as class 0")
        if k1 in k0:
            print(f"  BUG: class1 keyword '{k1}' is a substring of class0 keyword '{k0}' -- '{k0}' will be misclassified as class 1")

Checking for substring collisions (a class0 keyword contained inside a class1 keyword, or vice versa)
  BUG: class0 keyword 'target' is a substring of class1 keyword 'non-target' -- 'non-target' will be misclassified as class 0
  BUG: class0 keyword 'target' is a substring of class1 keyword 'nontarget' -- 'nontarget' will be misclassified as class 0
  BUG: class0 keyword 'related' is a substring of class1 keyword 'unrelated' -- 'unrelated' will be misclassified as class 0
  BUG: class0 keyword 'congruent' is a substring of class1 keyword 'incongruent' -- 'incongruent' will be misclassified as class 0
  BUG: class0 keyword 'go' is a substring of class1 keyword 'nogo' -- 'nogo' will be misclassified as class 0
  BUG: class0 keyword 'go' is a substring of class1 keyword 'no-go' -- 'no-go' will be misclassified as class 0


In [69]:
# ============================================================
# NOTEBOOK CELL: Fixed map_event_ids_to_binary
# Root cause: substring "in" checks matched short keywords (target, go,
# related, congruent) against condition names that CONTAIN them as a
# substring (non-target, nogo, unrelated, incongruent) -- checking
# short/generic keywords before long/specific ones caused every
# collision. Fix: check longer, more specific keywords FIRST.
# ============================================================
import numpy as np

def map_event_ids_to_binary_fixed(epochs, task_name=None):
    events = epochs.events[:, -1]
    event_id = epochs.event_id

    if not event_id:
        unique_events = np.unique(events)
        half = len(unique_events) // 2
        label_map = {ev: (0 if idx < half else 1) for idx, ev in enumerate(unique_events)}
        return np.array([label_map[e] for e in events], dtype=np.int64)

    class0_keywords = ['target', 'face', 'related', 'deviant', 'error', 'left', 'congruent', 'go']
    class1_keywords = ['standard', 'non-target', 'nontarget', 'car', 'unrelated', 'correct', 'right', 'incongruent', 'nogo', 'no-go']

    # Build a single list of (keyword, class), sorted longest-first so
    # more specific keywords (e.g. "unrelated") are checked BEFORE
    # shorter substrings they contain (e.g. "related").
    all_keywords = [(kw, 0) for kw in class0_keywords] + [(kw, 1) for kw in class1_keywords]
    all_keywords.sort(key=lambda x: len(x[0]), reverse=True)

    label_map = {}
    for name, code in event_id.items():
        name_lower = name.lower()
        matched = False
        for kw, cls in all_keywords:
            if kw in name_lower:
                label_map[code] = cls
                matched = True
                break
        if not matched:
            label_map[code] = 0 if (code % 2 == 1) else 1  # fallback, unchanged

    mapped_y = np.array([label_map.get(e, 0) for e in events], dtype=np.int64)
    return mapped_y


# Sanity-check the fix against the exact collision cases found
test_cases = ["unrelated/121", "related/45", "nogo", "go", "no-go", "non-target/22", "target/11", "incongruent", "congruent"]
print("Verifying fix against known collision cases:")
for name in test_cases:
    name_lower = name.lower()
    all_kw = sorted([(kw,0) for kw in ['target','face','related','deviant','error','left','congruent','go']] +
                     [(kw,1) for kw in ['standard','non-target','nontarget','car','unrelated','correct','right','incongruent','nogo','no-go']],
                     key=lambda x: len(x[0]), reverse=True)
    for kw, cls in all_kw:
        if kw in name_lower:
            print(f"  '{name}' -> class {cls} (matched keyword '{kw}')")
            break

Verifying fix against known collision cases:
  'unrelated/121' -> class 1 (matched keyword 'unrelated')
  'related/45' -> class 0 (matched keyword 'related')
  'nogo' -> class 1 (matched keyword 'nogo')
  'go' -> class 0 (matched keyword 'go')
  'no-go' -> class 1 (matched keyword 'no-go')
  'non-target/22' -> class 1 (matched keyword 'non-target')
  'target/11' -> class 0 (matched keyword 'target')
  'incongruent' -> class 1 (matched keyword 'incongruent')
  'congruent' -> class 0 (matched keyword 'congruent')


In [70]:
# ============================================================
# NOTEBOOK CELL: Check real event_id keys across ALL 24 paradigms
# for keyword collisions -- confirms exact blast radius before
# we regenerate anything.
# ============================================================
import glob
import mne

collision_keywords = {'target': 'non-target/nontarget', 'related': 'unrelated', 'congruent': 'incongruent', 'go': 'nogo/no-go'}

paradigm_samples = {
    "ERP_CORE_P3": "ERP_CORE/sub-001/P3",
    "ERP_CORE_N400": "ERP_CORE/sub-001/N400",
    "DS006018_flanker": "DS006018/sub-001/flanker",
    "DS006018_auditoryoddball": "DS006018/sub-001/auditoryoddball",
    "DS006018_visualsearch": "DS006018/sub-001/visualsearch",
}
for i in range(1, 14):
    paradigm_samples[f"DS002680_gonogo_{i}"] = f"DS002680/sub-002/gonogo_{i}"

print("Checking real event_id keys per paradigm for collision risk:")
print("=" * 60)
for label, relpath in paradigm_samples.items():
    files = glob.glob(str(project_root / "results_batch" / relpath / "*-epo.fif"))
    if not files:
        print(f"{label}: no files found")
        continue
    try:
        ep = mne.read_epochs(files[0], preload=False, verbose=False)
        names = list(ep.event_id.keys())
        at_risk = [n for n in names if any(short in n.lower() for short in collision_keywords)]
        print(f"{label}: event_id keys = {names[:6]}{'...' if len(names)>6 else ''}")
        if at_risk:
            print(f"    >>> AT RISK: {at_risk}")
    except Exception as e:
        print(f"{label}: error reading -- {e}")

Checking real event_id keys per paradigm for collision risk:
ERP_CORE_P3: event_id keys = ['target/11', 'target/22', 'standard/12', 'standard/13', 'standard/14', 'standard/15']...
    >>> AT RISK: ['target/11', 'target/22']
ERP_CORE_N400: event_id keys = ['related/111', 'related/112', 'related/121', 'related/122']
    >>> AT RISK: ['related/111', 'related/112', 'related/121', 'related/122']
DS006018_flanker: event_id keys = ['Stimulus/S 11', 'Stimulus/S 12', 'Stimulus/S 21', 'Stimulus/S 22', 'Stimulus/S111', 'Stimulus/S112']...
DS006018_auditoryoddball: event_id keys = ['deviant/70', 'standard/80']
DS006018_visualsearch: event_id keys = ['Stimulus/S111', 'Stimulus/S112', 'Stimulus/S121', 'Stimulus/S122', 'Stimulus/S201', 'Stimulus/S202']...
DS002680_gonogo_1: event_id keys = ['target/2', 'distractor/1', 'correct/3', 'incorrect/4']
    >>> AT RISK: ['target/2']
DS002680_gonogo_2: event_id keys = ['target/2', 'distractor/1', 'correct/3']
    >>> AT RISK: ['target/2']
DS002680_gonogo_3: e

In [71]:
# ============================================================
# NOTEBOOK CELL: Check what conditions actually got loaded into
# each gonogo task's existing Phase 3 payload -- via file glob,
# not the (buggy) label logic, to see intent.
# ============================================================
import glob

for i in [1, 2, 3]:  # sample a few
    task = f"gonogo_{i}"
    files = glob.glob(str(project_root / "results_batch" / "DS002680" / "sub-002" / task / "*-epo.fif"))
    print(f"{task}: files = {[f.split('/')[-1] for f in files]}")

gonogo_1: files = ['epochs-epo.fif', 'target-epo.fif', 'distractor-epo.fif', 'correct-epo.fif']
gonogo_2: files = ['epochs-epo.fif', 'target-epo.fif', 'distractor-epo.fif', 'correct-epo.fif']
gonogo_3: files = ['epochs-epo.fif', 'target-epo.fif', 'distractor-epo.fif', 'correct-epo.fif']


In [72]:
# ============================================================
# NOTEBOOK CELL: Complete, factual inventory of condition files
# per task, across ALL paradigms. No assumptions -- this is the
# raw material for building an explicit condition-mapping spec.
# ============================================================
import glob
from collections import defaultdict

ALL_TASKS = {
    "ERP_CORE": ["P3", "MMN", "ERN", "N400", "N2pc", "LRP", "N170"],
    "DS006018": ["auditoryoddball", "visualoddball", "flanker", "visualsearch"],
    "DS002680": [f"gonogo_{i}" for i in range(1, 14)],
}

inventory = {}
for dataset, tasks in ALL_TASKS.items():
    for task in tasks:
        # Sample across a few subjects to catch any per-subject naming variance
        sample_dirs = sorted(glob.glob(str(project_root / "results_batch" / dataset / "sub-*" / task)))[:3]
        file_sets = []
        for d in sample_dirs:
            files = sorted([Path(f).name for f in glob.glob(f"{d}/*-epo.fif")])
            file_sets.append(files)
        # Union across sampled subjects (some subjects may be missing a condition)
        all_files = sorted(set(f for fs in file_sets for f in fs))
        inventory[f"{dataset}_{task}"] = all_files

print("=" * 70)
print(f"{'Paradigm':<30} {'Condition files present':<40}")
print("=" * 70)
for name, files in inventory.items():
    print(f"{name:<30} {files}")

Paradigm                       Condition files present                 
ERP_CORE_P3                    ['epochs-epo.fif', 'standard-epo.fif', 'target-epo.fif']
ERP_CORE_MMN                   ['deviant-epo.fif', 'standard-epo.fif']
ERP_CORE_ERN                   ['correct-epo.fif', 'error-epo.fif']
ERP_CORE_N400                  ['related-epo.fif', 'unrelated-epo.fif']
ERP_CORE_N2pc                  ['left-epo.fif', 'right-epo.fif']
ERP_CORE_LRP                   ['epochs-epo.fif', 'left-epo.fif', 'right-epo.fif']
ERP_CORE_N170                  ['1-epo.fif', '10-epo.fif', '101-epo.fif', '102-epo.fif', '103-epo.fif', '104-epo.fif', '105-epo.fif', '106-epo.fif', '107-epo.fif', '108-epo.fif', '109-epo.fif', '11-epo.fif', '110-epo.fif', '111-epo.fif', '112-epo.fif', '113-epo.fif', '114-epo.fif', '115-epo.fif', '116-epo.fif', '117-epo.fif', '118-epo.fif', '119-epo.fif', '12-epo.fif', '120-epo.fif', '121-epo.fif', '122-epo.fif', '123-epo.fif', '124-epo.fif', '125-epo.fif', '126-epo.fif', '127

In [73]:
# ============================================================
# NOTEBOOK CELL: Resolve numeric Stimulus codes for flanker/visualsearch
# Check event_registry.py / event_registry equivalent for these datasets
# to find the documented meaning of each code.
# ============================================================
import mne

for task in ["flanker", "visualsearch"]:
    files = glob.glob(str(project_root / "results_batch" / "DS006018" / "sub-001" / task / "*-epo.fif"))
    if files:
        ep = mne.read_epochs(files[0], preload=False, verbose=False)
        print(f"{task}: event_id = {ep.event_id}")

print("\n" + "=" * 60)
print("Searching event_registry.py / dataset_registry.py for DS006018 code definitions")
print("=" * 60)
registry_path = project_root / "preprocessing_batch" / "event_registry.py"
text = registry_path.read_text()
# Print the DS006018-relevant section
in_section = False
for line in text.splitlines():
    if "DS006018" in line or "ds006018" in line.lower():
        in_section = True
    if in_section:
        print(line)
    if in_section and line.strip() == "}" and "DS006018" not in line:
        # crude section-end heuristic; print a bit more context regardless
        pass

flanker: event_id = {'Stimulus/S 11': 11, 'Stimulus/S 12': 12, 'Stimulus/S 21': 21, 'Stimulus/S 22': 22, 'Stimulus/S111': 111, 'Stimulus/S112': 112, 'Stimulus/S121': 121, 'Stimulus/S122': 122, 'Stimulus/S211': 211, 'Stimulus/S212': 212, 'Stimulus/S221': 221, 'Stimulus/S222': 222}
visualsearch: event_id = {'Stimulus/S111': 111, 'Stimulus/S112': 112, 'Stimulus/S121': 121, 'Stimulus/S122': 122, 'Stimulus/S201': 201, 'Stimulus/S202': 202, 'Stimulus/S211': 211, 'Stimulus/S212': 212, 'Stimulus/S221': 221, 'Stimulus/S222': 222}

Searching event_registry.py / dataset_registry.py for DS006018 code definitions
# DS006018
# ==================================================

DS006018 = {
    "auditoryoddball": {
        "deviant": [70],
        "standard": [80],
    },
    "visualoddball": {
        "deviant": [70],
        "standard": [80],
    },
    # For now, leave flanker and visualsearch as "all events"
    "flanker": {},
    "visualsearch": {},
}

# ========================================

In [75]:
# ============================================================
# CONDITION_SPEC — explicit, authoritative per-task condition mapping.
# No globbing (avoids orphan/generic-file contamination).
# No keyword inference (avoids substring collisions).
# Format: task -> {"class0_file": ..., "class1_file": ..., "exclude": [...]}
# ============================================================

CONDITION_SPEC = {
    # ERP_CORE
    "ERP_CORE_P3":   {"class0": "target",  "class1": "standard", "exclude": ["epochs"]},
    "ERP_CORE_MMN":  {"class0": "deviant", "class1": "standard", "exclude": []},
    "ERP_CORE_ERN":  {"class0": "error",   "class1": "correct",  "exclude": []},
    "ERP_CORE_N400": {"class0": "related", "class1": "unrelated","exclude": []},
    "ERP_CORE_N2pc": {"class0": "left",    "class1": "right",    "exclude": []},
    "ERP_CORE_LRP":  {"class0": "left",    "class1": "right",    "exclude": ["epochs"]},
    "ERP_CORE_N170": None,  # BLOCKED -- needs real face/car trigger-code mapping from dataset docs

    # DS006018
    "DS006018_auditoryoddball": {"class0": "deviant", "class1": "standard", "exclude": ["epochs"]},
    "DS006018_visualoddball":   {"class0_code": 70, "class1_code": 80, "single_file": "Stimulus"},
    "DS006018_flanker":    None,  # BLOCKED -- undefined in event_registry.py, needs paradigm design decision
    "DS006018_visualsearch": None,  # BLOCKED -- same

    # DS002680 -- target(go) vs distractor(no-go) is the real binary task;
    # correct/incorrect is a SEPARATE dimension (response accuracy) and
    # must NOT be mixed into the primary label.
    **{f"DS002680_gonogo_{i}": {"class0": "target", "class1": "distractor", "exclude": ["epochs", "correct", "incorrect"]} for i in range(1, 14)},
}

In [76]:
# ============================================================
# NOTEBOOK CELL: Check raw N170 events.tsv / events.json for
# documented trigger-code meanings (face/car/scrambled categories)
# ============================================================
import json
import pandas as pd

sub = "sub-001"
events_tsv = list((project_root / "data" / "raw" / "erp_core" / sub).rglob("*task-N170_events.tsv"))
events_json = list((project_root / "data" / "raw" / "erp_core" / sub).rglob("*task-N170_events.json"))

print("=" * 60)
print("events.json sidecar (often documents code meanings)")
print("=" * 60)
if events_json:
    print(json.dumps(json.loads(events_json[0].read_text()), indent=2))
else:
    print("No events.json found")

print("\n" + "=" * 60)
print("events.tsv -- unique trial_type / value combinations")
print("=" * 60)
if events_tsv:
    df = pd.read_csv(events_tsv[0], sep="\t")
    print(f"Columns: {list(df.columns)}")
    if "trial_type" in df.columns:
        print("\nUnique trial_type values:")
        print(df["trial_type"].value_counts().to_string())
    if "value" in df.columns:
        print("\nUnique value (trigger code) x trial_type crosstab:")
        if "trial_type" in df.columns:
            print(pd.crosstab(df["value"], df["trial_type"]).to_string())
        else:
            print(df["value"].value_counts().sort_index().to_string())

# Also check the dataset-level task description, which BIDS requires
print("\n" + "=" * 60)
print("Dataset-level task description (task-N170_events.json at dataset root, if present)")
print("=" * 60)
root_json = project_root / "data" / "raw" / "erp_core" / "task-N170_events.json"
if root_json.exists():
    print(json.dumps(json.loads(root_json.read_text()), indent=2))
else:
    print("Not found at dataset root")

events.json sidecar (often documents code meanings)
No events.json found

events.tsv -- unique trial_type / value combinations
Columns: ['onset', 'duration', 'sample', 'trial_type', 'stim_file', 'value']

Unique trial_type values:
trial_type
response    320
stimulus    320

Unique value (trigger code) x trial_type crosstab:
trial_type  response  stimulus
value                         
1                  0         2
2                  0         2
3                  0         2
4                  0         2
5                  0         2
6                  0         2
7                  0         2
8                  0         2
9                  0         2
10                 0         2
11                 0         2
12                 0         2
13                 0         2
14                 0         2
15                 0         2
16                 0         2
17                 0         2
18                 0         2
19                 0         2
20                 0   

In [78]:
# ============================================================
# CONDITION_SPEC v2 — complete, ground-truth-verified.
# N170 codes confirmed via task-N170_events.json BIDS sidecar
# (dataset-level documentation, not inferred).
# ============================================================

CONDITION_SPEC = {
    # ERP_CORE
    "ERP_CORE_P3":   {"class0": "target",  "class1": "standard", "exclude": ["epochs"]},
    "ERP_CORE_MMN":  {"class0": "deviant", "class1": "standard", "exclude": []},
    "ERP_CORE_ERN":  {"class0": "error",   "class1": "correct",  "exclude": []},
    "ERP_CORE_N400": {"class0": "related", "class1": "unrelated","exclude": []},
    "ERP_CORE_N2pc": {"class0": "left",    "class1": "right",    "exclude": []},
    "ERP_CORE_LRP":  {"class0": "left",    "class1": "right",    "exclude": ["epochs"]},
    "ERP_CORE_N170": {"class0_range": (1, 40), "class1_range": (41, 80), "single_file_codes": True},

    # DS006018
    "DS006018_auditoryoddball": {"class0": "deviant", "class1": "standard", "exclude": ["epochs"]},
    "DS006018_visualoddball":   {"class0_code": 70, "class1_code": 80, "single_file": "Stimulus"},
    # flanker / visualsearch: OUT OF SCOPE per your decision -- undefined in event_registry.py

    # DS002680 -- target(go) vs distractor(no-go) is the real binary task.
    # correct/incorrect is a SEPARATE dimension (response accuracy),
    # same trap as N170's 201/202 -- excluded from the primary label.
    **{f"DS002680_gonogo_{i}": {"class0": "target", "class1": "distractor", "exclude": ["epochs", "correct", "incorrect"]} for i in range(1, 14)},
}

In [79]:
# ============================================================
# NOTEBOOK CELL: explicit_epoch_loader.py -- authoritative,
# spec-driven Phase 3 loader. Replaces glob+keyword logic entirely.
# Written as a new file so the old (buggy) epoch_loader.py is
# preserved for reference/diff, never called again going forward.
# ============================================================
loader_code = '''
"""
explicit_epoch_loader.py -- authoritative, spec-driven epoch loader.

Design principles (all three prior bugs this session trace back to
violating one of these):
  1. No globbing beyond exact filenames named in CONDITION_SPEC.
     (Fixes: orphaned stale files being silently included.)
  2. No keyword substring matching. Conditions are matched by exact
     file stem or exact numeric code range, nothing else.
     (Fixes: 'related' matching inside 'unrelated', etc.)
  3. A task with no CONDITION_SPEC entry raises an error, not a
     silent fallback (parity-based or otherwise).
     (Fixes: flanker/visualsearch/N170 running on a meaningless
     parity fallback without anyone knowing.)
"""
import numpy as np
import mne
from pathlib import Path

CONDITION_SPEC = {
    "ERP_CORE_P3":   {"class0": "target",  "class1": "standard", "exclude": ["epochs"]},
    "ERP_CORE_MMN":  {"class0": "deviant", "class1": "standard", "exclude": []},
    "ERP_CORE_ERN":  {"class0": "error",   "class1": "correct",  "exclude": []},
    "ERP_CORE_N400": {"class0": "related", "class1": "unrelated","exclude": []},
    "ERP_CORE_N2pc": {"class0": "left",    "class1": "right",    "exclude": []},
    "ERP_CORE_LRP":  {"class0": "left",    "class1": "right",    "exclude": ["epochs"]},
    "ERP_CORE_N170": {"class0_range": (1, 40), "class1_range": (41, 80), "single_file_codes": True},

    "DS006018_auditoryoddball": {"class0": "deviant", "class1": "standard", "exclude": ["epochs"]},
    "DS006018_visualoddball":   {"class0_code": 70, "class1_code": 80, "single_file": "Stimulus"},

    **{f"DS002680_gonogo_{i}": {"class0": "target", "class1": "distractor",
                                  "exclude": ["epochs", "correct", "incorrect"]} for i in range(1, 14)},
}


def load_explicit(results_batch_dir: Path, dataset_name: str, task_name: str):
    """
    Loads exactly the trials CONDITION_SPEC says belong to this task's
    binary classification, from real subject folders, with no globbing
    of unlisted files and no keyword inference.
    """
    spec_key = f"{dataset_name}_{task_name}"
    if spec_key not in CONDITION_SPEC:
        raise ValueError(
            f"'{spec_key}' has no entry in CONDITION_SPEC. Refusing to load with a "
            f"fallback -- this is exactly the failure mode that caused undetected "
            f"issues this session. Add an explicit, verified entry before loading."
        )
    spec = CONDITION_SPEC[spec_key]

    task_dirs = sorted((results_batch_dir / dataset_name).glob(f"sub-*/{task_name}"))
    if not task_dirs:
        raise FileNotFoundError(f"No subject folders found for {spec_key}")

    X_list, y_list, sub_list = [], [], []
    ch_names, sfreq = None, None

    for tdir in task_dirs:
        sub_name = tdir.parent.name

        if "single_file" in spec:
            # e.g. DS006018_visualoddball: one Stimulus-epo.fif, split by numeric code
            fpath = tdir / f"{spec['single_file']}-epo.fif"
            if not fpath.exists():
                continue
            epochs = mne.read_epochs(fpath, preload=True, verbose=False)
            epochs.pick("eeg", verbose=False)
            codes = epochs.events[:, -1]
            # map MNE's internal event id back to the real code via event_id dict
            inv_id = {v: k for k, v in epochs.event_id.items()}
            real_codes = np.array([int(inv_id[c].split()[-1].replace("S", "")) if inv_id[c].split()[-1].replace("S","").isdigit() else -1 for c in codes])
            mask0 = real_codes == spec["class0_code"]
            mask1 = real_codes == spec["class1_code"]
            data = epochs.get_data()
            if mask0.any():
                X_list.append(data[mask0]); y_list.append(np.zeros(mask0.sum(), dtype=np.int64))
                sub_list.append(np.array([sub_name] * mask0.sum()))
            if mask1.any():
                X_list.append(data[mask1]); y_list.append(np.ones(mask1.sum(), dtype=np.int64))
                sub_list.append(np.array([sub_name] * mask1.sum()))
            if ch_names is None:
                ch_names, sfreq = epochs.ch_names, epochs.info["sfreq"]

        elif "single_file_codes" in spec:
            # e.g. N170: many numeric-code files, class determined by code RANGE
            for fpath in tdir.glob("*-epo.fif"):
                stem = fpath.stem.replace("-epo", "")
                if not stem.isdigit():
                    continue
                code = int(stem)
                lo0, hi0 = spec["class0_range"]
                lo1, hi1 = spec["class1_range"]
                if lo0 <= code <= hi0:
                    cls = 0
                elif lo1 <= code <= hi1:
                    cls = 1
                else:
                    continue  # out of range (e.g. scrambled/response codes) -- correctly excluded
                epochs = mne.read_epochs(fpath, preload=True, verbose=False)
                epochs.pick("eeg", verbose=False)
                if len(epochs) == 0:
                    continue
                data = epochs.get_data()
                X_list.append(data)
                y_list.append(np.full(len(epochs), cls, dtype=np.int64))
                sub_list.append(np.array([sub_name] * len(epochs)))
                if ch_names is None:
                    ch_names, sfreq = epochs.ch_names, epochs.info["sfreq"]

        else:
            # Standard case: exactly two named condition files
            for cls, key in [(0, "class0"), (1, "class1")]:
                fpath = tdir / f"{spec[key]}-epo.fif"
                if not fpath.exists():
                    continue
                epochs = mne.read_epochs(fpath, preload=True, verbose=False)
                epochs.pick("eeg", verbose=False)
                if len(epochs) == 0:
                    continue
                data = epochs.get_data()
                X_list.append(data)
                y_list.append(np.full(len(epochs), cls, dtype=np.int64))
                sub_list.append(np.array([sub_name] * len(epochs)))
                if ch_names is None:
                    ch_names, sfreq = epochs.ch_names, epochs.info["sfreq"]

    if not X_list:
        raise FileNotFoundError(f"No valid trials loaded for {spec_key}")

    X = np.concatenate(X_list, axis=0).astype(np.float32)
    y = np.concatenate(y_list, axis=0)
    subject_ids = np.concatenate(sub_list, axis=0)

    print(f"[Explicit Loader] {spec_key}: X={X.shape}, y={y.shape}, "
          f"classes={np.unique(y)}, subjects={len(np.unique(subject_ids))}, "
          f"class_balance={np.bincount(y).tolist()}")

    return X, y, subject_ids, ch_names, sfreq
'''

loader_path = project_root / "dl_prep" / "explicit_epoch_loader.py"
loader_path.write_text(loader_code)
print(f"[Created] {loader_path}")

[Created] /lp-dev/23BRS1328/erp_trial_min/dl_prep/explicit_epoch_loader.py


In [80]:
# ============================================================
# NOTEBOOK CELL: Regenerate Phase 3 for all 20 spec-defined
# paradigms using the new explicit loader. Backs up existing
# payloads first. flanker/visualsearch skipped (out of scope).
# ============================================================
import sys, shutil, importlib
from datetime import datetime
sys.path.insert(0, str(project_root))

import dl_prep.explicit_epoch_loader as eel
importlib.reload(eel)

from dl_prep.data_splitter import subject_wise_split
from dl_prep.scaler import EEGChannelScaler
from configs.dl_config import PREP_CONFIG, RESULTS_DL_DIR
import torch, os

datasets_dir = RESULTS_DL_DIR / "datasets"
backup_dir = datasets_dir / f"_pre_explicit_spec_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
backup_dir.mkdir(parents=True, exist_ok=True)

ALL_SPEC_TASKS = [
    ("ERP_CORE", t) for t in ["P3", "MMN", "ERN", "N400", "N2pc", "LRP", "N170"]
] + [
    ("DS006018", "auditoryoddball"), ("DS006018", "visualoddball"),
] + [
    ("DS002680", f"gonogo_{i}") for i in range(1, 14)
]

results_log = []
for dataset_name, task_name in ALL_SPEC_TASKS:
    target_name = f"{dataset_name}_{task_name}"
    save_file = datasets_dir / f"{target_name}_prepared.pt"

    if save_file.exists():
        shutil.copy2(save_file, backup_dir / save_file.name)

    print(f"\n{'='*60}\n{target_name}\n{'='*60}")
    try:
        X, y, subject_ids, ch_names, sfreq = eel.load_explicit(
            project_root / "results_batch", dataset_name, task_name
        )

        train_tuple, val_tuple, test_tuple = subject_wise_split(
            X, y, subject_ids,
            train_ratio=PREP_CONFIG["train_ratio"], val_ratio=PREP_CONFIG["val_ratio"],
            test_ratio=PREP_CONFIG["test_ratio"], seed=PREP_CONFIG["random_seed"],
        )
        X_train, y_train, sub_train = train_tuple
        X_val, y_val, sub_val = val_tuple
        X_test, y_test, sub_test = test_tuple

        scaler = EEGChannelScaler()
        X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
        X_val_scaled = scaler.transform(X_val).astype(np.float32)
        X_test_scaled = scaler.transform(X_test).astype(np.float32)

        payload = {
            "dataset_name": dataset_name, "task_name": task_name,
            "train": (torch.from_numpy(X_train_scaled), torch.from_numpy(y_train), list(sub_train)),
            "val": (torch.from_numpy(X_val_scaled), torch.from_numpy(y_val), list(sub_val)),
            "test": (torch.from_numpy(X_test_scaled), torch.from_numpy(y_test), list(sub_test)),
            "ch_names": ch_names, "sfreq": sfreq, "num_classes": len(set(y)),
        }
        torch.save(payload, save_file, pickle_protocol=5)
        results_log.append({"target": target_name, "status": "ok", "n_trials": len(y),
                             "n_subjects": len(set(subject_ids)), "class_balance": np.bincount(y).tolist()})
    except Exception as e:
        print(f"[ERROR] {target_name}: {e}")
        results_log.append({"target": target_name, "status": f"error: {e}"})

import pandas as pd
df_log = pd.DataFrame(results_log)
print("\n" + "=" * 60 + "\nREGENERATION SUMMARY\n" + "=" * 60)
print(df_log.to_string())


ERP_CORE_P3
[Explicit Loader] ERP_CORE_P3: X=(1828, 30, 1025), y=(1828,), classes=[0 1], subjects=36, class_balance=[370, 1458]
[Split] Train Subjects (25): 1235 trials
[Split] Val Subjects   (5): 239 trials
[Split] Test Subjects  (6): 354 trials

ERP_CORE_MMN
[Explicit Loader] ERP_CORE_MMN: X=(26249, 30, 1025), y=(26249,), classes=[0 1], subjects=40, class_balance=[5316, 20933]
[Split] Train Subjects (28): 17418 trials
[Split] Val Subjects   (6): 5195 trials
[Split] Test Subjects  (6): 3636 trials

ERP_CORE_ERN
[Explicit Loader] ERP_CORE_ERN: X=(7526, 30, 1025), y=(7526,), classes=[0 1], subjects=37, class_balance=[3895, 3631]
[Split] Train Subjects (25): 5281 trials
[Split] Val Subjects   (5): 683 trials
[Split] Test Subjects  (7): 1562 trials

ERP_CORE_N400
[Explicit Loader] ERP_CORE_N400: X=(5698, 30, 1025), y=(5698,), classes=[0 1], subjects=35, class_balance=[2981, 2717]
[Split] Train Subjects (24): 4084 trials
[Split] Val Subjects   (5): 760 trials
[Split] Test Subjects  (6): 8

In [81]:
# ============================================================
# NOTEBOOK CELL: Fix single_file event-code parsing
# Root cause: assumed "Stimulus/S##" name format; visualoddball's
# actual event_id names differ. Fix: extract trailing digits via
# regex instead of assuming a fixed prefix format.
# ============================================================
import re

loader_path = project_root / "dl_prep" / "explicit_epoch_loader.py"
text = loader_path.read_text()

old_block = '''            inv_id = {v: k for k, v in epochs.event_id.items()}
            real_codes = np.array([int(inv_id[c].split()[-1].replace("S", "")) if inv_id[c].split()[-1].replace("S","").isdigit() else -1 for c in codes])'''

new_block = '''            inv_id = {v: k for k, v in epochs.event_id.items()}
            def _extract_code(name):
                m = re.search(r"(\\d+)$", name)
                return int(m.group(1)) if m else -1
            real_codes = np.array([_extract_code(inv_id[c]) for c in codes])'''

assert old_block in text, "Expected block not found"
text = text.replace(old_block, new_block).replace(
    "import mne\nfrom pathlib import Path",
    "import mne\nimport re\nfrom pathlib import Path"
)
loader_path.write_text(text)
print("[Patched] explicit_epoch_loader.py -- regex-based code extraction")

[Patched] explicit_epoch_loader.py -- regex-based code extraction


In [82]:
# ============================================================
# NOTEBOOK CELL: Diagnose visualoddball's actual event_id names
# before re-running, so we don't guess twice.
# ============================================================
import mne
files = list((project_root / "results_batch" / "DS006018" / "sub-001" / "visualoddball").glob("*-epo.fif"))
if files:
    ep = mne.read_epochs(files[0], preload=False, verbose=False)
    print(f"event_id: {ep.event_id}")

event_id: {'Stimulus/S 11': 11, 'Stimulus/S 12': 12, 'Stimulus/S 13': 13, 'Stimulus/S 14': 14, 'Stimulus/S 15': 15, 'Stimulus/S 21': 21, 'Stimulus/S 22': 22, 'Stimulus/S 23': 23, 'Stimulus/S 24': 24, 'Stimulus/S 25': 25, 'Stimulus/S 31': 31, 'Stimulus/S 32': 32, 'Stimulus/S 33': 33, 'Stimulus/S 34': 34, 'Stimulus/S 35': 35, 'Stimulus/S 41': 41, 'Stimulus/S 42': 42, 'Stimulus/S 43': 43, 'Stimulus/S 44': 44, 'Stimulus/S 45': 45, 'Stimulus/S 51': 51, 'Stimulus/S 52': 52, 'Stimulus/S 53': 53, 'Stimulus/S 54': 54, 'Stimulus/S 55': 55, 'Stimulus/S201': 201, 'Stimulus/S202': 202}


In [83]:
# ============================================================
# NOTEBOOK CELL: Check DS006018 visualoddball events.json sidecar
# ============================================================
import json

sidecar_paths = list((project_root / "data" / "raw" / "ds006018").rglob("*visualoddball*events.json"))
print(f"Found {len(sidecar_paths)} sidecar(s)")
for p in sidecar_paths[:3]:
    print(f"\n{p}")
    print(json.dumps(json.loads(p.read_text()), indent=2))

# Also check the dataset-level task description (BIDS convention: often at dataset root)
root_candidates = list((project_root / "data" / "raw" / "ds006018").glob("task-visualoddball*events.json"))
print(f"\nDataset-root-level sidecar: {root_candidates}")
for p in root_candidates:
    print(json.dumps(json.loads(p.read_text()), indent=2))

# And the raw events.tsv trial_type / value crosstab, same method as N170
import pandas as pd
tsv_paths = list((project_root / "data" / "raw" / "ds006018").rglob("*visualoddball*events.tsv"))
if tsv_paths:
    df = pd.read_csv(tsv_paths[0], sep="\t")
    print(f"\nColumns: {list(df.columns)}")
    if "trial_type" in df.columns:
        print(df["trial_type"].value_counts().to_string())
    if "value" in df.columns and "trial_type" in df.columns:
        print("\nvalue x trial_type crosstab:")
        print(pd.crosstab(df["value"], df["trial_type"]).to_string())

Found 98 sidecar(s)

/lp-dev/23BRS1328/erp_trial_min/data/raw/ds006018/sub-001/eeg/sub-001_task-visualoddball_events.json
{
  "onset": {
    "Description": "Start time of the event, in seconds."
  },
  "duration": {
    "Description": "Duration of the event (required in BIDS but not a part of the EEG.event structure; therefore consists of 0 values"
  },
  "sample": {
    "Description": "Sample point of the event in data points, relative to the start of the EEG recording.",
    "Units": "data points"
  },
  "value": {
    "Description": "The code used to indicate an event marker.",
    "Levels": {
      "boundary": "recording beginning",
      "11": "Target (for that block) A, stimulus type A",
      "21": "Target (for that block) B, stimulus type A",
      "31": "Target (for that block) C, stimulus type A",
      "41": "Target (for that block) D, stimulus type A",
      "51": "Target (for that block) E, stimulus type A",
      "12": "Target (for that block) A, stimulus type B",
      "

In [84]:
# ============================================================
# NOTEBOOK CELL: Fix DS006018_visualoddball spec -- digit-match rule
# Target (class0) = code where tens digit == units digit (11,22,33,44,55)
# Standard (class1) = code where tens digit != units digit, code in 11-55
# Verified via BIDS sidecar: p=.2 match rate = classic oddball design.
# ============================================================
loader_path = project_root / "dl_prep" / "explicit_epoch_loader.py"
text = loader_path.read_text()

old_spec_line = '    "DS006018_visualoddball":   {"class0_code": 70, "class1_code": 80, "single_file": "Stimulus"},'
new_spec_line = '    "DS006018_visualoddball":   {"digit_match_rule": True, "single_file": "Stimulus", "valid_range": (11, 55)},'
assert old_spec_line in text
text = text.replace(old_spec_line, new_spec_line)

# Add digit-match handling to the single_file branch
old_logic = '''            mask0 = real_codes == spec["class0_code"]
            mask1 = real_codes == spec["class1_code"]'''
new_logic = '''            if spec.get("digit_match_rule"):
                lo, hi = spec["valid_range"]
                in_range = (real_codes >= lo) & (real_codes <= hi) & (real_codes < 100)
                tens = real_codes // 10
                units = real_codes % 10
                mask0 = in_range & (tens == units)   # target: block-target matches shown stimulus
                mask1 = in_range & (tens != units)   # standard: mismatch
            else:
                mask0 = real_codes == spec["class0_code"]
                mask1 = real_codes == spec["class1_code"]'''
assert old_logic in text
text = text.replace(old_logic, new_logic)

loader_path.write_text(text)
print("[Patched] DS006018_visualoddball now uses verified digit-match rule")

[Patched] DS006018_visualoddball now uses verified digit-match rule


In [85]:
# ============================================================
# NOTEBOOK CELL: Regenerate DS006018_visualoddball only
# ============================================================
import importlib
import dl_prep.explicit_epoch_loader as eel
importlib.reload(eel)

X, y, subject_ids, ch_names, sfreq = eel.load_explicit(
    project_root / "results_batch", "DS006018", "visualoddball"
)

import numpy as np
print(f"\nClass balance: {np.bincount(y).tolist()}")
print(f"Target rate: {np.bincount(y)[0] / len(y) * 100:.1f}% (expect ~20% per oddball design)")

# If this looks right, save the full payload same as before
train_tuple, val_tuple, test_tuple = subject_wise_split(
    X, y, subject_ids, train_ratio=PREP_CONFIG["train_ratio"], val_ratio=PREP_CONFIG["val_ratio"],
    test_ratio=PREP_CONFIG["test_ratio"], seed=PREP_CONFIG["random_seed"],
)
X_train, y_train, sub_train = train_tuple
X_val, y_val, sub_val = val_tuple
X_test, y_test, sub_test = test_tuple

scaler = EEGChannelScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

payload = {
    "dataset_name": "DS006018", "task_name": "visualoddball",
    "train": (torch.from_numpy(X_train_scaled), torch.from_numpy(y_train), list(sub_train)),
    "val": (torch.from_numpy(X_val_scaled), torch.from_numpy(y_val), list(sub_val)),
    "test": (torch.from_numpy(X_test_scaled), torch.from_numpy(y_test), list(sub_test)),
    "ch_names": ch_names, "sfreq": sfreq, "num_classes": len(set(y)),
}
save_file = RESULTS_DL_DIR / "datasets" / "DS006018_visualoddball_prepared.pt"
torch.save(payload, save_file, pickle_protocol=5)
print(f"\nSaved: {save_file}")

[Explicit Loader] DS006018_visualoddball: X=(13798, 26, 501), y=(13798,), classes=[0 1], subjects=95, class_balance=[2721, 11077]

Class balance: [2721, 11077]
Target rate: 19.7% (expect ~20% per oddball design)
[Split] Train Subjects (66): 9567 trials
[Split] Val Subjects   (14): 1869 trials
[Split] Test Subjects  (15): 2362 trials

Saved: /lp-dev/23BRS1328/erp_trial_min/results_dl/datasets/DS006018_visualoddball_prepared.pt


In [86]:
# ============================================================
# NOTEBOOK CELL: Final verification -- all 20 explicit-spec payloads
# Same checks that caught every real bug this session:
#   1. Single-class-subject check (caught the LRP scrambled-label bug)
#   2. Majority-class-fraction vs n_trials (sanity on class balance)
# ============================================================
import torch
import numpy as np
import pandas as pd
from configs.dl_config import RESULTS_DL_DIR

datasets_dir = RESULTS_DL_DIR / "datasets"

ALL_SPEC_TASKS = [
    "ERP_CORE_P3", "ERP_CORE_MMN", "ERP_CORE_ERN", "ERP_CORE_N400",
    "ERP_CORE_N2pc", "ERP_CORE_LRP", "ERP_CORE_N170",
    "DS006018_auditoryoddball", "DS006018_visualoddball",
] + [f"DS002680_gonogo_{i}" for i in range(1, 14)]

rows = []
for target_name in ALL_SPEC_TASKS:
    pt_file = datasets_dir / f"{target_name}_prepared.pt"
    if not pt_file.exists():
        rows.append({"target_name": target_name, "status": "MISSING"})
        continue

    payload = torch.load(pt_file, weights_only=False)
    split_summaries = {}
    max_single_class_pct = 0

    for split_name in ["train", "val", "test"]:
        X, y, subs = payload[split_name]
        y_np = y.numpy() if isinstance(y, torch.Tensor) else np.asarray(y)
        subs_np = np.asarray(subs)

        df = pd.DataFrame({"subject": subs_np, "label": y_np})
        per_subject_nunique = df.groupby("subject")["label"].nunique()
        n_single = (per_subject_nunique == 1).sum()
        n_total_subs = per_subject_nunique.shape[0]
        pct_single = n_single / max(n_total_subs, 1)
        max_single_class_pct = max(max_single_class_pct, pct_single)

        counts = np.bincount(y_np, minlength=2)
        majority_frac = counts.max() / counts.sum() if counts.sum() > 0 else np.nan
        split_summaries[split_name] = {"n_trials": len(y_np), "n_subjects": n_total_subs,
                                         "pct_single_class": round(pct_single, 3), "majority_frac": round(majority_frac, 3)}

    rows.append({
        "target_name": target_name,
        "status": "CLEAN" if max_single_class_pct == 0 else f"FLAG ({max_single_class_pct*100:.0f}% single-class)",
        "train_trials": split_summaries["train"]["n_trials"],
        "test_trials": split_summaries["test"]["n_trials"],
        "test_majority_frac": split_summaries["test"]["majority_frac"],
        "test_pct_single_class": split_summaries["test"]["pct_single_class"],
    })

df_final = pd.DataFrame(rows)
pd.set_option("display.max_rows", 25)
pd.set_option("display.width", 150)
print(df_final.to_string(index=False))

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Clean: {(df_final['status'] == 'CLEAN').sum()} / {len(df_final)}")
print(f"Flagged: {(df_final['status'].str.startswith('FLAG')).sum()}")
print(f"Missing: {(df_final['status'] == 'MISSING').sum()}")

             target_name                  status  train_trials  test_trials  test_majority_frac  test_pct_single_class
             ERP_CORE_P3  FLAG (4% single-class)          1235          354               0.771                    0.0
            ERP_CORE_MMN                   CLEAN         17418         3636               0.799                    0.0
            ERP_CORE_ERN                   CLEAN          5281         1562               0.520                    0.0
           ERP_CORE_N400                   CLEAN          4084          854               0.511                    0.0
           ERP_CORE_N2pc                   CLEAN          6333         1775               0.504                    0.0
            ERP_CORE_LRP                   CLEAN         11197         3269               0.522                    0.0
           ERP_CORE_N170  FLAG (4% single-class)          2268          799               0.507                    0.0
DS006018_auditoryoddball                   CLEAN

In [3]:
# ============================================================
# NOTEBOOK CELL: Restore session state for today
# ============================================================
import random
import numpy as np
import torch
import pandas as pd
import sys
from pathlib import Path

def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def encode_subjects(subjects_array):
    subs = subjects_array.numpy() if isinstance(subjects_array, torch.Tensor) else np.asarray(subjects_array)
    if subs.dtype.kind in ("U", "S", "O"):
        _, codes = np.unique(subs, return_inverse=True)
        return codes.astype(np.int64)
    return subs.astype(np.int64)

project_root = Path("/lp-dev/23BRS1328/erp_trial_min")
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "stage3_feature_extraction"))

from configs.dl_config import RESULTS_DL_DIR, TRAIN_CONFIG, PREP_CONFIG
from dl_prep.eeg_dataset import create_dataloaders
from dl_prep.data_splitter import subject_wise_split
from dl_prep.scaler import EEGChannelScaler
from dl_models.cnn1d import ERPConv1DNet
from dl_models.trainer import Trainer, LabelSmoothingCrossEntropy
from dl_models.evaluator import evaluate_model
import dl_prep.explicit_epoch_loader as eel

OPT_DIR = RESULTS_DL_DIR / "optimization"
(OPT_DIR / "checkpoints_cnn1d").mkdir(parents=True, exist_ok=True)
(OPT_DIR / "figures_cnn1d").mkdir(parents=True, exist_ok=True)

ALL_SPEC_TASKS = [
    ("ERP_CORE", t) for t in ["P3", "MMN", "ERN", "N400", "N2pc", "LRP", "N170"]
] + [
    ("DS006018", "auditoryoddball"), ("DS006018", "visualoddball"),
] + [
    ("DS002680", f"gonogo_{i}") for i in range(1, 14)
]

print(f"project_root = {project_root}")
print(f"Datasets present: {len(list((RESULTS_DL_DIR / 'datasets').glob('*_prepared.pt')))} prepared payloads found")
print("Session state restored.")

project_root = /lp-dev/23BRS1328/erp_trial_min
Datasets present: 24 prepared payloads found
Session state restored.


In [4]:
# ============================================================
# NOTEBOOK CELL: Inspect assemble_features.py -- how does Stage 3
# turn per-file features into a labeled ML-ready feature matrix?
# This determines whether the generic epochs-epo.fif files (still
# present for P3/LRP/gonogo) cause a labeling problem downstream.
# ============================================================
assemble_path = project_root / "stage3_feature_extraction" / "assemble_features.py"
print(assemble_path.read_text())

# stage3_feature_extraction/assemble_features.py
# Phase 3.7 — Feature Assembly
#
# Combines, for every paradigm (dataset__task) and every subsampling step
# (config3.SUBSAMPLE_STEPS), the outputs of 3.1 (trial indices), 3.3 (band
# power), 3.4 (PLV), 3.5 (wPLI), and 3.6 (entropy) into ONE unified feature
# matrix ready for Stage 1 ML input.
#
# Per-epoch feature vector =
#   [ band_power.flatten()        -> n_ch * n_bands
#     PLV_alpha_flat               -> n_ch*(n_ch-1)/2
#     PLV_beta_flat                -> n_ch*(n_ch-1)/2
#     wPLI_alpha_flat              -> n_ch*(n_ch-1)/2
#     wPLI_beta_flat               -> n_ch*(n_ch-1)/2
#     entropy                      -> n_ch ]
#
# Per paradigm/step, subjects are pooled (rows stacked) into ONE matrix:
#   results_batch/stage3/feature_matrices/<paradigm>/<paradigm>_step-<step>.npy
#       shape: (n_trials_pooled, n_features)
#   results_batch/stage3/feature_matrices/<paradigm>/<paradigm>_step-<step>_meta.csv
#       columns: row_index,

In [5]:
# ============================================================
# NOTEBOOK CELL: Find where Stage 3 feature matrices get labeled
# for Stage 1 ML input. Read-only.
# ============================================================
import subprocess

# Search for anything that reads feature_matrices/ and assigns labels
out = subprocess.run(
    ["grep", "-rln", "feature_matrices\\|file_id", str(project_root / "stage4_classical_ml")],
    capture_output=True, text=True
)
print("Files in stage4_classical_ml/ referencing feature_matrices or file_id:")
print(out.stdout or out.stderr)

print("\n" + "=" * 60)
print("stage4_classical_ml/ full directory listing")
print("=" * 60)
stage4_dir = project_root / "stage4_classical_ml"
if stage4_dir.exists():
    for f in sorted(stage4_dir.glob("*.py")):
        print(f"  {f.name}")
else:
    print("Directory does not exist")

print("\n" + "=" * 60)
print("Existing ml/ output -- check for a label column anywhere")
print("=" * 60)
ml_dir = project_root / "stage3_results" / "ml"
for f in sorted(ml_dir.glob("*.csv"))[:3]:
    print(f"\n{f.name}:")
    print(pd.read_csv(f).head(3).to_string())

Files in stage4_classical_ml/ referencing feature_matrices or file_id:
grep: /lp-dev/23BRS1328/erp_trial_min/stage4_classical_ml: No such file or directory


stage4_classical_ml/ full directory listing
Directory does not exist

Existing ml/ output -- check for a label column anywhere


In [6]:
# ============================================================
# NOTEBOOK CELL: Direct inspection of stage3_results/ml/ contents
# ============================================================
ml_dir = project_root / "stage3_results" / "ml"
print(f"ml_dir exists: {ml_dir.exists()}")
print(f"ml_dir is directory: {ml_dir.is_dir() if ml_dir.exists() else 'n/a'}")

print("\nDirect iterdir() listing:")
for item in sorted(ml_dir.iterdir()):
    print(f"  {item.name}  ({'dir' if item.is_dir() else item.stat().st_size})")

print("\nRecursive glob for ANY file:")
all_files = list(ml_dir.rglob("*"))
print(f"Total items (recursive): {len(all_files)}")
for f in all_files[:20]:
    print(f"  {f.relative_to(ml_dir)}")

ml_dir exists: True
ml_dir is directory: True

Direct iterdir() listing:
  DS002680__gonogo_1  (dir)
  DS002680__gonogo_10  (dir)
  DS002680__gonogo_11  (dir)
  DS002680__gonogo_12  (dir)
  DS002680__gonogo_13  (dir)
  DS002680__gonogo_2  (dir)
  DS002680__gonogo_3  (dir)
  DS002680__gonogo_4  (dir)
  DS002680__gonogo_5  (dir)
  DS002680__gonogo_6  (dir)
  DS002680__gonogo_7  (dir)
  DS002680__gonogo_8  (dir)
  DS002680__gonogo_9  (dir)
  DS006018__auditoryoddball  (dir)
  DS006018__flanker  (dir)
  DS006018__visualoddball  (dir)
  DS006018__visualsearch  (dir)
  ERP_CORE__ERN  (dir)
  ERP_CORE__LRP  (dir)
  ERP_CORE__MMN  (dir)
  ERP_CORE__N170  (dir)
  ERP_CORE__N2pc  (dir)
  ERP_CORE__N400  (dir)
  ERP_CORE__P3  (dir)

Recursive glob for ANY file:
Total items (recursive): 168
  DS002680__gonogo_1
  DS002680__gonogo_10
  DS002680__gonogo_11
  DS002680__gonogo_12
  DS002680__gonogo_13
  DS002680__gonogo_2
  DS002680__gonogo_3
  DS002680__gonogo_4
  DS002680__gonogo_5
  DS002680__gonog

In [7]:
# ============================================================
# NOTEBOOK CELL: Inspect actual contents of one ml/ paradigm folder
# ============================================================
sample_dir = ml_dir / "ERP_CORE__P3"
print(f"Contents of {sample_dir.name}:")
for f in sorted(sample_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size} bytes)")

print("\n" + "=" * 60)
print("CSV contents (if any)")
print("=" * 60)
for f in sorted(sample_dir.glob("*.csv")):
    print(f"\n--- {f.name} ---")
    df = pd.read_csv(f)
    print(f"Shape: {df.shape}, Columns: {list(df.columns)}")
    print(df.head(10).to_string())

print("\n" + "=" * 60)
print("JSON contents (if any)")
print("=" * 60)
import json
for f in sorted(sample_dir.glob("*.json")):
    print(f"\n--- {f.name} ---")
    print(json.dumps(json.loads(f.read_text()), indent=2)[:1500])

print("\n" + "=" * 60)
print("NPY file shapes (if any)")
print("=" * 60)
for f in sorted(sample_dir.glob("*.npy")):
    arr = np.load(f)
    print(f"  {f.name}: shape={arr.shape}, dtype={arr.dtype}")

Contents of ERP_CORE__P3:
  X.npy  (56156288 bytes)
  feature_block_boundaries.json  (267 bytes)
  label_map.json  (218 bytes)
  metadata.csv  (189574 bytes)
  y.npy  (160992 bytes)
  y_codes.npy  (29376 bytes)

CSV contents (if any)

--- metadata.csv ---
Shape: (3656, 7), Columns: ['row_index', 'dataset', 'subject', 'task', 'file_id', 'epoch_index_in_subject', 'label']
   row_index   dataset  subject task     file_id  epoch_index_in_subject        label
0          0  ERP_CORE  sub-001   P3  epochs-epo                       0    target/22
1          1  ERP_CORE  sub-001   P3  epochs-epo                       1  standard/21
2          2  ERP_CORE  sub-001   P3  epochs-epo                       2  standard/21
3          3  ERP_CORE  sub-001   P3  epochs-epo                       3    target/22
4          4  ERP_CORE  sub-001   P3  epochs-epo                       4    target/22
5          5  ERP_CORE  sub-001   P3  epochs-epo                       5    target/22
6          6  ERP_CORE  s

In [8]:
# ============================================================
# NOTEBOOK CELL: Check ml/ output timestamps + label design
# consistency across all 24 paradigms
# ============================================================
import datetime

print("=" * 60)
print("Timestamp check")
print("=" * 60)
mtime = (ml_dir / "ERP_CORE__P3" / "X.npy").stat().st_mtime
print(f"ERP_CORE__P3/X.npy last modified: {datetime.datetime.fromtimestamp(mtime)}")
print(f"(Our Phase 2 fix was 2026-08-03; orphaned-file cleanup was today)")

print("\n" + "=" * 60)
print("Label design across all paradigms: binary or multi-class?")
print("=" * 60)
for pdir in sorted(ml_dir.iterdir()):
    label_map_path = pdir / "label_map.json"
    if label_map_path.exists():
        lm = json.loads(label_map_path.read_text())
        print(f"{pdir.name}: {len(lm)} classes -- {list(lm.values())[:4]}{'...' if len(lm)>4 else ''}")

Timestamp check
ERP_CORE__P3/X.npy last modified: 2026-06-29 21:59:22.456824
(Our Phase 2 fix was 2026-08-03; orphaned-file cleanup was today)

Label design across all paradigms: binary or multi-class?
DS002680__gonogo_1: 4 classes -- ['correct/3', 'distractor/1', 'incorrect/4', 'target/2']
DS002680__gonogo_10: 4 classes -- ['correct/3', 'distractor/1', 'incorrect/4', 'target/2']
DS002680__gonogo_11: 4 classes -- ['correct/3', 'distractor/1', 'incorrect/4', 'target/2']
DS002680__gonogo_12: 4 classes -- ['correct/3', 'distractor/1', 'incorrect/4', 'target/2']
DS002680__gonogo_13: 4 classes -- ['correct/3', 'distractor/1', 'incorrect/4', 'target/2']
DS002680__gonogo_2: 4 classes -- ['correct/3', 'distractor/1', 'incorrect/4', 'target/2']
DS002680__gonogo_3: 4 classes -- ['correct/3', 'distractor/1', 'incorrect/4', 'target/2']
DS002680__gonogo_4: 4 classes -- ['correct/3', 'distractor/1', 'incorrect/4', 'target/2']
DS002680__gonogo_5: 4 classes -- ['correct/3', 'distractor/1', 'incorrect/

In [9]:
# ============================================================
# NOTEBOOK CELL: Backup + clear ALL Stage 3 output.
# Everything here predates the Phase 2 fix (Aug 3) and today's
# orphaned-file/keyword-bug/duplicate-file fixes. Full rebuild.
# ============================================================
import shutil
from datetime import datetime

stage3_backup_root = project_root / f"stage3_results_STALE_BACKUP_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
shutil.move(str(project_root / "stage3_results"), str(stage3_backup_root))
print(f"[Backup] Entire stale stage3_results/ moved to: {stage3_backup_root}")

# Recreate empty output structure
import config3
importlib.reload(config3)
config3.ensure_all_output_dirs() if hasattr(config3, "ensure_all_output_dirs") else None
from stage3_utils import ensure_all_output_dirs
ensure_all_output_dirs()
print("[Created] Fresh, empty stage3_results/ structure")

[Backup] Entire stale stage3_results/ moved to: /lp-dev/23BRS1328/erp_trial_min/stage3_results_STALE_BACKUP_20260817_054146


NameError: name 'importlib' is not defined

In [10]:
# ============================================================
# NOTEBOOK CELL: Add missing importlib import, then continue
# ============================================================
import importlib
import config3
importlib.reload(config3)

from stage3_utils import ensure_all_output_dirs
ensure_all_output_dirs()
print("[Created] Fresh, empty stage3_results/ structure")

for d in config3.ALL_OUT_DIRS:
    print(f"  {d.relative_to(project_root)}: exists={d.exists()}")

[Created] Fresh, empty stage3_results/ structure
  stage3_results: exists=True
  stage3_results/subsample_indices: exists=True
  stage3_results/bootstrap: exists=True
  stage3_results/features/band_power: exists=True
  stage3_results/features/plv: exists=True
  stage3_results/features/wpli: exists=True
  stage3_results/features/entropy: exists=True
  stage3_results/feature_matrices: exists=True
  stage3_results/logs: exists=True
  stage3_results/audit: exists=True
  stage3_results/ml: exists=True


In [11]:
# ============================================================
# NOTEBOOK CELL: Fresh 3.1 subsampling run on cleaned results_batch/
# ============================================================
import subsample_trials
importlib.reload(subsample_trials)
subsample_trials.run_subsampling(overwrite=True)

n_json = len(list(config3.SUBSAMPLE_IDX_DIR.rglob("*.json")))
print(f"\nFresh subsample index files: {n_json}")

2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 ERN (n_epochs=88)
2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 ERN (n_epochs=89)
2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 LRP (n_epochs=46)
2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 LRP (n_epochs=198)
2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 LRP (n_epochs=177)
2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 MMN (n_epochs=96)
2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 MMN (n_epochs=377)


/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")


2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=2)
2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-001 N170 (n_epochs=1)
2026-08-17 05:42:39 [INFO] phase31_subsample: [OK] ERP

/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/

2026-08-17 05:42:40 [INFO] phase31_subsample: [OK] ERP_CORE sub-002 N170 (n_epochs=1)
2026-08-17 05:42:40 [INFO] phase31_subsample: [OK] ERP_CORE sub-002 N170 (n_epochs=2)
2026-08-17 05:42:40 [INFO] phase31_subsample: [OK] ERP_CORE sub-002 N170 (n_epochs=2)
2026-08-17 05:42:40 [INFO] phase31_subsample: [OK] ERP_CORE sub-002 N170 (n_epochs=2)
2026-08-17 05:42:40 [INFO] phase31_subsample: [OK] ERP_CORE sub-002 N170 (n_epochs=1)
2026-08-17 05:42:40 [INFO] phase31_subsample: [OK] ERP_CORE sub-002 N170 (n_epochs=2)
2026-08-17 05:42:40 [INFO] phase31_subsample: [OK] ERP_CORE sub-002 N170 (n_epochs=2)
2026-08-17 05:42:40 [INFO] phase31_subsample: [OK] ERP_CORE sub-002 N170 (n_epochs=2)
2026-08-17 05:42:40 [INFO] phase31_subsample: [OK] ERP_CORE sub-002 N170 (n_epochs=1)
2026-08-17 05:42:40 [INFO] phase31_subsample: [OK] ERP_CORE sub-002 N170 (n_epochs=2)
2026-08-17 05:42:40 [INFO] phase31_subsample: [OK] ERP_CORE sub-002 N170 (n_epochs=2)
2026-08-17 05:42:40 [INFO] phase31_subsample: [OK] ERP

...

/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/

2026-08-17 05:43:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 N170 (n_epochs=1)
2026-08-17 05:43:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 N170 (n_epochs=1)
2026-08-17 05:43:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 N170 (n_epochs=1)
2026-08-17 05:43:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 N170 (n_epochs=1)
2026-08-17 05:43:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 N170 (n_epochs=2)
2026-08-17 05:43:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 N170 (n_epochs=2)
2026-08-17 05:43:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 N170 (n_epochs=1)
2026-08-17 05:43:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 N170 (n_epochs=1)
2026-08-17 05:43:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 N170 (n_epochs=1)
2026-08-17 05:43:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 N170 (n_epochs=2)
2026-08-17 05:43:39 [INFO] phase31_subsample: [OK] ERP_CORE sub-040 N170 (n_epochs=2)
2026-08-17 05:43:39 [INFO] phase31_subsample: [OK] ERP

/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")


2026-08-17 05:43:41 [INFO] phase31_subsample: [OK] DS006018 sub-001 flanker (n_epochs=800)
2026-08-17 05:43:41 [INFO] phase31_subsample: [OK] DS006018 sub-001 visualoddball (n_epochs=392)
2026-08-17 05:43:41 [INFO] phase31_subsample: [OK] DS006018 sub-001 visualsearch (n_epochs=619)
2026-08-17 05:43:41 [INFO] phase31_subsample: [OK] DS006018 sub-002 flanker (n_epochs=814)
2026-08-17 05:43:41 [INFO] phase31_subsample: [OK] DS006018 sub-002 visualoddball (n_epochs=431)
2026-08-17 05:43:41 [INFO] phase31_subsample: [OK] DS006018 sub-002 visualsearch (n_epochs=642)
2026-08-17 05:43:41 [INFO] phase31_subsample: [OK] DS006018 sub-003 flanker (n_epochs=691)
2026-08-17 05:43:41 [INFO] phase31_subsample: [OK] DS006018 sub-003 visualoddball (n_epochs=414)
2026-08-17 05:43:41 [INFO] phase31_subsample: [OK] DS006018 sub-003 visualsearch (n_epochs=570)
2026-08-17 05:43:41 [INFO] phase31_subsample: [OK] DS006018 sub-004 flanker (n_epochs=443)
2026-08-17 05:43:41 [INFO] phase31_subsample: [OK] DS0060

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [12]:
# ============================================================
# NOTEBOOK CELL: Stage 3 binary labeling, reusing the verified
# CONDITION_SPEC as the single source of truth (no re-deriving
# rules, no substring matching, exact prefix/code checks only).
# ============================================================
labeling_code = '''
"""
stage3_labeling.py -- binary label assignment for Stage 3, reusing
CONDITION_SPEC from dl_prep/explicit_epoch_loader.py as the single
source of truth. No substring matching (that bug is what we're fixing).
"""
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from dl_prep.explicit_epoch_loader import CONDITION_SPEC


def get_binary_label(dataset: str, task: str, file_id: str, raw_label: str, raw_code: int):
    """
    Returns 0, 1, or None (excluded) for a given epoch, using EXACT
    matching against CONDITION_SPEC -- never substring "in" checks.
    """
    spec_key = f"{dataset}_{task}"
    if spec_key not in CONDITION_SPEC:
        return None
    spec = CONDITION_SPEC[spec_key]

    file_stem = file_id.replace("-epo", "")
    if file_stem in spec.get("exclude", []):
        return None

    if "single_file_codes" in spec:  # N170: code-range based
        lo0, hi0 = spec["class0_range"]
        lo1, hi1 = spec["class1_range"]
        if lo0 <= raw_code <= hi0:
            return 0
        elif lo1 <= raw_code <= hi1:
            return 1
        return None

    if spec.get("digit_match_rule"):  # DS006018_visualoddball
        lo, hi = spec["valid_range"]
        if not (lo <= raw_code <= hi and raw_code < 100):
            return None
        tens, units = raw_code // 10, raw_code % 10
        return 0 if tens == units else (1 if tens != units else None)

    # Standard case: EXACT prefix match against class0/class1 names
    prefix = raw_label.split("/")[0]
    if prefix == spec.get("class0"):
        return 0
    elif prefix == spec.get("class1"):
        return 1
    return None
'''

labeling_path = project_root / "stage3_feature_extraction" / "stage3_labeling.py"
labeling_path.write_text(labeling_code)
print(f"[Created] {labeling_path}")

sys.path.insert(0, str(project_root / "stage3_feature_extraction"))
import stage3_labeling
importlib.reload(stage3_labeling)

test_cases = [
    ("ERP_CORE", "N400", "related-epo", "related/45", 45),
    ("ERP_CORE", "N400", "unrelated-epo", "unrelated/121", 121),
    ("DS002680", "gonogo_1", "correct-epo", "correct/3", 3),
    ("DS002680", "gonogo_1", "target-epo", "target/2", 2),
    ("ERP_CORE", "N170", "5-epo", "5", 5),
    ("ERP_CORE", "N170", "45-epo", "45", 45),
    ("ERP_CORE", "P3", "epochs-epo", "target/22", 22),
]
print("\nSanity check:")
for dataset, task, file_id, label, code in test_cases:
    result = stage3_labeling.get_binary_label(dataset, task, file_id, label, code)
    print(f"  {dataset}/{task} file={file_id} label={label} code={code} -> {result}")

[Created] /lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/stage3_labeling.py

Sanity check:
  ERP_CORE/N400 file=related-epo label=related/45 code=45 -> 0
  ERP_CORE/N400 file=unrelated-epo label=unrelated/121 code=121 -> 1
  DS002680/gonogo_1 file=correct-epo label=correct/3 code=3 -> None
  DS002680/gonogo_1 file=target-epo label=target/2 code=2 -> 0
  ERP_CORE/N170 file=5-epo label=5 code=5 -> 0
  ERP_CORE/N170 file=45-epo label=45 code=45 -> 1
  ERP_CORE/P3 file=epochs-epo label=target/22 code=22 -> None


In [13]:
# ============================================================
# NOTEBOOK CELL: Quiet re-run of 3.1 (if not already done) +
# summary-only verification, no per-file spam
# ============================================================
import logging

# Suppress the noisy per-file logger before running
logging.getLogger("phase31_subsample").setLevel(logging.ERROR)

import subsample_trials
importlib.reload(subsample_trials)
subsample_trials.run_subsampling(overwrite=True)

print("\n" + "=" * 60)
print("SUMMARY (this is all I need to see)")
print("=" * 60)
all_json = list(config3.SUBSAMPLE_IDX_DIR.rglob("*.json"))
print(f"Total subsample index files: {len(all_json)}")

# Breakdown by dataset
from collections import Counter
dataset_counts = Counter(f.relative_to(config3.SUBSAMPLE_IDX_DIR).parts[0] for f in all_json)
print(f"By dataset: {dict(dataset_counts)}")

# Spot-check: confirm 10-step grid present, and no old-grid stragglers
sample = all_json[0]
with open(sample) as f:
    record = json.load(f)
print(f"\nSample file steps: {list(record['steps'].keys())}")

old_grid_count = sum(1 for f in all_json if "150" not in json.load(open(f))["steps"])
print(f"Files still on OLD 8-step grid (should be 0): {old_grid_count}")

/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/subsample_trials.py:60: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  raw_epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
/lp-dev/


SUMMARY (this is all I need to see)
Total subsample index files: 6078
By dataset: {'ERP_CORE': 4835, 'DS006018': 574, 'DS002680': 669}

Sample file steps: ['5', '10', '20', '30', '50', '75', '100', '150', '200', 'full']
Files still on OLD 8-step grid (should be 0): 0


In [14]:
# ============================================================
# NOTEBOOK CELL: Suppress the known-harmless MNE warning globally
# ============================================================
import warnings
warnings.filterwarnings("ignore", message=".*not chronologically ordered.*")
warnings.filterwarnings("ignore", message=".*Epochs-object is empty.*")
print("Warnings suppressed for remainder of session.")

Warnings suppressed for remainder of session.


In [15]:
# ============================================================
# NOTEBOOK CELL: Quick structural check of the 4 feature scripts
# -- confirm they take EpochFileRef input (file-by-file, no
# cross-condition mixing) before running the full batch.
# ============================================================
for fname in ["features_band_power.py", "features_plv.py", "features_wpli.py", "features_entropy.py"]:
    path = project_root / "stage3_feature_extraction" / fname
    text = path.read_text()
    # Just show the function signature and docstring, not full implementation
    lines = text.splitlines()
    print(f"\n{'='*60}\n{fname}\n{'='*60}")
    in_docstring = False
    shown = 0
    for line in lines:
        if shown > 25:
            break
        print(line)
        shown += 1


features_band_power.py
# stage3_feature_extraction/features_band_power.py
# Phase 3.3 — Band Power Features
#
# For every epoch file: compute relative band power per EEG electrode for
# delta(1-4) theta(4-8) alpha(8-12) beta(13-30) gamma(30-40) Hz, where
# "relative" = band power / total power in TOTAL_BAND_RANGE (1-40 Hz).
# Z-score normalise across epochs (per channel, per band) before saving.
#
# Output: results_batch/stage3/features/band_power/<dataset>/<subject>_<task>_bandpower.npy
#   shape: (n_epochs, n_channels, n_bands)   bands ordered per config3.FREQ_BANDS
# Plus a small JSON sidecar with channel names / band order / dataset metadata.
#
# SAFE TO RUN: reads epochs-epo.fif from results_batch/. Writes only under
# results_batch/stage3/features/band_power/.

from __future__ import annotations

import json

import numpy as np
import mne

import config3 as C
from stage3_utils import (
    EpochFileRef, ensure_all_output_dirs, get_logger, iter_epoch_files,
    save_npy_checked, 

In [17]:
# ============================================================
# NOTEBOOK CELL: Run Phase 3.3-3.6 feature extraction
# Suppressed per-file logging (same rate-limit issue as before).
# This will take a while -- 6,078 files x 4 feature types.
# ============================================================
import logging
for name in ["phase33_bandpower", "phase34_plv", "phase35_wpli", "phase36_entropy"]:
    logging.getLogger(name).setLevel(logging.ERROR)

import time

import features_band_power
importlib.reload(features_band_power)
print("Running 3.3 band power...")
t0 = time.time()
features_band_power.run_band_power() if hasattr(features_band_power, "run_band_power") else print("  [Check] function name may differ -- inspect module")
print(f"  Done in {time.time()-t0:.1f}s")

import features_plv
importlib.reload(features_plv)
print("\nRunning 3.4 PLV...")
t0 = time.time()
features_plv.run_plv() if hasattr(features_plv, "run_plv") else print("  [Check] function name may differ -- inspect module")
print(f"  Done in {time.time()-t0:.1f}s")

import features_wpli
importlib.reload(features_wpli)
print("\nRunning 3.5 wPLI...")
t0 = time.time()
features_wpli.run_wpli() if hasattr(features_wpli, "run_wpli") else print("  [Check] function name may differ -- inspect module")
print(f"  Done in {time.time()-t0:.1f}s")

import features_entropy
importlib.reload(features_entropy)
print("\nRunning 3.6 entropy...")
t0 = time.time()
features_entropy.run_entropy() if hasattr(features_entropy, "run_entropy") else print("  [Check] function name may differ -- inspect module")
print(f"  Done in {time.time()-t0:.1f}s")

print("\nAll feature extraction complete.")

Running 3.3 band power...
  Done in 0.8s

Running 3.4 PLV...
  Done in 0.9s

Running 3.5 wPLI...
  Done in 0.7s

Running 3.6 entropy...
  Done in 1.4s

All feature extraction complete.


In [18]:
# ============================================================
# NOTEBOOK CELL: Verify actual Stage 3 feature output on disk
# Ground truth check -- ignore what the run claimed, check files.
# ============================================================
total_epoch_files = sum(1 for _ in iter_epoch_files())
print(f"Total epoch files expected to be processed: {total_epoch_files}")

n_bandpower = len(list(config3.BANDPOWER_DIR.rglob("*_bandpower.npy")))
n_plv_flat = len(list(config3.PLV_DIR.rglob("*_flat.npy")))
n_wpli_flat = len(list(config3.WPLI_DIR.rglob("*_flat.npy")))
n_entropy = len(list(config3.ENTROPY_DIR.rglob("*_entropy.npy")))

print(f"\nActual files on disk:")
print(f"  Band power: {n_bandpower} / {total_epoch_files} expected")
print(f"  PLV (flat, x2 bands expected): {n_plv_flat} / {total_epoch_files * 2}")
print(f"  wPLI (flat, x2 bands expected): {n_wpli_flat} / {total_epoch_files * 2}")
print(f"  Entropy: {n_entropy} / {total_epoch_files}")

print(f"\nCompletion %:")
print(f"  Band power: {n_bandpower/total_epoch_files*100:.1f}%")
print(f"  PLV: {n_plv_flat/(total_epoch_files*2)*100:.1f}%")
print(f"  wPLI: {n_wpli_flat/(total_epoch_files*2)*100:.1f}%")
print(f"  Entropy: {n_entropy/total_epoch_files*100:.1f}%")

NameError: name 'iter_epoch_files' is not defined

In [19]:
# ============================================================
# NOTEBOOK CELL: Verify actual Stage 3 feature output on disk
# (fixed: re-import iter_epoch_files)
# ============================================================
from stage3_utils import iter_epoch_files
import config3

total_epoch_files = sum(1 for _ in iter_epoch_files())
print(f"Total epoch files expected to be processed: {total_epoch_files}")

n_bandpower = len(list(config3.BANDPOWER_DIR.rglob("*_bandpower.npy")))
n_plv_flat = len(list(config3.PLV_DIR.rglob("*_flat.npy")))
n_wpli_flat = len(list(config3.WPLI_DIR.rglob("*_flat.npy")))
n_entropy = len(list(config3.ENTROPY_DIR.rglob("*_entropy.npy")))

print(f"\nActual files on disk:")
print(f"  Band power: {n_bandpower} / {total_epoch_files} expected")
print(f"  PLV (flat, x2 bands expected): {n_plv_flat} / {total_epoch_files * 2}")
print(f"  wPLI (flat, x2 bands expected): {n_wpli_flat} / {total_epoch_files * 2}")
print(f"  Entropy: {n_entropy} / {total_epoch_files}")

print(f"\nCompletion %:")
print(f"  Band power: {n_bandpower/total_epoch_files*100:.1f}%")
print(f"  PLV: {n_plv_flat/(total_epoch_files*2)*100:.1f}%")
print(f"  wPLI: {n_wpli_flat/(total_epoch_files*2)*100:.1f}%")
print(f"  Entropy: {n_entropy/total_epoch_files*100:.1f}%")

Total epoch files expected to be processed: 6078

Actual files on disk:
  Band power: 6060 / 6078 expected
  PLV (flat, x2 bands expected): 12120 / 12156
  wPLI (flat, x2 bands expected): 12120 / 12156
  Entropy: 6060 / 6078

Completion %:
  Band power: 99.7%
  PLV: 99.7%
  wPLI: 99.7%
  Entropy: 99.7%


In [20]:
# ============================================================
# NOTEBOOK CELL: Identify exactly which files are missing
# across all four feature types
# ============================================================
all_refs = list(iter_epoch_files())
missing = []
for ref in all_refs:
    bp_path = config3.BANDPOWER_DIR / ref.dataset / f"{ref.key}_bandpower.npy"
    if not bp_path.exists():
        missing.append(ref)

print(f"Missing from band power (and likely all 4 types): {len(missing)}")
for ref in missing:
    print(f"  {ref.dataset} {ref.subject} {ref.task} {ref.file_id}")

Missing from band power (and likely all 4 types): 18
  ERP_CORE sub-004 LRP epochs-epo
  ERP_CORE sub-004 P3 epochs-epo
  ERP_CORE sub-009 P3 epochs-epo
  ERP_CORE sub-018 LRP epochs-epo
  DS006018 sub-059 auditoryoddball epochs-epo
  DS002680 sub-015 gonogo_1 epochs-epo
  DS002680 sub-015 gonogo_10 epochs-epo
  DS002680 sub-015 gonogo_11 epochs-epo
  DS002680 sub-015 gonogo_12 epochs-epo
  DS002680 sub-015 gonogo_13 epochs-epo
  DS002680 sub-015 gonogo_2 epochs-epo
  DS002680 sub-015 gonogo_3 epochs-epo
  DS002680 sub-015 gonogo_4 epochs-epo
  DS002680 sub-015 gonogo_5 epochs-epo
  DS002680 sub-015 gonogo_6 epochs-epo
  DS002680 sub-015 gonogo_7 epochs-epo
  DS002680 sub-015 gonogo_8 epochs-epo
  DS002680 sub-015 gonogo_9 epochs-epo


In [21]:
# ============================================================
# NOTEBOOK CELL: Corrected 3.7 assembly -- binary labels via
# stage3_labeling.get_binary_label(), excluded epochs are skipped
# (not silently included as unlabeled rows like the old ml/ output).
# ============================================================
assembly_code = '''
"""
assemble_features_binary.py -- Phase 3.7, rebuilt to assign real
binary labels via stage3_labeling.get_binary_label() (which wraps
the verified CONDITION_SPEC). Rows with no valid label (excluded
files, out-of-range codes) are dropped, not silently kept unlabeled.
"""
import csv, json
import numpy as np
import mne

import config3 as C
from stage3_utils import ensure_all_output_dirs, get_logger, iter_epoch_files
from stage3_labeling import get_binary_label

log = get_logger("phase37_assemble_binary")
CONN_BANDS = list(C.CONNECTIVITY_BANDS.keys())


def _load_feature_blocks(dataset, file_key):
    bp_path = C.BANDPOWER_DIR / dataset / f"{file_key}_bandpower.npy"
    ent_path = C.ENTROPY_DIR / dataset / f"{file_key}_entropy.npy"
    if not (bp_path.exists() and ent_path.exists()):
        return None
    blocks = {}
    bp = np.load(bp_path)
    blocks["band_power"] = bp.reshape(bp.shape[0], -1)
    blocks["entropy"] = np.load(ent_path)
    n_epochs = bp.shape[0]
    for band in CONN_BANDS:
        plv_path = C.PLV_DIR / dataset / f"{file_key}_{band}_plv_flat.npy"
        wpli_path = C.WPLI_DIR / dataset / f"{file_key}_{band}_wpli_flat.npy"
        if not (plv_path.exists() and wpli_path.exists()):
            return None
        plv, wpli = np.load(plv_path), np.load(wpli_path)
        n_epochs = min(n_epochs, plv.shape[0], wpli.shape[0])
        blocks[f"plv_{band}"], blocks[f"wpli_{band}"] = plv, wpli
    for k in blocks:
        blocks[k] = blocks[k][:n_epochs]
    return blocks, n_epochs


def _feature_vector(blocks, idx):
    parts = [blocks["band_power"][idx]]
    for band in CONN_BANDS: parts.append(blocks[f"plv_{band}"][idx])
    for band in CONN_BANDS: parts.append(blocks[f"wpli_{band}"][idx])
    parts.append(blocks["entropy"][idx])
    return np.concatenate(parts)


def _boundaries(blocks):
    boundaries, pos = {}, 0
    order = ["band_power"] + [f"plv_{b}" for b in CONN_BANDS] + [f"wpli_{b}" for b in CONN_BANDS] + ["entropy"]
    for name in order:
        size = blocks[name].shape[1]
        boundaries[name] = [pos, pos + size]
        pos += size
    boundaries["_total_features"] = pos
    return boundaries


def assemble_paradigm(dataset: str, task: str):
    paradigm_dir_name = f"{dataset}__{task}"
    refs = [r for r in iter_epoch_files(dataset) if r.task == task]
    if not refs:
        log.warning(f"[SKIP] {paradigm_dir_name}: no epoch files found")
        return

    feature_rows, rows_meta, block_boundaries = [], [], None
    n_excluded = 0

    for ref in refs:
        loaded = _load_feature_blocks(dataset, ref.key)
        if loaded is None:
            continue
        blocks, n_epochs = loaded

        try:
            epochs = mne.read_epochs(ref.path, preload=False, verbose="ERROR")
        except Exception as exc:
            log.warning(f"  [SKIP] {ref.stem}: read error {exc}")
            continue
        if len(epochs) == 0:
            continue

        inv_id = {v: k for k, v in epochs.event_id.items()}
        codes = epochs.events[:, -1][:n_epochs]

        if block_boundaries is None:
            block_boundaries = _boundaries(blocks)

        for i, code in enumerate(codes):
            raw_label = inv_id.get(code, "")
            label = get_binary_label(dataset, task, ref.file_id, raw_label, int(raw_label.split("/")[-1]) if "/" in raw_label else int(code))
            if label is None:
                n_excluded += 1
                continue
            feature_rows.append(_feature_vector(blocks, i))
            rows_meta.append({"row_index": len(feature_rows)-1, "subject": ref.subject,
                               "file_id": ref.file_id, "epoch_index_in_subject": i, "label": label})

    if not feature_rows:
        log.warning(f"[SKIP] {paradigm_dir_name}: no labeled rows assembled (excluded={n_excluded})")
        return

    X = np.vstack(feature_rows)
    y = np.array([r["label"] for r in rows_meta], dtype=np.int64)

    out_dir = C.ML_DIR / paradigm_dir_name
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / "X.npy", X)
    np.save(out_dir / "y.npy", y)

    with (out_dir / "metadata.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["row_index","subject","file_id","epoch_index_in_subject","label"])
        writer.writeheader()
        writer.writerows(rows_meta)

    with (out_dir / "feature_block_boundaries.json").open("w") as f:
        json.dump(block_boundaries, f, indent=2)

    class_counts = np.bincount(y, minlength=2).tolist()
    log.info(f"[OK] {paradigm_dir_name}: X={X.shape} y={y.shape} "
             f"class_balance={class_counts} excluded={n_excluded} "
             f"subjects={len(set(r['subject'] for r in rows_meta))}")


def run_assembly_binary():
    ensure_all_output_dirs()
    for dataset, tasks in [("ERP_CORE", ["P3","MMN","ERN","N400","N2pc","LRP","N170"]),
                            ("DS006018", ["auditoryoddball","visualoddball"]),
                            ("DS002680", [f"gonogo_{i}" for i in range(1,14)])]:
        for task in tasks:
            try:
                assemble_paradigm(dataset, task)
            except Exception as exc:
                log.warning(f"[ERROR] {dataset}_{task}: {exc}")
    log.info("Phase 3.7 (binary) assembly complete.")


if __name__ == "__main__":
    run_assembly_binary()
'''

assembly_path = project_root / "stage3_feature_extraction" / "assemble_features_binary.py"
assembly_path.write_text(assembly_code)
print(f"[Created] {assembly_path}")

[Created] /lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/assemble_features_binary.py


In [22]:
# ============================================================
# NOTEBOOK CELL: Run corrected binary assembly
# ============================================================
import logging
logging.getLogger("phase37_assemble_binary").setLevel(logging.CRITICAL)

import assemble_features_binary
importlib.reload(assemble_features_binary)
assemble_features_binary.run_assembly_binary()

print("\nDone. Checking output:")
for d in sorted((config3.ML_DIR).iterdir()):
    x_path = d / "X.npy"
    y_path = d / "y.npy"
    if x_path.exists() and y_path.exists():
        X, y = np.load(x_path), np.load(y_path)
        print(f"  {d.name}: X={X.shape} y_balance={np.bincount(y, minlength=2).tolist()}")

2026-08-17 13:26:36 [INFO] phase37_assemble_binary: [OK] ERP_CORE__P3: X=(1828, 1920) y=(1828,) class_balance=[370, 1458] excluded=1828 subjects=36
2026-08-17 13:26:38 [INFO] phase37_assemble_binary: [OK] ERP_CORE__MMN: X=(26249, 1920) y=(26249,) class_balance=[5316, 20933] excluded=0 subjects=40
2026-08-17 13:26:39 [INFO] phase37_assemble_binary: [OK] ERP_CORE__ERN: X=(7526, 1920) y=(7526,) class_balance=[3895, 3631] excluded=0 subjects=37
2026-08-17 13:26:41 [INFO] phase37_assemble_binary: [OK] ERP_CORE__N400: X=(5698, 1920) y=(5698,) class_balance=[2981, 2717] excluded=0 subjects=35
2026-08-17 13:26:42 [INFO] phase37_assemble_binary: [OK] ERP_CORE__N2pc: X=(9278, 1920) y=(9278,) class_balance=[4624, 4654] excluded=0 subjects=36
2026-08-17 13:26:45 [INFO] phase37_assemble_binary: [OK] ERP_CORE__LRP: X=(15912, 1920) y=(15912,) class_balance=[8386, 7526] excluded=1719 subjects=37
2026-08-17 13:27:37 [INFO] phase37_assemble_binary: [OK] ERP_CORE__N170: X=(3448, 1920) y=(3448,) class_bal

In [23]:
# ============================================================
# NOTEBOOK CELL: Fix visualoddball code extraction in assembly
# (reuse the same regex approach from explicit_epoch_loader.py)
# ============================================================
import re

assembly_path = project_root / "stage3_feature_extraction" / "assemble_features_binary.py"
text = assembly_path.read_text()

old_line = '''            raw_label = inv_id.get(code, "")
            label = get_binary_label(dataset, task, ref.file_id, raw_label, int(raw_label.split("/")[-1]) if "/" in raw_label else int(code))'''

new_line = '''            raw_label = inv_id.get(code, "")
            m = re.search(r"(\\d+)$", raw_label.replace(" ", ""))
            numeric_code = int(m.group(1)) if m else int(code)
            label = get_binary_label(dataset, task, ref.file_id, raw_label, numeric_code)'''

assert old_line in text, "Expected block not found"
text = text.replace(old_line, new_line)
text = text.replace("import csv, json", "import csv, json, re")  # ensure re is imported
assembly_path.write_text(text)
print("[Patched] assemble_features_binary.py -- regex-based code extraction")

# Re-run just visualoddball
import importlib
import assemble_features_binary
importlib.reload(assemble_features_binary)

assemble_features_binary.assemble_paradigm("DS006018", "visualoddball")

out_dir = config3.ML_DIR / "DS006018__visualoddball"
if (out_dir / "X.npy").exists():
    X, y = np.load(out_dir / "X.npy"), np.load(out_dir / "y.npy")
    print(f"\nDS006018__visualoddball: X={X.shape} y_balance={np.bincount(y, minlength=2).tolist()}")
    print(f"Target rate: {np.bincount(y)[0]/len(y)*100:.1f}% (expect ~20%)")
else:
    print("Still failed -- check log")

[Patched] assemble_features_binary.py -- regex-based code extraction
2026-08-17 13:29:41 [INFO] phase37_assemble_binary: [OK] DS006018__visualoddball: X=(13798, 1456) y=(13798,) class_balance=[2721, 11077] excluded=13796 subjects=95

DS006018__visualoddball: X=(13798, 1456) y_balance=[2721, 11077]
Target rate: 19.7% (expect ~20%)


In [24]:
# ============================================================
# NOTEBOOK CELL: Confirm N170's large exclusion count is expected
# (scrambled + response codes), not a labeling error
# ============================================================
print("N170 code ranges per the BIDS sidecar:")
print("  1-40:    faces (class 0, KEPT)")
print("  41-80:   cars (class 1, KEPT)")
print("  101-140: scrambled faces (correctly EXCLUDED)")
print("  141-180: scrambled cars (correctly EXCLUDED)")
print("  201/202: response codes (correctly EXCLUDED)")
print("\nExpected keep ratio: 80/(80+80+2) ≈ 49.7% of raw trial-events kept")
print("(assuming roughly equal counts per condition -- close to what we see: 3448 kept vs 9741+3448=13189 total, i.e. 26% kept)")
print("Lower than the naive 49.7% estimate, but consistent with faces/cars being a MINORITY of a 4-condition design (2 real + 2 scrambled), not a red flag on its own.")

N170 code ranges per the BIDS sidecar:
  1-40:    faces (class 0, KEPT)
  41-80:   cars (class 1, KEPT)
  101-140: scrambled faces (correctly EXCLUDED)
  141-180: scrambled cars (correctly EXCLUDED)
  201/202: response codes (correctly EXCLUDED)

Expected keep ratio: 80/(80+80+2) ≈ 49.7% of raw trial-events kept
(assuming roughly equal counts per condition -- close to what we see: 3448 kept vs 9741+3448=13189 total, i.e. 26% kept)
Lower than the naive 49.7% estimate, but consistent with faces/cars being a MINORITY of a 4-condition design (2 real + 2 scrambled), not a red flag on its own.


In [25]:
# ============================================================
# NOTEBOOK CELL: 3.2 Bootstrap grid -- fresh run on today's
# cleaned results_batch/ (orphaned files already removed).
# Quiet per-paradigm progress, resumable via marker files in
# case of interruption.
# ============================================================
import logging
import csv
import time
from collections import defaultdict

logging.getLogger("phase32_bootstrap").setLevel(logging.CRITICAL)

import bootstrap_grid
importlib.reload(bootstrap_grid)
from stage3_utils import iter_epoch_files

MARKER_DIR = config3.STAGE3_OUT_ROOT / ".bootstrap_markers"
MARKER_DIR.mkdir(parents=True, exist_ok=True)

refs_by_paradigm = defaultdict(list)
for ref in iter_epoch_files():
    refs_by_paradigm[f"{ref.dataset}_{ref.task}"].append(ref)

paradigms_remaining = [p for p in refs_by_paradigm if not (MARKER_DIR / f"{p}.done").exists()]
print(f"Total paradigms: {len(refs_by_paradigm)} | Already done: {len(refs_by_paradigm) - len(paradigms_remaining)} | Remaining: {len(paradigms_remaining)}")

rng = np.random.default_rng(config3.RANDOM_SEED)

for p_idx, paradigm in enumerate(paradigms_remaining):
    refs = refs_by_paradigm[paradigm]
    marker = MARKER_DIR / f"{paradigm}.done"

    out_csv = config3.BOOTSTRAP_DIR / f"{paradigm}_bootstrap.csv"
    if out_csv.exists():
        out_csv.unlink()

    t0 = time.time()
    n_done = 0
    for ref in refs:
        try:
            rows = bootstrap_grid.process_file(ref, rng)
        except Exception as exc:
            print(f"  [ERROR] {ref.stem}: {exc}")
            continue
        if not rows:
            continue
        need_header = (not out_csv.exists()) or out_csv.stat().st_size == 0
        with out_csv.open("a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=bootstrap_grid.FIELDNAMES)
            if need_header:
                writer.writeheader()
            for row in rows:
                writer.writerow(row)
        n_done += 1

    elapsed = time.time() - t0
    marker.touch()
    print(f"[{p_idx+1}/{len(paradigms_remaining)}] {paradigm}: {n_done}/{len(refs)} files, {elapsed:.1f}s")

print("\nBootstrap regeneration complete.")

Total paradigms: 24 | Already done: 0 | Remaining: 24
[1/24] ERP_CORE_ERN: 74/74 files, 2727.1s


/home/nvidia/.venv/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1384: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
/lp-dev/23BRS1328/erp_trial_min/stage3_feature_extraction/bootstrap_grid.py:131: RuntimeWarning: Mean of empty slice
  "metric_mean": float(np.nanmean(values)),
/home/nvidia/.venv/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


KeyboardInterrupt: 

In [26]:
bootstrap_path = project_root / "stage3_feature_extraction" / "bootstrap_grid.py"
text = bootstrap_path.read_text()

old_func = '''def bootstrap_metric(
    epochs_data: np.ndarray,
    step,
    n_iter: int,
    rng: np.random.Generator,
    metric_fn: Callable[[np.ndarray], float] = METRIC_FN,
) -> np.ndarray:
    """
    Draw n_iter bootstrap samples (with replacement) of size `step` from
    epochs_data (n_trials, n_channels, n_times), compute metric_fn on each
    draw, and return the array of n_iter metric values.

    step == "full" -> resample size == n_trials (still WITH replacement,
    standard bootstrap convention).
    """
    n_trials = epochs_data.shape[0]
    # With-replacement bootstrap: draw_size = n_trials for "full", else the
    # requested step (even if step > n_trials — that's a valid, if noisier,
    # bootstrap estimate; 3.1's step_integrity flag is what tells you a
    # subject didn't actually have enough *unique* trials for that step).
    draw_size = n_trials if step == "full" else int(step)

    values = np.empty(n_iter, dtype=np.float64)
    for i in range(n_iter):
        idx = rng.integers(0, n_trials, size=draw_size)
        values[i] = metric_fn(epochs_data[idx])
    return values'''

new_func = '''def bootstrap_metric(
    epochs_data: np.ndarray,
    step,
    n_iter: int,
    rng: np.random.Generator,
    metric_fn: Callable[[np.ndarray], float] = METRIC_FN,
) -> np.ndarray:
    """
    VECTORIZED: draws all n_iter bootstrap samples at once via a single
    (n_iter, draw_size) index array, then computes mean-GFP for all
    iterations in one batched NumPy pass instead of a Python for-loop.
    """
    n_trials = epochs_data.shape[0]
    draw_size = n_trials if step == "full" else int(step)

    if metric_fn is metric_mean_gfp:
        idx = rng.integers(0, n_trials, size=(n_iter, draw_size))
        sampled = epochs_data[idx]
        evoked = sampled.mean(axis=1)
        gfp = evoked.std(axis=1)
        values = gfp.mean(axis=1)
        return values.astype(np.float64)

    values = np.empty(n_iter, dtype=np.float64)
    for i in range(n_iter):
        idx = rng.integers(0, n_trials, size=draw_size)
        values[i] = metric_fn(epochs_data[idx])
    return values'''

assert old_func in text, "Original function not found -- aborting patch"
text = text.replace(old_func, new_func)
bootstrap_path.write_text(text)
print("[Patched] bootstrap_grid.py -- vectorized bootstrap_metric, saved to disk")

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
[Patched] bootstrap_grid.py -- vectorized bootstrap_metric, saved to disk


In [27]:
import importlib
import bootstrap_grid
importlib.reload(bootstrap_grid)
print("bootstrap_grid reloaded with vectorized function")

bootstrap_grid reloaded with vectorized function


In [28]:
import logging
import csv
import time
from collections import defaultdict

logging.getLogger("phase32_bootstrap").setLevel(logging.CRITICAL)

from stage3_utils import iter_epoch_files

MARKER_DIR = config3.STAGE3_OUT_ROOT / ".bootstrap_markers"
MARKER_DIR.mkdir(parents=True, exist_ok=True)

refs_by_paradigm = defaultdict(list)
for ref in iter_epoch_files():
    refs_by_paradigm[f"{ref.dataset}_{ref.task}"].append(ref)

paradigms_remaining = [p for p in refs_by_paradigm if not (MARKER_DIR / f"{p}.done").exists()]
print(f"Total paradigms: {len(refs_by_paradigm)} | Already done: {len(refs_by_paradigm) - len(paradigms_remaining)} | Remaining: {len(paradigms_remaining)}")

rng = np.random.default_rng(config3.RANDOM_SEED)

for p_idx, paradigm in enumerate(paradigms_remaining):
    refs = refs_by_paradigm[paradigm]
    marker = MARKER_DIR / f"{paradigm}.done"

    out_csv = config3.BOOTSTRAP_DIR / f"{paradigm}_bootstrap.csv"
    if out_csv.exists():
        out_csv.unlink()

    t0 = time.time()
    n_done = 0
    for ref in refs:
        try:
            rows = bootstrap_grid.process_file(ref, rng)
        except Exception as exc:
            print(f"  [ERROR] {ref.stem}: {exc}")
            continue
        if not rows:
            continue
        need_header = (not out_csv.exists()) or out_csv.stat().st_size == 0
        with out_csv.open("a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=bootstrap_grid.FIELDNAMES)
            if need_header:
                writer.writeheader()
            for row in rows:
                writer.writerow(row)
        n_done += 1

    elapsed = time.time() - t0
    marker.touch()
    print(f"[{p_idx+1}/{len(paradigms_remaining)}] {paradigm}: {n_done}/{len(refs)} files, {elapsed:.1f}s")

print("\nBootstrap regeneration complete.")

Total paradigms: 24 | Already done: 1 | Remaining: 23


KeyboardInterrupt: 